# AI Technologies (Basic Course) · Notebook 05
## Topic 3. Classical methods of feature extraction in images

| Lesson | Type, duration | Title |
|---|---|---|
| **3.1** | Lecture, 2 h | Extraction of edges, silhouettes and contours of objects by classical methods |
| **3.3** | Practical class, 2 h | Features of silhouettes and thermal signatures in thermal frames |
| **3.4** | Practical class, 2 h | Stages of the stabilisation of the video stream of an FPV drone |

Course: *AI Technologies (Basic Course)*, AIT-BC · Content module 1 · Kruty Heroes Military Institute of Telecommunications and Information Technology, Kyiv, 2026.

**How to work with this notebook**

1. Run the cells from top to bottom: *Runtime → Run all*, or `Shift+Enter` cell by cell. Section 0 is always run first; after a restart of the runtime it is run again.
2. Every block begins with a text cell: the purpose, the parameters of the form, the formulas, the expected result and the control question. The answers are written in the text cell **Report** at the end of each practical lesson.
3. The blocks of the lecture are demonstrations: change the parameters of the forms and observe the result. Cells marked **(code shown)** in the practical lessons are the practical skills: read the code, change it and run it again. Cells marked **[auto]** run again after every change of a parameter.
4. The results are saved in the course project on Google Drive (`data`, `outputs`, `project.json`), created in Notebook 02.

**Data.** Only real frames are used, with the annotation made by the authors of the data sets:
- **Kust4K** — pixel-aligned visible and thermal frames of a UAV over city roads, day and night, with pixel labels (the boundaries of the labelled regions are the reference for the edge detectors);
- **HIT-UAV** — thermal frames of a reconnaissance UAV at 60–130 m, day and night, with boxes of persons, cars and bicycles, flight altitude and camera angle in the file names;
- **M3OT** — thermal frames of a DJI Mavic 3T (black-hot palette) with vehicle tracks;
- **TII-RATM** — the video stream of the onboard camera of an FPV racing quadrotor flown by a human pilot (125 frames per second, wide-angle lens) with the measurements of its gyroscope at 500 Hz, which serve as an independent reference of the rotation of the camera.

**Contents**

- **0. Preparation** — 0.1 runtime, course library and data sets · 0.2 course project
- **Lesson 3.1 (lecture)**
  - 3.1.1 The edge of an object and the gradient of brightness (Sobel)
  - 3.1.2 Edges by the Sobel and Canny methods: comparison with the boundaries of the annotation
  - 3.1.3 Binarisation of an image, global threshold of Otsu
  - 3.1.4 Contour analysis and geometric characteristics of objects
- **Lesson 3.3 (practical class)**
  - 3.3.1 Geometric transformations: scaling, rotation, translation, perspective transformation
  - 3.3.2 Preparation of a thermal frame: grey levels, CLAHE, polarity of thermal signatures
  - 3.3.3 Morphological processing of the binary mask
  - 3.3.4 Contours and their filtering by geometric characteristics
  - 3.3.5 Visualisation: contours and boxes on the original frame
- **Lesson 3.4 (practical class)**
  - 3.4.1 Reading a sequence of frames of a test FPV video fragment
  - 3.4.2 ORB keypoints on a pair of consecutive frames
  - 3.4.3 Matching of keypoints and quantitative estimation of the inter-frame shift
  - 3.4.4 Homography and geometric alignment of frames
  - 3.4.5 Saving of the stabilised video stream and comparative analysis
- **Summary**: self-check of Topic 3, data sources and licences

---
# 0. Preparation

### Block 0.1. Runtime, course library and data sets

**Purpose.** To prepare the runtime: the libraries, the course library `aitbc` (version 2.4.0 adds the functions of Topic 3 and the reader of the FPV data set TII-RATM) and the open data sets of the course.

**Parameters.**
- `CACHE_ON_DRIVE` — keep the downloaded frames on Google Drive (they remain after the end of the session); otherwise they are kept on the disk of the session.
- `PREFETCH_ALL` — download now the frames of Lessons 3.1 and 3.3 (about 35 MB); the FPV frames of Lesson 3.4 (about 55 MB in the cache, streamed once from an archive of about 200 MB) are downloaded in block 3.4.1.

**Output.** A table of versions and the number of frames in the cache.

In [ ]:
#@title 0.1 Runtime, course library and data sets (run once) { display-mode: "form" }
CACHE_ON_DRIVE = False #@param {type:"boolean"}
PREFETCH_ALL = True #@param {type:"boolean"}

import sys, os, platform, time, json, math, importlib, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORK_DIR = "/content" if IN_COLAB else os.getcwd()

LIB_SOURCE = r'''"""
aitbc - course library of "AI Technologies (Basic Course)" (AIT-BC).

Kruty Heroes Military Institute of Telecommunications and Information Technology,
Kyiv, 2026.

All demonstrations of the course use REAL open data sets recorded by UAVs and
ground vehicles. Only the files that a block needs are downloaded: large archives
are read selectively with HTTP range requests, and every file is cached in
DATA_ROOT (a Google Drive folder keeps the cache between sessions).

Data sets
---------
HIT-UAV   2898 thermal (LWIR) images 640x512 and 5 thermal videos. DJI Matrice
          M210 V2 with a Zenmuse XT2 camera (25 mm lens, field of view 25 x 20 deg),
          flight altitude 60-130 m, camera angle 30-90 deg, day and night. Boxes of
          Person, Car, Bicycle, OtherVehicle, DontCare.
          Suo J. et al. Scientific Data 10, 227 (2023). Licence CC BY 4.0.
Kust4K    4024 pixel-aligned visible (RGB) and thermal (TIR) image pairs 640x512
          from a hexacopter over urban roads, day and night; pixel labels of 8 classes.
          Scientific Data 12, 1701 (2025). Licence CC BY 4.0.
VisDrone  RGB images of drones over cities (VisDrone2019-DET validation part,
          548 images) with boxes of pedestrians, vehicles and other objects.
          Zhu P. et al. IEEE TPAMI 44(11), 7380-7399 (2022).
GOOSE     camera images of a ground vehicle on roads, field tracks, in forests and on
          a training area, with pixel labels of 64 classes (validation part, 962 images).
          IEEE ICRA 2024. Licence CC BY-SA 4.0.
M3OT      synchronised visible and thermal image sequences of two DJI Mavic 3T drones
          (100-120 m, 10 fps, 640x512, day, dusk, night) with vehicle tracks.
          Nie Z. et al. Scientific Data 12, 1927 (2025). Licence CC BY 4.0.

TII-RATM  frames 640x480 (125 fps), IMU 500 Hz and motion capture of an FPV racing quadrotor
          flown by human pilots in an indoor arena with racing gates (wide-angle camera
          155 x 115 deg). Bosello M. et al. IEEE RA-L 9(4), 3799-3806 (2024). Licence CC BY 4.0.

Units: metres, seconds, degrees. Images are NumPy arrays H x W (grey) or H x W x 3 (RGB).
"""

__version__ = "2.4.0"

import csv
import io
import json
import math
import os
import struct
import time
import urllib.error
import urllib.request
import zlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import cv2

DATA_ROOT = os.environ.get("AITBC_DATA") or (
    "/content/aitbc_data" if os.path.isdir("/content") else os.path.abspath("aitbc_data"))


def set_data_root(path):
    """Set the folder of the download cache (for example a folder on Google Drive)."""
    global DATA_ROOT
    DATA_ROOT = os.path.abspath(path)
    os.makedirs(DATA_ROOT, exist_ok=True)
    return DATA_ROOT


def _cache(*parts):
    p = os.path.join(DATA_ROOT, *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def cache_size():
    """Number of files and size of the download cache, MB."""
    n, s = 0, 0
    for dp, _, fn in os.walk(DATA_ROOT):
        for f in fn:
            n += 1
            s += os.path.getsize(os.path.join(dp, f))
    return n, s / 1e6


# =============================================================================
# 1. Downloading
# =============================================================================
class _HTTPSRedirect(urllib.request.HTTPRedirectHandler):
    """Follow redirects, keep the Range header and upgrade http:// targets to https://."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if newurl.startswith("http://"):
            newurl = "https://" + newurl[len("http://"):]
        new = super().redirect_request(req, fp, code, msg, headers, newurl)
        if new is not None and req.has_header("Range"):
            new.add_header("Range", req.get_header("Range"))
        return new


_OPENER = urllib.request.build_opener(_HTTPSRedirect)
_OPENER.addheaders = [("User-Agent", "aitbc-course/2.0")]


def _urlopen(url, headers=None, timeout=60, retries=4):
    """Open url with retries; a new Request object is built for every attempt."""
    err = None
    for k in range(retries):
        try:
            return _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # network error: retry after a pause
            err = e
        time.sleep(1.5 * (k + 1))
    raise err


def _retry(transfer, retries=5):
    """Run a complete transfer (request and reading of the whole body) with retries: a broken connection,
    a read timeout or an incomplete body repeats the transfer after a pause; client errors are not repeated."""
    err = None
    for k in range(retries):
        try:
            return transfer()
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # timeout, reset connection, incomplete read
            err = e
        time.sleep(2.0 * (k + 1))
    raise err


def _read_url(url, headers=None, timeout=60, min_bytes=0, retries=5):
    """Whole body of a response (e.g. of a range request) with retries; a body shorter than min_bytes is
    an incomplete transfer and is repeated."""
    def transfer():
        with _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout) as r:
            data = r.read()
        if len(data) < min_bytes:
            raise IOError(f"incomplete transfer: {len(data)} of at least {min_bytes} bytes")
        return data
    return _retry(transfer, retries)


def fetch(url, path, timeout=120, retries=5):
    """Download url to path with retries; nothing is done when the file is already in the cache.
    The file appears under its name only after a complete transfer."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".part"

    def transfer():
        with _OPENER.open(urllib.request.Request(url), timeout=timeout) as r, open(tmp, "wb") as fh:
            expected = int(r.headers.get("Content-Length") or 0)
            n = 0
            while True:
                b = r.read(1 << 20)
                if not b:
                    break
                fh.write(b)
                n += len(b)
        if expected and n < expected:
            raise IOError(f"incomplete download of {url}: {n} of {expected} bytes")
        os.replace(tmp, path)
        return path
    return _retry(transfer, retries)


def fetch_many(jobs, threads=8):
    """jobs: list of (url, path); parallel download, returns the list of paths."""
    with ThreadPoolExecutor(threads) as ex:
        return list(ex.map(lambda j: fetch(*j), jobs))


class _RangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests (reads the directory of a remote zip)."""

    def __init__(self, url, chunk=1 << 17, timeout=60):
        self.url, self.chunk, self.timeout = url, chunk, timeout
        r = _urlopen(url, {"Range": "bytes=0-0"}, timeout)
        cr = r.headers.get("Content-Range")
        r.read()
        if r.status != 206 or not cr:
            raise IOError(f"the server of {url} does not support range requests")
        self.size = int(cr.split("/")[-1])
        self.pos, self.buf_start, self.buf = 0, 0, b""

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        if not (self.buf_start <= self.pos and self.pos + n <= self.buf_start + len(self.buf)):
            start = self.pos
            end = min(self.size, start + max(n, self.chunk)) - 1
            self.buf = _read_url(self.url, {"Range": f"bytes={start}-{end}"}, self.timeout, min_bytes=end - start + 1)
            self.buf_start = start
        off = self.pos - self.buf_start
        out = self.buf[off:off + n]
        self.pos += len(out)
        return out

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


class RemoteZip:
    """Selective reading of members of a large remote zip archive: only the central
    directory and the requested members are transferred; members are cached on disk."""

    def __init__(self, url, cache_name):
        import zipfile
        self.url, self.cache_name = url, cache_name
        idx_path = _cache("_index", cache_name + ".json")
        if os.path.exists(idx_path):
            with open(idx_path) as fh:
                self.index = json.load(fh)
        else:
            zf = zipfile.ZipFile(_RangeFile(url))
            self.index = {i.filename: (i.header_offset, i.compress_size, i.file_size, i.compress_type)
                          for i in zf.infolist() if not i.filename.endswith("/")}
            with open(idx_path, "w") as fh:
                json.dump(self.index, fh)

    def namelist(self):
        return list(self.index)

    def _fetch_member(self, name):
        path = _cache(self.cache_name, name)
        if os.path.exists(path):
            return path
        off, csize, usize, ctype = self.index[name]
        raw = _read_url(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, min_bytes=30 + csize)
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {name}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = raw[30 + n + m: 30 + n + m + csize]
        if ctype == 8:
            data = zlib.decompressobj(-15).decompress(data)
        elif ctype != 0:
            raise IOError(f"unsupported compression {ctype} in {name}")
        if len(data) != usize:
            raise IOError(f"incomplete member {name}")
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return path

    def get(self, names, threads=8):
        """Download members (a name or a list of names); returns local path(s)."""
        single = isinstance(names, str)
        names = [names] if single else list(names)
        with ThreadPoolExecutor(threads) as ex:
            paths = list(ex.map(self._fetch_member, names))
        return paths[0] if single else paths


def _rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# =============================================================================
# 2. Data sets
# =============================================================================
class HITUAV:
    """HIT-UAV: thermal images and videos of a DJI Matrice M210 V2 with a Zenmuse XT2.

    File name T_HHH_AA_W_NNNNN: T = 0 day / 1 night, HHH = flight altitude, m,
    AA = camera angle, deg (90 = vertical view), W = 0 sunny / 1 rainy.
    Images are 8-bit white-hot (warm = bright) after the automatic gain control
    of the camera: brightness is a relative, not a calibrated, temperature."""

    PALETTE = "white-hot"
    BASE = "https://raw.githubusercontent.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/main/"
    CLASSES = ("Person", "Car", "Bicycle", "OtherVehicle", "DontCare")
    COLORS = {"Person": (0, 229, 255), "Car": (255, 235, 59), "Bicycle": (118, 255, 3),
              "OtherVehicle": (255, 152, 0), "DontCare": (190, 190, 190)}
    CAMERA = dict(model="DJI Zenmuse XT2, uncooled VOx microbolometer", width=640, height=512,
                  focal_mm=25.0, pixel_pitch_um=17.0, hfov_deg=25.0, vfov_deg=20.0, band_um="7.5-13.5")
    EXAMPLES = ('1_60_30_0_00047.jpg', '0_60_60_0_01734.jpg', '1_70_90_0_07776.jpg', '0_100_50_0_08162.jpg', '1_120_90_0_06161.jpg', '0_130_40_0_05603.jpg', '1_90_70_0_04990.jpg', '0_110_80_0_08518.jpg')
    VIDEOS = {"60m-30_1.mov": "60 m, camera angle 30 deg: people on a sports ground",
              "60m-40_3.mov": "60 m, 40 deg: cars on a parking lot",
              "70m-90_1.mov": "70 m, 90 deg (vertical view): people on a square",
              "120m-30_3.mov": "120 m, 30 deg: road along buildings",
              "130m-60_3.mov": "130 m, 60 deg: parking lot and roads"}

    def __init__(self, splits=("train", "val", "test")):
        import pandas as pd
        imgs, objs = [], []
        jobs = [(self.BASE + f"normal_json/annotations/{sp}.json", _cache("hituav", "annotations", sp + ".json"))
                for sp in splits]
        for sp, path in zip(splits, fetch_many(jobs, 3)):
            with open(path) as fh:
                d = json.load(fh)
            cat = {c["id"]: c["name"] for c in d["categories"]}
            id2f = {}
            for im in d["images"]:
                fn = im.get("filename") or im.get("file_name")
                id2f[im["id"]] = fn
                t, h, a, w = fn.split("_")[:4]
                imgs.append(dict(file=fn, split=sp, night=t == "1", altitude=int(h), angle=int(a), rainy=w == "1"))
            for an in d.get("annotation", d.get("annotations", [])):
                x, y, w, h = an["bbox"]
                objs.append(dict(ann_id=an["id"], file=id2f[an["image_id"]], cls=cat[an["category_id"]],
                                 x1=float(x), y1=float(y), x2=float(x + w), y2=float(y + h)))
        # oriented boxes [xc, yc, w, h, angle]: true width and length of rotated objects
        rjobs = [(self.BASE + f"rotate_json/annotations/{sp}.json", _cache("hituav", "annotations", "rot_" + sp + ".json"))
                 for sp in splits]
        rot = {}
        for path in fetch_many(rjobs, 3):
            with open(path) as fh:
                d = json.load(fh)
            for an in d.get("annotations", d.get("annotation", [])):
                b = an["bbox"]
                rot[an["id"]] = (min(b[2], b[3]), max(b[2], b[3]), b[4] if len(b) > 4 else 0.0)
        self.images = pd.DataFrame(imgs).sort_values("file").reset_index(drop=True)
        o = pd.DataFrame(objs)
        o["w"], o["h"] = o.x2 - o.x1, o.y2 - o.y1
        o["width_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[0])     # oriented box: short side
        o["length_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[1])    # oriented box: long side
        self.objects = o.merge(self.images[["file", "night", "altitude", "angle"]], on="file", how="left")
        self._split = dict(zip(self.images.file, self.images.split))
        self._groups = {f: g for f, g in self.objects.groupby("file")}
        counts = self.objects.pivot_table(index="file", columns="cls", values="x1", aggfunc="count", fill_value=0)
        self.images = self.images.merge(counts, left_on="file", right_index=True, how="left").fillna(0)

    def _url(self, f):
        return self.BASE + f"normal_json/{self._split[f]}/{f}"

    def path(self, file):
        return fetch(self._url(file), _cache("hituav", self._split[file], file))

    def prefetch(self, files, threads=8):
        return fetch_many([(self._url(f), _cache("hituav", self._split[f], f)) for f in files], threads)

    def load(self, file):
        """Thermal image (H x W uint8) and the list of annotated objects."""
        img = cv2.imread(self.path(file), cv2.IMREAD_GRAYSCALE)
        g = self._groups.get(file)
        objs = [] if g is None else [dict(cls=r.cls, box=[r.x1, r.y1, r.x2, r.y2]) for r in g.itertuples()]
        return img, objs

    def video(self, name):
        return fetch(self.BASE + "video_sample/" + name, _cache("hituav", "video", name))

    @staticmethod
    def video_geometry(name):
        """Flight altitude, m, and camera angle, deg, from a video name such as '60m-30_1.mov'."""
        alt, rest = name.split("m-")
        return int(alt), int(rest.split("_")[0])

    def gsd(self, altitude, angle=90):
        """Ground sampling distance at the image centre, m/px (formula of Lesson 1.1)."""
        rng = altitude / max(math.sin(math.radians(angle)), 1e-3)
        return 2 * rng * math.tan(math.radians(self.CAMERA["hfov_deg"]) / 2) / self.CAMERA["width"]


class Kust4K:
    """Kust4K: pixel-aligned RGB and thermal (TIR) pairs 640x512 of a hexacopter
    HET-3021C over urban roads, day (id ...D) and night (id ...N), with pixel labels.

    The thermal images are pseudo-colour (palette from purple = cold to yellow = warm);
    tir_gray is their luminance, a relative thermal brightness. The data set contains
    frames with SIMULATED failures (visible channel set to zero, thermal channel
    blurred) made by its authors; they are excluded from the curated lists below."""

    PALETTE = "pseudo-colour (purple = cold, yellow = warm)"
    URL = "https://ndownloader.figshare.com/files/{}"
    FILES = {"rgb": 55979147, "tir": 55979165, "seg": 55979159}
    CLASSES = ("unlabelled", "road", "building", "motorcycle", "car", "truck", "tree", "person", "traffic facilities")
    COLORS = np.array([(0, 0, 0), (150, 150, 150), (229, 57, 53), (255, 152, 0), (255, 235, 59),
                       (236, 64, 122), (46, 160, 67), (0, 229, 255), (30, 136, 229)], np.uint8)
    OBJECTS = (3, 4, 5, 7)
    CAMERA = dict(platform="hexacopter HET-3021C", tir_focal_mm=9.1, tir_pixel_um=12.0,
                  rgb="1920x1080, f = 8 mm, registered to the thermal image and resized to 640x512")
    DAY = ("00720D", "00719D", "00031D", "01944D", "00851D", "00580D", "00990D", "00355D", "02094D", "02293D")
    NIGHT = ("02566N", "02807N", "02921N", "03135N", "03685N", "03895N", "03780N", "04004N", "02867N", "02599N")

    def __init__(self):
        self._z = {}

    def _zip(self, key):
        if key not in self._z:
            self._z[key] = RemoteZip(self.URL.format(self.FILES[key]), "kust4k_" + key)
        return self._z[key]

    def prefetch(self, ids, threads=6):
        names = [i + ".png" for i in ids]
        with ThreadPoolExecutor(3) as ex:
            list(ex.map(lambda k: self._zip(k).get(names, threads), ("rgb", "tir", "seg")))

    def pair(self, fid):
        """dict(id, night, rgb, tir, tir_gray, labels) for a frame id such as '00720D'."""
        self.prefetch([fid])
        name = fid + ".png"
        rgb = _rgb(self._zip("rgb").get(name))
        tir = _rgb(self._zip("tir").get(name))
        lab = cv2.imread(self._zip("seg").get(name), cv2.IMREAD_UNCHANGED)
        return dict(id=fid, night=fid.endswith("N"), rgb=rgb, tir=tir,
                    tir_gray=cv2.cvtColor(tir, cv2.COLOR_RGB2GRAY), labels=lab)


class VisDrone:
    """VisDrone2019-DET validation part: RGB images of drones with object boxes.
    Annotation line: x, y, w, h, score, category, truncation, occlusion."""

    URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
    CLASSES = ("ignored region", "pedestrian", "people", "bicycle", "car", "van", "truck", "tricycle",
               "awning-tricycle", "bus", "motor", "others")
    COLORS = {"pedestrian": (0, 229, 255), "people": (0, 176, 255), "bicycle": (118, 255, 3), "car": (255, 235, 59),
              "van": (255, 193, 7), "truck": (255, 152, 0), "tricycle": (186, 104, 200),
              "awning-tricycle": (156, 39, 176), "bus": (244, 67, 54), "motor": (236, 64, 122)}

    CURATED = ('0000213_03920_d_0000243', '0000291_03801_d_0000887', '0000117_00112_d_0000087', '0000001_08414_d_0000013', '0000155_00401_d_0000001', '0000193_01705_d_0000112', '0000295_02400_d_0000033', '0000289_04601_d_0000834', '0000215_00000_d_0000256', '0000327_04001_d_0000731')

    def __init__(self):
        self.z = RemoteZip(self.URL, "visdrone_val")

    def names(self):
        return sorted(n.split("/")[-1][:-4] for n in self.z.namelist() if n.endswith(".jpg"))

    def _members(self, name):
        return [f"VisDrone2019-DET-val/images/{name}.jpg", f"VisDrone2019-DET-val/annotations/{name}.txt"]

    def prefetch(self, names, threads=8):
        self.z.get([m for n in names for m in self._members(n)], threads)

    def load(self, name):
        """RGB image and the list of objects (ignored regions and 'others' are skipped)."""
        pi, pa = self.z.get(self._members(name))
        objs = []
        with open(pa) as fh:
            for line in fh:
                v = line.strip().strip(",").split(",")
                if len(v) < 8:
                    continue
                x, y, w, h, _, c, trunc, occ = (int(float(s)) for s in v[:8])
                if c in (0, 11):
                    continue
                objs.append(dict(cls=self.CLASSES[c], box=[x, y, x + w, y + h], truncation=trunc, occlusion=occ))
        return _rgb(pi), objs


class M3OT:
    """M3OT: synchronised visible (RGB) and thermal (IR) image sequences of two DJI Mavic 3T
    drones flying at 100-120 m (10 frames per second, 640 x 512 px), day, dusk and night, with
    multi-object tracking annotation of vehicles (MOT format: frame, id, x, y, w, h, flag, class,
    confidence). The track identifiers are the same in the RGB and IR sequences of one drone.
    The two channels were aligned by the authors with an affine transform; a residual spatial
    offset of several pixels and a time offset of up to several frames remain (Lesson 1.4).

    The IR frames are 8-bit grey images in the BLACK-HOT palette (warm = dark). The data
    descriptor does not state the palette; it is established from the scene: in all nine
    sequences of drone 1 the road surface is darker than vegetation by day, at dusk and at night,
    while asphalt is, as a rule, the warmer surface (Lesson 1.3, task 1.3.4). Vehicles are
    bright, i.e. cooler than the road. to_white_hot() gives the white-hot convention of HIT-UAV."""

    PALETTE = "black-hot"
    URL = "https://ndownloader.figshare.com/files/52023875"
    CAMERA = dict(platform="DJI Mavic 3T", altitude_m="100-120", fps=10, width=640, height=512,
                  rgb="telephoto camera, f = 29.85 mm (35 mm equivalent 162 mm), frames registered to IR",
                  ir="uncooled thermal camera 640 x 512, f = 9.1 mm (35 mm equivalent 40 mm)")
    SEQUENCES = {"1-01": "day, suburban road", "1-02": "night, city crossroads", "1-03": "day, city road",
                 "1-04": "day, suburban highway", "1-05": "day, crossroads", "1-06": "dusk, highway",
                 "1-07": "day, crossroads", "1-08": "dusk, crossroads", "1-09": "day, city street with parked cars"}

    def __init__(self):
        self.z = RemoteZip(self.URL, "m3ot")
        self._seq = {}
        for n in self.z.namelist():
            parts = n.split("/")        # M3OT/<drone>/<rgb|ir>/<split>/<sequence>/img1/<frame>.PNG
            if len(parts) == 7 and parts[5] == "img1":
                self._seq.setdefault((parts[4].rstrip("T"), parts[2]), []).append(n)
        for k in self._seq:
            self._seq[k].sort()

    def sequences(self):
        return sorted({k[0] for k in self._seq})

    def n_frames(self, seq, channel="rgb"):
        return len(self._seq[(seq, channel)])

    def frame_names(self, seq, channel="rgb", start=0, count=None, step=1):
        names = self._seq[(seq, channel)]
        end = len(names) if count is None else min(len(names), start + count * step)
        return names[start:end:step]

    def prefetch(self, seq, channel="rgb", start=0, count=None, step=1, threads=8):
        return self.z.get(self.frame_names(seq, channel, start, count, step), threads)

    def frames(self, seq, channel="rgb", start=0, count=None, step=1):
        """List of RGB images (H x W x 3) of a sequence; IR frames are grey images in 3 equal channels."""
        return [_rgb(p) for p in self.prefetch(seq, channel, start, count, step)]

    def tracks(self, seq, channel="rgb"):
        """Annotation table: frame (from 1), id, x, y, w, h, cx, cy."""
        import pandas as pd
        names = self._seq[(seq, channel)]
        gt = names[0].rsplit("/img1/", 1)[0] + "/gt/gt.txt"
        g = pd.read_csv(self.z.get(gt), header=None, names=["frame", "id", "x", "y", "w", "h", "flag", "cls", "conf"])
        g["cx"], g["cy"] = g.x + g.w / 2, g.y + g.h / 2
        return g[["frame", "id", "x", "y", "w", "h", "cx", "cy"]]

    def objects(self, table, frame):
        """Objects of one frame in the format of draw_boxes."""
        t = table[table.frame == frame]
        return [dict(cls="vehicle", id=int(r.id), box=[r.x, r.y, r.x + r.w, r.y + r.h]) for r in t.itertuples()]


class GOOSE:
    """GOOSE: front camera of a ground vehicle (2048x1000) with pixel labels of 64 classes."""

    URL = "https://goose-dataset.de/storage/goose_2d_val.zip"
    TRAVERSABLE = ("asphalt", "cobble", "gravel", "soil", "low_grass", "moss", "leaves", "snow",
                   "road_marking", "pedestrian_crossing", "sidewalk", "bikeway")
    CONDITIONAL = ("high_grass", "crops", "debris", "tree_root")
    IGNORE = ("undefined", "sky", "ego_vehicle", "outlier")
    CURATED = (
        'labels/val/2022-07-22_flight/2022-07-22_flight__0078_1658494246011438339_labelids.png',
        'labels/val/2022-07-22_flight/2022-07-22_flight__0158_1658494465090456964_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0528_1661860582736903436_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0551_1661860697315889089_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0002_1663755194931038715_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0038_1663755353192423465_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0114_1677850404544550420_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0156_1677850669766430912_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0007_1670420722571435228_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0057_1670421449778652057_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0037_1670421230548919820_labelids.png',
        'labels/val/2023-05-15_neubiberg_rain/2023-05-15_neubiberg_rain__0650_1684157928693659670_labelids.png',
    )

    def __init__(self):
        self.z = RemoteZip(self.URL, "goose_val")
        self.names, self.colors = {}, {}
        with open(self.z.get("goose_label_mapping.csv"), newline="") as fh:
            for row in csv.DictReader(fh):
                k = int(row["label_key"])
                h = row["hex"].lstrip("#")
                self.names[k] = row["class_name"]
                self.colors[k] = tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))
        self.lut = np.zeros((256, 3), np.uint8)
        for k, c in self.colors.items():
            self.lut[k] = c
        self.id_of = {v: k for k, v in self.names.items()}

    def frames(self):
        return sorted(n for n in self.z.namelist() if n.endswith("_labelids.png"))

    @staticmethod
    def _image_member(label_member):
        return label_member.replace("labels/", "images/").replace("_labelids.png", "_windshield_vis.png")

    def prefetch(self, members, threads=6):
        self.z.get([m for lm in members for m in (self._image_member(lm), lm)], threads)

    def load(self, label_member, scale=0.5):
        """RGB image and label map (both resized by `scale`)."""
        pi, pl = self.z.get([self._image_member(label_member), label_member])
        img, lab = _rgb(pi), cv2.imread(pl, cv2.IMREAD_UNCHANGED)
        if scale != 1:
            w, h = int(img.shape[1] * scale), int(img.shape[0] * scale)
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
            lab = cv2.resize(lab, (w, h), interpolation=cv2.INTER_NEAREST)
        return img, lab

    def traversability(self, lab):
        """0 = ignored (sky, own vehicle), 1 = traversable, 2 = conditionally traversable, 3 = obstacle."""
        lut = np.full(256, 3, np.uint8)
        for group, v in ((self.TRAVERSABLE, 1), (self.CONDITIONAL, 2), (self.IGNORE, 0)):
            for n in group:
                if n in self.id_of:
                    lut[self.id_of[n]] = v
        return lut[lab]


# =============================================================================
# 3. Camera geometry
# =============================================================================
def gsd_nadir(altitude, hfov_deg, width_px):
    """Ground sampling distance of a vertical view, m/px: GSD = 2 H tan(alpha/2) / N."""
    return 2 * altitude * math.tan(math.radians(hfov_deg) / 2) / width_px


def hfov_from_sensor(focal_mm, pixel_um, width_px):
    """Horizontal field of view, deg, from the focal length and the pixel pitch."""
    return math.degrees(2 * math.atan(width_px * pixel_um * 1e-3 / 2 / focal_mm))


# =============================================================================
# 4. Degradations of the video signal (applied to real frames)
# =============================================================================
def fractal_noise(shape, rng, octaves=5, start=3, persistence=0.5):
    """Smooth random field in [0, 1] (sum of up-sampled random grids)."""
    h, w = shape
    out = np.zeros((h, w), np.float32)
    amp, m = 1.0, max(h, w)
    for o in range(octaves):
        n = start * (2 ** o)
        grid = rng.random((max(2, int(round(n * h / m)) + 1), max(2, int(round(n * w / m)) + 1)), dtype=np.float32)
        out += amp * cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        amp *= persistence
    out -= out.min()
    return out / max(float(out.max()), 1e-6)


def _f(img):
    return img.astype(np.float32)


def _u8(x):
    return np.clip(x, 0, 255).astype(np.uint8)


def motion_blur(img, length=9.0, angle=0.0):
    """Linear motion blur: the point spread function is a line of `length` px at `angle` deg."""
    if length < 1.5:
        return img.copy()
    size = int(math.ceil(length)) | 1
    k = np.zeros((size * 4, size * 4), np.uint8)
    c = size * 2
    dx = math.cos(math.radians(angle)) * (length - 1) * 2
    dy = -math.sin(math.radians(angle)) * (length - 1) * 2
    cv2.line(k, (int(c - dx), int(c - dy)), (int(c + dx), int(c + dy)), 255, 4, cv2.LINE_AA)
    k = cv2.resize(k.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    return cv2.filter2D(img, -1, k / max(k.sum(), 1e-6), borderType=cv2.BORDER_REFLECT)


def defocus(img, sigma=1.5):
    return img.copy() if sigma <= 0 else cv2.GaussianBlur(img, (0, 0), sigma)


def add_noise(img, sigma=10.0, seed=None):
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def low_light(img, level=0.15, gain_max=8.0, seed=None):
    """Dark scene and automatic gain: light reduced by `level`, photon and read noise added."""
    if level >= 0.999:
        return img.copy()
    rng = np.random.default_rng(seed)
    lin = (_f(img) / 255.0) ** 2.2 * level
    full_well = 3000.0
    noisy = lin + rng.standard_normal(lin.shape).astype(np.float32) * np.sqrt(lin / full_well + (4.0 / full_well) ** 2)
    gain = float(np.clip(0.16 / max(float(lin.mean()), 1e-6), 1.0, gain_max))
    return _u8(np.clip(noisy * gain, 0, 1) ** (1 / 2.2) * 255)


def haze(img, density=0.4, airlight=200.0):
    """Uniform haze: I = J t + A (1 - t), transmission t = 1 - density."""
    t = 1.0 - float(np.clip(density, 0, 0.98))
    return _u8(_f(img) * t + airlight * (1 - t))


def smoke(img, density=0.5, seed=0, color=170.0, opacity=0.92):
    """Smoke: spatially variable veil."""
    if density <= 0:
        return img.copy()
    n = fractal_noise(img.shape[:2], np.random.default_rng(seed))
    m = np.clip((n - (1.0 - density)) * 3.0, 0, 1) * opacity
    if img.ndim == 3:
        m = m[..., None]
    return _u8(_f(img) * (1 - m) + color * m)


def jpeg(img, quality=30):
    """JPEG compression artefacts."""
    if quality >= 100:
        return img.copy()
    x = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode(".jpg", x, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    y = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    return y if img.ndim == 2 else cv2.cvtColor(y, cv2.COLOR_BGR2RGB)


def downscale(img, factor=0.5):
    """Loss of resolution: down-sample by `factor`, then up-sample back."""
    if factor >= 0.999:
        return img.copy()
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def rolling_shutter(img, amplitude=6.0, cycles=1.5, phase=0.0):
    """Rolling-shutter distortion: every row is shifted by a sinusoid of the row number."""
    if amplitude <= 0:
        return img.copy()
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    shift = amplitude * np.sin(2 * np.pi * cycles * yy / h + phase)
    return cv2.remap(img, xx + shift, yy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)


def analog_interference(img, strength=0.5, seed=None):
    """Analog FPV video link with a weak or jammed signal: colour loss, snow, line
    noise, horizontal tearing (loss of synchronisation) and a rolling brightness bar."""
    if strength <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    h = img.shape[0]
    x = _f(img)
    if x.ndim == 3:
        gray = x.mean(axis=2, keepdims=True)
        x = gray + (x - gray) * (1 - 0.85 * min(strength * 1.3, 1))
    x += rng.normal(0, 45 * strength, x.shape)
    for r in rng.integers(0, h, int(h * 0.06 * strength)):
        x[r] += rng.normal(70, 40) * strength
    if strength > 0.25:
        y0, band = rng.integers(0, h), int(h * 0.25 * strength)
        offs = np.cumsum(rng.normal(0, 3.0 * strength, band))
        for i in range(band):
            r = (y0 + i) % h
            x[r] = np.roll(x[r], int(offs[i] + 30 * strength), axis=0)
    bar = 1.0 + 0.3 * strength * np.sin(2 * np.pi * (np.arange(h, dtype=np.float32)[:, None] / h * 1.3 + rng.random()))
    return _u8(x * (bar if x.ndim == 2 else bar[..., None]))


def packet_loss(img, rate=0.05, block=16, seed=None):
    """Digital link with lost packets: damaged macroblock slices are filled with a smear
    of the row above or with a grey block."""
    if rate <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    out = img.copy()
    gh, gw = img.shape[0] // block, img.shape[1] // block
    lost = rng.random((gh, gw)) < rate
    for r in range(gh):
        c = 0
        while c < gw:
            if lost[r, c]:
                run = int(rng.integers(1, max(2, gw // 3)))
                lost[r, c:c + run] = True
                c += run
            c += 1
    for r, c in zip(*np.nonzero(lost)):
        ys, xs = slice(r * block, (r + 1) * block), slice(c * block, (c + 1) * block)
        if r > 0 and rng.random() < 0.6:
            out[ys, xs] = out[r * block - 1:r * block, xs]
        else:
            out[ys, xs] = 128 if img.ndim == 2 else (60, 128, 60)
    return out


def degrade(img, sensor="visible", motion_px=0.0, motion_angle=0.0, defocus_sigma=0.0, light=1.0, noise=0.0,
            haze_density=0.0, smoke_density=0.0, rolling=0.0, scale=1.0, jpeg_quality=100, analog=0.0,
            packet=0.0, seed=0):
    """Chain of degradations in physical order: atmosphere -> optics and motion -> sensor ->
    resolution -> compression -> radio link. For the thermal sensor haze and smoke are
    attenuated (long-wave infrared passes thin smoke and haze) and low light does not apply."""
    x, ir = img, sensor == "thermal"
    if haze_density > 0:
        x = haze(x, haze_density * (0.25 if ir else 1.0), airlight=128 if ir else 200)
    if smoke_density > 0:
        x = smoke(x, smoke_density, seed=seed, color=128 if ir else 170, opacity=0.18 if ir else 0.92)
    if motion_px > 0:
        x = motion_blur(x, motion_px, motion_angle)
    if defocus_sigma > 0:
        x = defocus(x, defocus_sigma)
    if rolling > 0:
        x = rolling_shutter(x, rolling)
    if light < 1.0 and not ir:
        x = low_light(x, light, seed=seed)
    if noise > 0:
        x = add_noise(x, noise, seed=seed + 1)
    if scale < 1.0:
        x = downscale(x, scale)
    if jpeg_quality < 100:
        x = jpeg(x, jpeg_quality)
    if analog > 0:
        x = analog_interference(x, analog, seed=seed + 2)
    if packet > 0:
        x = packet_loss(x, packet, seed=seed + 3)
    return x


# =============================================================================
# 5. Measures, classical detection, tracking
# =============================================================================
def gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def psnr(ref, test):
    """Peak signal-to-noise ratio, dB."""
    mse = float(np.mean((_f(ref) - _f(test)) ** 2))
    return float("inf") if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)


def ssim(ref, test):
    """Structural similarity index of grey levels (Gaussian window 11 x 11, sigma 1.5)."""
    a, b = _f(gray(ref)), _f(gray(test))
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    blur = lambda z: cv2.GaussianBlur(z, (11, 11), 1.5)
    ma, mb = blur(a), blur(b)
    va, vb, cab = blur(a * a) - ma * ma, blur(b * b) - mb * mb, blur(a * b) - ma * mb
    return float((((2 * ma * mb + c1) * (2 * cab + c2)) / ((ma * ma + mb * mb + c1) * (va + vb + c2))).mean())


def sharpness(img):
    """Variance of the Laplacian: a measure of focus (small values = blurred image)."""
    return float(cv2.Laplacian(gray(img), cv2.CV_64F).var())


def to_white_hot(img, palette):
    """Thermal frame in the white-hot convention (warm = bright) as a 3-channel uint8 image.
    palette: "white-hot" (HIT-UAV), "black-hot" (M3OT, inverted here) or a pseudo-colour palette
    whose luminance grows with the temperature (Kust4K: purple = cold, yellow = warm)."""
    g = gray(img)
    if palette.startswith("black"):
        g = 255 - g
    return np.dstack([g, g, g])


def vegetation_mask(rgb, exg_min=0.12, v_min=35, erode_px=15):
    """Vegetation of a visible RGB frame by the excess-green index ExG = (2G - R - B) / (R + G + B).
    The mask is eroded by erode_px so that a residual offset between the visible and the thermal
    channel does not move it onto neighbouring surfaces."""
    f = rgb.astype(np.float32)
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    exg = (2 * g - r - b) / (r + g + b + 1e-6)
    m = ((exg > exg_min) & (f.max(axis=2) > v_min)).astype(np.uint8)
    if erode_px > 1:
        m = cv2.erode(m, np.ones((erode_px, erode_px), np.uint8))
    return m.astype(bool)


def surroundings_mask(shape, boxes, scale=1.0, gap=3):
    """Background ring around annotated objects (e.g. the road surface around vehicles): every box
    [x1, y1, x2, y2] is expanded by scale x its longer side; the boxes themselves, expanded by gap px,
    are excluded."""
    ring = np.zeros(shape[:2], np.uint8)
    core = np.zeros(shape[:2], np.uint8)
    for x1, y1, x2, y2 in boxes:
        s = int(round(scale * max(x2 - x1, y2 - y1)))
        cv2.rectangle(ring, (int(x1) - s, int(y1) - s), (int(x2) + s, int(y2) + s), 1, -1)
        cv2.rectangle(core, (int(x1) - gap, int(y1) - gap), (int(x2) + gap, int(y2) + gap), 1, -1)
    return (ring > 0) & (core == 0)


def palette_check(thermal, road, vegetation, margin=15, min_pixels=200):
    """Palette of a thermal frame established from the scene. In a road scene the asphalt is, as a
    rule, warmer than vegetation (by day it absorbs most of the solar radiation, at night it gives off
    the stored heat), therefore a road darker than vegetation indicates the black-hot palette
    (warm = dark) and a brighter road the white-hot convention (warm = bright). A shaded or wet road
    and sunlit tree crowns can reverse the order in single frames: check several frames."""
    g = gray(thermal)
    road = road & ~vegetation
    nr, nv = int(road.sum()), int(vegetation.sum())
    r = float(np.median(g[road])) if nr >= min_pixels else float("nan")
    v = float(np.median(g[vegetation])) if nv >= min_pixels else float("nan")
    if np.isnan(r) or np.isnan(v):
        verdict = "not determined: too few road or vegetation pixels"
    elif r < v - margin:
        verdict = "road darker than vegetation: black-hot (warm = dark)"
    elif r > v + margin:
        verdict = "road brighter than vegetation: white-hot (warm = bright)"
    else:
        verdict = f"difference below {margin} grey levels: not determined"
    return dict(road_median=r, vegetation_median=v, road_pixels=nr, vegetation_pixels=nv, verdict=verdict)


def signal_to_clutter(img, mask, exclude=None, gap=2, ring=8):
    """Signal-to-clutter ratio of an object: |mean of object - mean of background ring| /
    standard deviation of the background ring (grey levels)."""
    g = _f(gray(img))
    m = mask.astype(np.uint8)
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * gap + 1, 2 * gap + 1))
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * (gap + ring) + 1, 2 * (gap + ring) + 1))
    bg = (cv2.dilate(m, k2) > 0) & (cv2.dilate(m, k1) == 0)
    if exclude is not None:
        bg &= ~exclude
    t, b = g[m > 0], g[bg]
    if t.size < 3 or b.size < 10:
        return float("nan")
    return abs(float(t.mean()) - float(b.mean())) / max(float(b.std()), 1.0)


def box_iou(a, b):
    """Intersection over union of two boxes [x1, y1, x2, y2]."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def boxes_from_labels(labels, class_ids, names, min_area=20):
    """Boxes of connected regions of the given classes in a label map."""
    out = []
    for c in class_ids:
        n, cc, stats, _ = cv2.connectedComponentsWithStats((labels == c).astype(np.uint8), connectivity=8)
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a >= min_area:
                out.append(dict(cls=names[c], class_id=int(c), box=[float(x), float(y), float(x + w), float(y + h)],
                                area=int(a)))
    return out


def patch_labels(labels, tile, object_ids, names, min_px=10):
    """Patch classification from a label map: every tile receives the object class with
    the largest number of pixels in it, or 'background'."""
    rows, cols = labels.shape[0] // tile, labels.shape[1] // tile
    grid = np.empty((rows, cols), object)
    for r in range(rows):
        for c in range(cols):
            t = labels[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile]
            n, k = max((int((t == k).sum()), k) for k in object_ids)
            grid[r, c] = names[k] if n >= min_px else "background"
    return grid


def detect_hotspots(img, k=5.0, se_px=21, min_area=4, max_area=5000):
    """Classical detector of small warm objects in a white-hot thermal image: white
    top-hat (local contrast) -> threshold mean + k std -> connected components.
    Returns boxes (N, 4) [x1, y1, x2, y2] and the binary mask."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    th = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, se).astype(np.float32)
    mask = (th > th.mean() + k * th.std()).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    boxes = [[x, y, x + w, y + h] for x, y, w, h, a in stats[1:] if min_area <= a <= max_area]
    return np.array(boxes, float).reshape(-1, 4), mask


def expected_size_px(gsd, size_m=0.45, margin_px=6.0):
    """Expected size of an object in an annotated box, px: size_m / GSD + margin_px.
    Defaults: nominal shoulder width of a person 0.45 m and a margin of 6 px (blur, thermal halo,
    annotation margin). For comparison, a least-squares fit to the HIT-UAV boxes of persons in
    vertical views gives d = 0.38 m and m = 6.6 px; block 1.1.1-B computes the fit for every class."""
    return size_m / gsd + margin_px


def detect_warm_objects(img, gsd, k=6.0, size_m=0.45, margin_px=6.0):
    """Hot-spot detector adapted to the ground sampling distance: the structuring element
    and the admissible blob areas follow the expected size of a person in pixels."""
    s = expected_size_px(gsd, size_m, margin_px)
    return detect_hotspots(img, k=k, se_px=int(max(7, round(2.5 * s))) | 1,
                           min_area=max(3, int((0.3 * s) ** 2)), max_area=int((2.0 * s) ** 2))


def match_detections(pred, objects, target_classes, pad=2.0):
    """A detection is correct when its centre lies inside a (padded) box of an object of a
    target class. Detections inside boxes of other annotated objects are counted apart.
    Returns dict(tp, fp, other, missed, recall, precision)."""
    tgt = [o for o in objects if o["cls"] in target_classes]
    oth = [o for o in objects if o["cls"] not in target_classes]
    used, tp, fp, other = set(), [], [], []
    inside = lambda o, cx, cy: (o["box"][0] - pad <= cx <= o["box"][2] + pad and o["box"][1] - pad <= cy <= o["box"][3] + pad)
    for b in pred:
        cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        hit = next((i for i, o in enumerate(tgt) if i not in used and inside(o, cx, cy)), None)
        if hit is not None:
            used.add(hit)
            tp.append(list(b))
        elif any(inside(o, cx, cy) for o in oth) or any(inside(o, cx, cy) for o in tgt):
            other.append(list(b))
        else:
            fp.append(list(b))
    n = len(tgt)
    return dict(tp=tp, fp=fp, other=other, missed=[o for i, o in enumerate(tgt) if i not in used],
                recall=len(tp) / n if n else float("nan"), precision=len(tp) / max(len(tp) + len(fp), 1))


def global_shift(prev, cur):
    """Translation of the image content from `prev` to `cur` (phase correlation), px."""
    (dx, dy), _ = cv2.phaseCorrelate(_f(gray(prev)), _f(gray(cur)))
    return dx, dy


class NearestNeighbourTracker:
    """Simple multi-object tracker: the detections of a frame are linked to the tracks of
    the previous frame by the smallest distance after compensation of the camera motion.
    A track is closed after `max_missed` frames without a detection."""

    def __init__(self, max_dist=12.0, max_missed=4):
        self.max_dist, self.max_missed = max_dist, max_missed
        self.tracks, self.closed, self.next_id = {}, {}, 1
        self.offset = np.zeros(2)                    # accumulated camera motion, px

    def update(self, centres, shift=(0.0, 0.0), frame=0):
        """centres: (N, 2) detections of the frame; shift: image motion since the previous frame.
        Returns the list of (track id, centre) of the frame."""
        self.offset += np.asarray(shift, float)
        pts = np.asarray(centres, float).reshape(-1, 2) - self.offset      # coordinates of the first frame
        ids = list(self.tracks)
        pairs = []
        for i, tid in enumerate(ids):
            last = self.tracks[tid]["hist"][-1][1]
            for j, p in enumerate(pts):
                d = float(np.hypot(*(p - last)))
                if d <= self.max_dist:
                    pairs.append((d, tid, j))
        pairs.sort()
        used_t, used_d, out = set(), set(), []
        for d, tid, j in pairs:
            if tid in used_t or j in used_d:
                continue
            used_t.add(tid)
            used_d.add(j)
            self.tracks[tid]["hist"].append((frame, pts[j]))
            self.tracks[tid]["missed"] = 0
            out.append((tid, pts[j] + self.offset))
        for tid in ids:
            if tid not in used_t:
                self.tracks[tid]["missed"] += 1
                if self.tracks[tid]["missed"] > self.max_missed:
                    self.closed[tid] = self.tracks.pop(tid)
        for j, p in enumerate(pts):
            if j not in used_d:
                self.tracks[self.next_id] = dict(hist=[(frame, p)], missed=0)
                out.append((self.next_id, p + self.offset))
                self.next_id += 1
        return out

    def history(self, tid):
        """Trajectory of a track in the coordinates of the current frame."""
        t = self.tracks.get(tid) or self.closed.get(tid)
        return np.array([p for _, p in t["hist"]]) + self.offset


def track_warm_objects(frames, gsd, fps, k=5.0, max_speed_ms=2.5, max_missed=4, min_len=12, min_disp_m=2.0):
    """Detection and tracking of warm objects in a thermal video:
    detect_warm_objects -> camera motion by phase correlation -> NearestNeighbourTracker.
    A track is reported as a moving object when it lasts at least `min_len` frames and its
    start and end points are at least `min_disp_m` apart on the ground.
    Returns (tracker, moving: {id: trajectory (K, 2) in the last frame}, rows: list of
    (frame, track id, x, y), detections per frame)."""
    tr = NearestNeighbourTracker(max_dist=max(5.0, max_speed_ms / gsd / max(fps, 1e-6) + 3.0), max_missed=max_missed)
    prev, rows, n_det = None, [], []
    for i, f in enumerate(frames):
        g = gray(f)
        b, _ = detect_warm_objects(g, gsd, k=k)
        c = np.c_[(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2] if len(b) else np.zeros((0, 2))
        shift = global_shift(prev, g) if prev is not None else (0.0, 0.0)
        for tid, p in tr.update(c, shift, i):
            rows.append((i, tid, float(p[0]), float(p[1])))
        n_det.append(len(b))
        prev = g
    moving = {}
    for tid, t in {**tr.closed, **tr.tracks}.items():
        pts = np.array([p for _, p in t["hist"]])
        if len(pts) >= min_len and float(np.hypot(*(pts[-1] - pts[0]))) * gsd >= min_disp_m:
            moving[tid] = pts + tr.offset
    return tr, moving, rows, n_det


def video_info(path):
    """Properties of a video file read by OpenCV."""
    cap = cv2.VideoCapture(path)
    info = dict(path=path, opened=cap.isOpened(), fps=cap.get(cv2.CAP_PROP_FPS),
                frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                codec=int(cap.get(cv2.CAP_PROP_FOURCC)).to_bytes(4, "little").decode("latin-1", "replace"))
    cap.release()
    info["duration_s"] = info["frames"] / info["fps"] if info["fps"] else float("nan")
    info["size_mb"] = os.path.getsize(path) / 1e6 if os.path.exists(path) else float("nan")
    return info


def iter_video(path, every_n=1, max_frames=None, start=0):
    """Generator of (frame number, timestamp s, RGB frame) of every n-th frame of a video:
    frames are decoded one by one, so a long video does not fill the memory."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    k, n_out = start, 0
    try:
        while True:
            ok, f = cap.read()
            if not ok or (max_frames is not None and n_out >= max_frames):
                break
            if (k - start) % every_n == 0:
                yield k, k / fps, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                n_out += 1
            k += 1
    finally:
        cap.release()


def write_video(path, frames, fps, codec="mp4v"):
    """Write RGB or grey frames to a video file; returns the path. Codec 'mp4v' (.mp4) works in
    Colab and in most OpenCV builds; 'MJPG' (.avi) stores every frame as JPEG."""
    h, w = frames[0].shape[:2]
    wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*codec), float(fps), (w, h))
    if not wr.isOpened():
        raise IOError(f"OpenCV cannot write the codec {codec}")
    for f in frames:
        f = to_rgb(f)
        if f.shape[:2] != (h, w):
            f = cv2.resize(f, (w, h))
        wr.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    wr.release()
    return path


def contact_sheet(frames, cols=6, width=1600, labels=None, font=0.5):
    """Grid of frames for a quick visual review of a sequence."""
    n = len(frames)
    rows = int(math.ceil(n / cols))
    tw = width // cols
    th = int(round(tw * frames[0].shape[0] / frames[0].shape[1]))
    sheet = np.zeros((rows * th, cols * tw, 3), np.uint8)
    for i, f in enumerate(frames):
        r, c = divmod(i, cols)
        t = cv2.resize(to_rgb(f), (tw, th), interpolation=cv2.INTER_AREA)
        if labels is not None:
            cv2.rectangle(t, (0, 0), (tw, 18), (0, 0, 0), -1)
            cv2.putText(t, str(labels[i]), (4, 13), cv2.FONT_HERSHEY_SIMPLEX, font * 0.8, (255, 255, 0), 1, cv2.LINE_AA)
        sheet[r * th:(r + 1) * th, c * tw:(c + 1) * tw] = t
    return sheet


def frame_difference(a, b):
    """Mean absolute difference of the grey levels of two frames."""
    return float(np.mean(np.abs(_f(gray(a)) - _f(gray(b)))))


def grad_magnitude(img, sigma=1.5):
    """Normalised gradient magnitude; edges of a visible and a thermal image of the same scene
    are similar even when their brightness is not, so it is used to compare the two channels."""
    g = cv2.GaussianBlur(_f(gray(img)), (0, 0), sigma)
    m = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    return (m - m.mean()) / (m.std() + 1e-6)


def channel_shift(img_a, img_b):
    """Translation between two images of different sensors (phase correlation of gradient
    magnitudes): (dx, dy, response). The response is low when the images have little common structure."""
    a, b = grad_magnitude(img_a), grad_magnitude(img_b)
    win = cv2.createHanningWindow((a.shape[1], a.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(a, b, win)
    return float(dx), float(dy), float(resp)


def track_offsets(table_a, table_b, lag=0):
    """Offsets of the centres of the same track (same id) in two annotated channels:
    channel b at frame t is compared with channel a at frame t + lag. Returns arrays dx, dy."""
    bb = table_b[["id", "frame", "cx", "cy"]].copy()
    bb["frame"] = bb["frame"] + lag
    m = bb.merge(table_a[["id", "frame", "cx", "cy"]], on=["id", "frame"], suffixes=("_b", "_a"))
    return (m.cx_b - m.cx_a).values, (m.cy_b - m.cy_a).values


def read_video(path, step=1, max_frames=None, start=0):
    """Frames of a video file (BGR from OpenCV converted to RGB) and the frame rate."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames, k = [], 0
    while True:
        ok, f = cap.read()
        if not ok or (max_frames and len(frames) >= max_frames):
            break
        if k % step == 0:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        k += 1
    cap.release()
    return frames, fps


# =============================================================================
# 6. Visualisation
# =============================================================================
def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img


def colorize(labels, colors):
    """Label map -> RGB image with the colour table `colors` (array K x 3)."""
    return np.asarray(colors, np.uint8)[np.clip(labels, 0, len(colors) - 1)]


def overlay(img, color_img, alpha=0.5, mask=None):
    base, col = _f(to_rgb(img)), _f(color_img)
    a = np.full(base.shape[:2], alpha, np.float32) if mask is None else mask.astype(np.float32) * alpha
    return _u8(base * (1 - a[..., None]) + col * a[..., None])


def draw_boxes(img, objects, colors=None, labels=True, scores=None, ids=None, thickness=1, font=0.4, color=None):
    """Draw boxes of objects (dicts with 'box' and 'cls')."""
    out = to_rgb(img).copy()
    for i, o in enumerate(objects):
        x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
        c = color if color is not None else (colors or {}).get(o["cls"], (255, 255, 0))
        c = tuple(int(v) for v in c)
        cv2.rectangle(out, (x1, y1), (x2, y2), c, thickness, cv2.LINE_AA)
        txt = []
        if labels:
            txt.append(str(o["cls"]))
        if ids is not None:
            txt.append(f"#{ids[i]}")
        if scores is not None:
            txt.append(f"{scores[i]:.2f}")
        if txt:
            s = " ".join(txt)
            (tw, th), _ = cv2.getTextSize(s, cv2.FONT_HERSHEY_SIMPLEX, font, 1)
            y0 = max(y1 - 3, th + 2)
            cv2.rectangle(out, (x1, y0 - th - 2), (x1 + tw + 2, y0 + 2), c, -1)
            cv2.putText(out, s, (x1 + 1, y0), cv2.FONT_HERSHEY_SIMPLEX, font, (0, 0, 0), 1, cv2.LINE_AA)
    return out


def legend(ax, names, colors, loc="lower right", fontsize=7, ncol=1):
    import matplotlib.patches as mpatches
    ax.legend(handles=[mpatches.Patch(color=np.array(c) / 255.0, label=n) for n, c in zip(names, colors)],
              loc=loc, fontsize=fontsize, ncol=ncol, framealpha=0.85)


def setup_plots():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8,
                         "ytick.labelsize": 8, "legend.fontsize": 8, "figure.facecolor": "white"})


def catalogue():
    """Table of the data sets of the course."""
    return [
        ("HIT-UAV", "UAV DJI M210 V2, thermal camera Zenmuse XT2 (640x512, 25 x 20 deg)",
         "2898 thermal images, 5 thermal videos; boxes of persons and vehicles; 60-130 m, day and night",
         "CC BY 4.0", "Suo J. et al., Scientific Data 10, 227 (2023)",
         "https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset"),
        ("Kust4K", "UAV hexacopter HET-3021C, visible + thermal cameras",
         "4024 pixel-aligned RGB-thermal pairs 640x512 over urban roads, day and night; pixel labels of 8 classes",
         "CC BY 4.0", "Scientific Data 12, 1701 (2025)", "https://doi.org/10.6084/m9.figshare.29476610"),
        ("VisDrone2019-DET (val)", "camera drones of different types, 14 Chinese cities",
         "548 RGB images; boxes of pedestrians, cars, vans, trucks, buses, motors and others",
         "research use, see the data set site", "Zhu P. et al., IEEE TPAMI 44(11), 7380-7399 (2022)",
         "https://github.com/VisDrone/VisDrone-Dataset"),
        ("GOOSE (val)", "ground vehicle, front RGB camera 2048x1000",
         "962 images of roads, field tracks, forests and a training area; pixel labels of 64 classes",
         "CC BY-SA 4.0", "IEEE ICRA 2024 (arXiv:2310.16788)", "https://goose-dataset.de"),
    ]


# =====================================================================================
# Topic 2. Digital image, video frame formation and preprocessing
# =====================================================================================
def quantize(img, bits):
    """Brightness quantised to `bits` bits per channel (2**bits levels), shown again in 0...255."""
    step = 256.0 / (2 ** int(bits))
    return np.clip(np.floor(img.astype(np.float32) / step) * step + step / 2, 0, 255).astype(np.uint8)


def entropy(img):
    """Entropy of the brightness histogram, bit (8 bit = all 256 levels used equally)."""
    hist = np.bincount(gray(img).ravel(), minlength=256).astype(float)
    p = hist[hist > 0] / hist.sum()
    return float(-(p * np.log2(p)).sum())


def rms_contrast(img):
    """RMS contrast: standard deviation of the grey levels."""
    return float(_f(gray(img)).std())


def average_gradient(img):
    """Average gradient: mean of sqrt((gx^2 + gy^2) / 2) of the grey levels (central differences)."""
    g = _f(gray(img))
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=1)
    return float(np.sqrt((gx ** 2 + gy ** 2) / 2.0).mean())


def hsv_mask(rgb, h=(35, 85), s=(40, 255), v=(40, 255)):
    """Pixels whose hue (degrees, 0...360), saturation and value (0...255) lie in the given ranges.
    OpenCV stores the hue of 8-bit images as degrees / 2 (0...179); a range with h[0] > h[1]
    passes through 0 degrees (red)."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hh = hsv[..., 0].astype(np.int32) * 2
    mh = ((hh >= h[0]) & (hh <= h[1])) if h[0] <= h[1] else ((hh >= h[0]) | (hh <= h[1]))
    return mh & (hsv[..., 1] >= s[0]) & (hsv[..., 1] <= s[1]) & (hsv[..., 2] >= v[0]) & (hsv[..., 2] <= v[1])


def mask_scores(pred, truth, valid=None):
    """IoU, precision and recall of a binary mask against a reference mask; pixels outside `valid` are ignored."""
    if valid is not None:
        pred, truth = pred & valid, truth & valid
    tp = int((pred & truth).sum())
    fp = int((pred & ~truth).sum())
    fn = int((~pred & truth).sum())
    return dict(iou=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))


def box_mask(shape, box):
    """Binary mask of a box [x1, y1, x2, y2]."""
    m = np.zeros(shape[:2], bool)
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = True
    return m


def colour_planes(rgb, thermal=None):
    """8-bit planes of the colour models of a frame: R, G, B, grey (luma Y = 0.299 R + 0.587 G + 0.114 B),
    H (degrees / 2), S, V and, if given, the thermal channel."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    planes = {"R": rgb[..., 0], "G": rgb[..., 1], "B": rgb[..., 2], "grey": cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY),
              "H": hsv[..., 0], "S": hsv[..., 1], "V": hsv[..., 2]}
    if thermal is not None:
        planes["thermal"] = gray(thermal)
    return planes


def wide_angle_lens(img, barrel=0.2, vignetting=0.0):
    """Wide-angle lens of an FPV camera: barrel distortion and vignetting. The output pixel at the
    normalised radius r (r = 1 in the corners) shows the point of the ideal image at r (1 + b r^2) / (1 + b):
    the corners stay in place, the centre is magnified by 1 + b, straight lines bend outwards.
    OpenCV describes the same distortion with a negative coefficient k1 of its camera model.
    Vignetting: the brightness is multiplied by 1 - v r^2."""
    h, w = img.shape[:2]
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0
    rmax = math.hypot(cx, cy)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    dx, dy = (xx - cx) / rmax, (yy - cy) / rmax
    r2 = dx * dx + dy * dy
    f = (1 + barrel * r2) / (1 + barrel)
    out = cv2.remap(img, (cx + dx * f * rmax).astype(np.float32), (cy + dy * f * rmax).astype(np.float32),
                    cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    if vignetting > 0:
        v = np.clip(1 - vignetting * r2, 0, 1)
        out = _u8(_f(out) * (v if out.ndim == 2 else v[..., None]))
    return out


def free_space_loss_db(distance_m, freq_mhz):
    """Free-space path loss, dB: 20 lg(d, km) + 20 lg(f, MHz) + 32.44."""
    return 20 * math.log10(max(distance_m, 1.0) / 1000.0) + 20 * math.log10(freq_mhz) + 32.44


def link_budget(p_tx_mw, distance_m, freq_mhz=5800.0, g_tx_dbi=2.0, g_rx_dbi=2.0, extra_loss_db=0.0,
                bandwidth_mhz=20.0, noise_figure_db=6.0, interference_db=0.0):
    """Line-of-sight radio link: received power P_rx = P_tx + G_tx + G_rx - L_fs - L_add, dBm; noise power
    N = -174 + 10 lg(B, Hz) + NF, dBm, raised by `interference_db` (jamming); carrier-to-noise ratio CNR = P_rx - N."""
    p_tx = 10 * math.log10(p_tx_mw)
    fspl = free_space_loss_db(distance_m, freq_mhz)
    p_rx = p_tx + g_tx_dbi + g_rx_dbi - fspl - extra_loss_db
    noise = -174.0 + 10 * math.log10(bandwidth_mhz * 1e6) + noise_figure_db + interference_db
    return dict(p_tx_dbm=p_tx, fspl_db=fspl, p_rx_dbm=p_rx, noise_dbm=noise, cnr_db=p_rx - noise)


def analog_link_frame(img, cnr_db, threshold_db=10.0, fm_gain_db=25.0, seed=0):
    """Frame at the output of an analog FM video receiver (ILLUSTRATIVE model of the course, not a standard):
    above the FM threshold the picture has Gaussian noise with SNR = CNR + fm_gain_db; below the threshold
    impulse noise ("sparkles"), loss of colour and loss of line synchronisation grow with the deficit
    threshold - CNR; at a deficit of 6 dB and more the picture is replaced by noise."""
    rng = np.random.default_rng(seed)
    x = _f(img)
    x = x + rng.normal(0, 255.0 * 10 ** (-(cnr_db + fm_gain_db) / 20.0), x.shape)
    deficit = threshold_db - cnr_db
    if deficit >= 6:
        return _u8(rng.normal(128, 60, img.shape))
    if deficit > 0:
        imp = rng.random(x.shape[:2]) < min(0.5, 0.012 * deficit ** 2)
        vals = rng.choice(np.array([0.0, 255.0], np.float32), size=int(imp.sum()))
        x[imp] = vals[:, None] if x.ndim == 3 else vals
        return analog_interference(_u8(x), strength=min(1.0, deficit / 6.0), seed=seed)
    return _u8(x)


def jpeg_for_budget(img, bytes_per_frame):
    """Highest JPEG quality (5...95) whose file of this frame fits into `bytes_per_frame`: intra-frame
    equivalent of a digital video link with a given bit rate. Returns (decoded frame, quality, bytes)."""
    bgr = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    best = None
    lo, hi = 5, 95
    while lo <= hi:
        q = (lo + hi) // 2
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        if len(buf) <= bytes_per_frame:
            best, lo = (q, buf), q + 1
        else:
            hi = q - 1
    if best is None:
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 5])
        best = (5, buf)
    dec = cv2.imdecode(best[1], cv2.IMREAD_UNCHANGED)
    return (dec if dec.ndim == 2 else cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)), best[0], len(best[1])


def digital_packet_loss_rate(cnr_db, required_db=8.0, width_db=0.4):
    """Packet loss rate of a digital link near its sensitivity (ILLUSTRATIVE logistic model of the "cliff
    effect"): 0.5 at CNR = required_db, 0.1 % at a margin of about 2.8 dB."""
    return float(1.0 / (1.0 + math.exp((cnr_db - required_db) / width_db)))


def noise_sigma(img, exclude_edges=0.1):
    """Standard deviation of the noise of a frame by the method of Immerkaer (1996): the frame is filtered with
    the mask [[1,-2,1],[-2,4,-2],[1,-2,1]], which suppresses the smooth structure of the scene, and
    sigma = sqrt(pi/2) / 6 * mean |filtered|. Edges and fine texture increase the estimate, therefore the
    share `exclude_edges` of the pixels with the strongest gradient of the smoothed frame is excluded.
    For a colour frame the estimate is the mean over the R, G, B channels (noise of one channel)."""
    if img.ndim == 3 and not (np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2])):
        return float(np.mean([noise_sigma(img[..., c], exclude_edges) for c in range(3)]))
    g = _f(gray(img))
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    r = np.abs(cv2.filter2D(g, -1, k))[1:-1, 1:-1]
    if exclude_edges > 0:
        sm = cv2.GaussianBlur(g, (0, 0), 2.0)           # edges of the scene, not of the noise
        gx = cv2.Sobel(sm, cv2.CV_32F, 1, 0, ksize=3)[1:-1, 1:-1]
        gy = cv2.Sobel(sm, cv2.CV_32F, 0, 1, ksize=3)[1:-1, 1:-1]
        gm = gx * gx + gy * gy
        r = r[gm <= np.quantile(gm, 1 - exclude_edges)]
    return float(math.sqrt(math.pi / 2) * r.mean() / 6.0)


def add_gaussian_noise(img, sigma, seed=0):
    """Additive Gaussian noise with the standard deviation sigma (grey levels)."""
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def add_impulse_noise(img, amount, seed=0):
    """Impulse ("salt and pepper") noise: the share `amount` of the pixels becomes black or white."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    m = rng.random(img.shape[:2]) < amount
    vals = rng.choice(np.array([0, 255], np.uint8), size=int(m.sum()))
    out[m] = vals[:, None] if img.ndim == 3 else vals
    return out


def equalize(img, method="clahe", clip=2.0, tiles=8):
    """Global histogram equalisation (method "he") or CLAHE ("clahe": tiles x tiles regions, histogram of every
    region clipped at clip x its mean height) of a grey frame or of the lightness L* of a colour frame (CIE L*a*b*)."""
    def one(g):
        if method == "he":
            return cv2.equalizeHist(g)
        return cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(tiles), int(tiles))).apply(g)
    if img.ndim == 2:
        return one(img)
    if np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2]):
        g = one(img[..., 0])
        return np.dstack([g, g, g])
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = one(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def letterbox(img, size=640, stride=32, auto=False, scaleup=True, color=114):
    """Letterbox transformation to the input of a detector: the frame is scaled by r = min(S / H, S / W)
    without a change of its aspect ratio and padded symmetrically with the grey value `color` to S x S
    (auto=True: only to the nearest multiple of `stride`). Returns the image, r and the padding (left, top)."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    if not scaleup:
        r = min(r, 1.0)
    nw, nh = int(round(w * r)), int(round(h * r))
    dw, dh = size - nw, size - nh
    if auto:
        dw, dh = dw % stride, dh % stride
    dw, dh = dw / 2.0, dh / 2.0
    if (nw, nh) != (w, h):
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    value = (color, color, color) if img.ndim == 3 else color
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=value), r, (left, top)


def boxes_to_input(boxes, r, pad):
    """Boxes [x1, y1, x2, y2] of the frame mapped to the letterboxed input: x' = r x + left, y' = r y + top."""
    return [[b[0] * r + pad[0], b[1] * r + pad[1], b[2] * r + pad[0], b[3] * r + pad[1]] for b in boxes]


def boxes_from_input(boxes, r, pad):
    """Inverse mapping of boxes of the letterboxed input to the frame: x = (x' - left) / r, y = (y' - top) / r."""
    return [[(b[0] - pad[0]) / r, (b[1] - pad[1]) / r, (b[2] - pad[0]) / r, (b[3] - pad[1]) / r] for b in boxes]


def to_input_tensor(img):
    """Standard input of a neural network: RGB, float32 in 0...1, layout N x C x H x W (N = 1)."""
    x = to_rgb(img).astype(np.float32) / 255.0
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])


def time_call(fn, repeats=5, warmup=1):
    """Median execution time of fn() in milliseconds."""
    for _ in range(warmup):
        fn()
    t = []
    for _ in range(max(1, repeats)):
        t0 = time.perf_counter()
        fn()
        t.append((time.perf_counter() - t0) * 1000.0)
    return float(np.median(t))


def step_overlap(tracks, step):
    """Overlap and displacement of every annotated object between the frames t and t + step of a track table
    (columns frame, id, x, y, w, h): IoU of its two boxes, displacement of the centre (px) and box size."""
    import pandas as pd
    a = tracks[["frame", "id", "x", "y", "w", "h"]]
    b = a.copy()
    b["frame"] = b["frame"] - step
    m = a.merge(b, on=["id", "frame"], suffixes=("", "_n"))
    ix = (np.minimum(m.x + m.w, m.x_n + m.w_n) - np.maximum(m.x, m.x_n)).clip(lower=0)
    iy = (np.minimum(m.y + m.h, m.y_n + m.h_n) - np.maximum(m.y, m.y_n)).clip(lower=0)
    inter = ix * iy
    union = m.w * m.h + m.w_n * m.h_n - inter
    return pd.DataFrame({"frame": m.frame, "id": m.id, "iou": inter / union.clip(lower=1e-6),
                         "displacement_px": np.hypot(m.x_n + m.w_n / 2 - m.x - m.w / 2, m.y_n + m.h_n / 2 - m.y - m.h / 2),
                         "size_px": np.minimum(m.w, m.h)})


# =====================================================================================
# Topic 2, practical lessons 2.3-2.4: noise models, quality and suitability measures
# =====================================================================================
def add_poisson_noise(img, photons_per_level=2.0, seed=0):
    """Photon (shot) noise: every value I is replaced by Poisson(k I) / k with k photons per grey level,
    so the variance of the noise equals I / k and grows with the brightness (signal-dependent noise)."""
    rng = np.random.default_rng(seed)
    k = max(float(photons_per_level), 1e-3)
    return _u8(rng.poisson(_f(img) * k) / k)


def add_column_noise(img, sigma=8.0, seed=0, row_sigma=0.0):
    """Fixed-pattern noise of a sensor read out by columns (stripes typical of uncooled thermal
    cameras between two non-uniformity corrections): every column receives its own offset N(0, sigma^2),
    optionally every row an offset N(0, row_sigma^2)."""
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    off = rng.normal(0, sigma, (1, w)) + (rng.normal(0, row_sigma, (h, 1)) if row_sigma > 0 else 0)
    x = _f(img)
    return _u8(x + (off[..., None] if x.ndim == 3 else off))


def column_correction(img, smooth_px=15):
    """Correction of column stripes (scene-based non-uniformity correction, simplest form): the mean of
    every column is compared with a smoothed profile of the column means; the difference is the offset of
    the column and is subtracted. Smooth structures of the scene wider than smooth_px are kept."""
    x = _f(img)
    planes = [x] if x.ndim == 2 else [x[..., c] for c in range(x.shape[2])]
    out = []
    for p in planes:
        m = p.mean(axis=0)
        ms = cv2.GaussianBlur(m.reshape(1, -1).astype(np.float32), (0, 0), smooth_px).ravel()
        out.append(p - (m - ms)[None, :])
    return _u8(out[0] if x.ndim == 2 else np.dstack(out))


def edge_f1(ref, test, low=60, high=140, tol=1):
    """Agreement of the Canny edges of a test frame with the edges of a reference frame: an edge pixel is
    matched when the other map has an edge within tol pixels; returns dict(precision, recall, f1)."""
    e1 = cv2.Canny(gray(ref), low, high) > 0
    e2 = cv2.Canny(gray(test), low, high) > 0
    k = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8)
    d1 = cv2.dilate(e1.astype(np.uint8), k) > 0
    d2 = cv2.dilate(e2.astype(np.uint8), k) > 0
    p = float((e2 & d1).sum()) / max(int(e2.sum()), 1)
    r = float((e1 & d2).sum()) / max(int(e1.sum()), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-9))


def rank_correlation(a, b, sample=200000, seed=0):
    """Spearman rank correlation of the grey levels of two frames of the same scene (pixel by pixel).
    A global monotonic mapping of the brightness (stretching, gamma, global HE) gives 1 up to ties;
    a local mapping (CLAHE) changes the order of brightness between regions and gives less than 1."""
    x, y = gray(a).ravel().astype(np.float64), gray(b).ravel().astype(np.float64)
    if x.size > sample:
        idx = np.random.default_rng(seed).choice(x.size, sample, replace=False)
        x, y = x[idx], y[idx]
    def ranks(v):
        order = np.argsort(v, kind="mergesort")
        r = np.empty(v.size, np.float64)
        r[order] = np.arange(v.size)
        _, inv, cnt = np.unique(v, return_inverse=True, return_counts=True)
        sums = np.bincount(inv, weights=r)
        return (sums / cnt)[inv]                       # mean rank of tied values
    rx, ry = ranks(x), ranks(y)
    return float(np.corrcoef(rx, ry)[0, 1])


def hue_shift(ref, test, s_min=40, v_min=40):
    """Mean absolute change of the hue (degrees, circular) of the saturated pixels between two colour frames."""
    h1 = cv2.cvtColor(to_rgb(ref), cv2.COLOR_RGB2HSV)
    h2 = cv2.cvtColor(to_rgb(test), cv2.COLOR_RGB2HSV)
    m = (h1[..., 1] >= s_min) & (h1[..., 2] >= v_min)
    if not m.any():
        return float("nan")
    d = np.abs(h1[..., 0][m].astype(np.int32) - h2[..., 0][m].astype(np.int32)) * 2
    return float(np.minimum(d, 360 - d).mean())


def linear_stretch(img, p_low=1.0, p_high=99.0):
    """Global linear stretch of the brightness between two percentiles to 0...255 (the operation of the
    automatic gain control of a thermal camera); monotonic, keeps the order of brightness."""
    g = gray(img)
    lo, hi = np.percentile(g, [p_low, p_high])
    hi = max(hi, lo + 1)
    lut = np.clip((np.arange(256) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


def gamma_correction(img, gamma=0.6):
    """Global power-law mapping I' = 255 (I / 255)^gamma; gamma < 1 brightens the dark part of the range."""
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** float(gamma), 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


def ambe(a, b):
    """Absolute mean brightness error |mean(b) - mean(a)| of the grey levels."""
    return abs(float(gray(b).mean()) - float(gray(a).mean()))


# =============================================================================
# 9. Topic 3: edges, binarisation, contours, ground geometry, FPV video
# =============================================================================

def label_boundaries(labels, ignore=(0,)):
    """Boundaries between regions of a label image (pixel labels of a data set): a pixel is a boundary
    pixel when one of its 4 neighbours has another label; pixels of the classes in `ignore` are excluded."""
    lab = np.asarray(labels)
    b = np.zeros(lab.shape, bool)
    b[:, 1:] |= lab[:, 1:] != lab[:, :-1]
    b[:, :-1] |= lab[:, 1:] != lab[:, :-1]
    b[1:, :] |= lab[1:, :] != lab[:-1, :]
    b[:-1, :] |= lab[1:, :] != lab[:-1, :]
    for c in ignore:
        b &= lab != c
    return b


def edge_map_scores(pred, ref, tol=2):
    """Agreement of two binary edge maps within `tol` pixels: precision (share of predicted edge pixels near a
    reference edge), recall (share of reference edge pixels near a predicted edge) and F1."""
    p, r = np.asarray(pred) > 0, np.asarray(ref) > 0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol + 1, 2 * tol + 1))
    rd = cv2.dilate(r.astype(np.uint8), k) > 0
    pd_ = cv2.dilate(p.astype(np.uint8), k) > 0
    prec = float((p & rd).sum()) / max(int(p.sum()), 1)
    rec = float((r & pd_).sum()) / max(int(r.sum()), 1)
    return dict(precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-9),
                edge_share=float(p.mean()))


def otsu(img, mask=None):
    """Otsu threshold computed by its definition: for every threshold t the between-class variance
    sigma_B^2(t) = w0(t) w1(t) (mu0(t) - mu1(t))^2 of the histogram is computed, and the threshold with the
    largest value is chosen. Returns (t, sigma_B^2 for t = 0...255, histogram)."""
    g = gray(img)
    v = g[mask > 0] if mask is not None else g.ravel()
    h = np.bincount(v.ravel(), minlength=256).astype(np.float64)
    p = h / max(h.sum(), 1)
    w0 = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    mt = mu[-1]
    w1 = 1 - w0
    with np.errstate(divide="ignore", invalid="ignore"):
        sb = (mt * w0 - mu) ** 2 / (w0 * w1)
    sb = np.nan_to_num(sb, nan=0.0, posinf=0.0)
    return int(np.argmax(sb)), sb, h


def oblique_gsd(altitude, angle, rows, height=512, vfov_deg=20.0, hfov_deg=25.0, width=640):
    """Ground sampling distance of a tilted camera in the image rows, m/px: (across, along).
    The depression angle of the line of sight of row y is delta = angle + (y - H/2)/H * vfov
    (angle = 90 deg for a vertical view); the slant range is h / sin(delta), the footprint of a pixel
    across the line of sight is range * IFOV and along it range * IFOV / sin(delta)."""
    rows = np.asarray(rows, float)
    dep = np.radians(np.clip(angle + (rows - height / 2.0) / height * vfov_deg, 1.0, 179.0))
    rng = altitude / np.sin(dep)
    across = rng * 2 * math.tan(math.radians(hfov_deg) / 2) / width
    return across, across / np.sin(dep)


def camera_matrix(width, height, hfov_deg, vfov_deg=None):
    """Pinhole camera matrix K from the size of the image and the field of view."""
    fx = width / 2.0 / math.tan(math.radians(hfov_deg) / 2)
    fy = fx if vfov_deg is None else height / 2.0 / math.tan(math.radians(vfov_deg) / 2)
    return np.array([[fx, 0, (width - 1) / 2.0], [0, fy, (height - 1) / 2.0], [0, 0, 1.0]])


def ground_homography(altitude, angle, K, size, gsd_out=None, max_range_m=None):
    """Homography that maps an image of a tilted camera to a top view of the flat ground (perspective
    transformation of Lesson 3.3). The camera is at the height h over the plane Z = 0 and looks forward
    with the depression angle `angle` (90 = vertical view); image x is the ground X axis.
    H_g = K [r1, r2, -h r3] maps the ground point (X, Y, 1) to the image; the returned matrix maps
    image pixels to pixels of the top view with the step gsd_out, m/px (forward = up).
    Returns (H, (width, height) of the top view, gsd_out, extent (Xmin, Xmax, Ymin, Ymax) in metres)."""
    th = math.radians(angle)
    R = np.array([[1, 0, 0], [0, -math.sin(th), -math.cos(th)], [0, math.cos(th), -math.sin(th)]], float)
    Hg = K @ np.column_stack([R[:, 0], R[:, 1], -altitude * R[:, 2]])
    Hi = np.linalg.inv(Hg)
    w, h = size
    corners = np.array([[0, 0, 1], [w - 1, 0, 1], [w - 1, h - 1, 1], [0, h - 1, 1]], float).T
    g = Hi @ corners
    X, Y = g[0] / g[2], g[1] / g[2]
    if max_range_m:
        Y = np.minimum(Y, max(float(max_range_m), float(Y.min()) + 20.0))    # at least 20 m beyond the nearest row
    if gsd_out is None:
        gsd_out = altitude / math.sin(th) / K[0, 0]          # GSD across the line of sight at the image centre
    x0, x1, y0, y1 = X.min(), X.max(), Y.min(), Y.max()
    W, Ht = int(math.ceil((x1 - x0) / gsd_out)), int(math.ceil((y1 - y0) / gsd_out))
    S = np.array([[1 / gsd_out, 0, -x0 / gsd_out], [0, -1 / gsd_out, y1 / gsd_out], [0, 0, 1]])
    return S @ Hi, (W, Ht), gsd_out, (x0, x1, y0, y1)


def transform_boxes(boxes, H):
    """Boxes [x1, y1, x2, y2] after the perspective transformation H (box of the four mapped corners)."""
    out = []
    for b in boxes:
        c = np.array([[b[0], b[1], 1], [b[2], b[1], 1], [b[2], b[3], 1], [b[0], b[3], 1]], float).T
        m = H @ c
        x, y = m[0] / m[2], m[1] / m[2]
        out.append([x.min(), y.min(), x.max(), y.max()])
    return np.array(out, float).reshape(-1, 4)


def thermal_mask(img, polarity="warm", method="top-hat + Otsu", se_px=31, k=4.0, block=51, offset=-5):
    """Binary mask of thermal signatures. polarity: 'warm' (brighter than the surroundings in a white-hot
    frame), 'cold' (darker) or 'both'. method: 'Otsu (global)', 'top-hat + Otsu', 'top-hat + mean + k*std',
    'adaptive (local mean)'. The white top-hat I - open(I) keeps bright details smaller than the structuring
    element se_px, the black top-hat close(I) - I keeps dark details."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    parts = (["warm"] if polarity == "warm" else ["cold"] if polarity == "cold" else ["warm", "cold"])
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    out = np.zeros(g.shape, np.uint8)
    for pol in parts:
        src = g if pol == "warm" else 255 - g
        if method.startswith("Otsu"):
            _, m = cv2.threshold(src, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif method.startswith("adaptive"):
            m = cv2.adaptiveThreshold(src, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, int(block) | 1, offset)
        else:
            th = cv2.morphologyEx(src, cv2.MORPH_TOPHAT, se)
            if "Otsu" in method:
                _, m = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            else:
                t = th.astype(np.float32)
                m = ((t > t.mean() + k * t.std()) * 255).astype(np.uint8)
        out |= m
    return out


def contour_features(c, gsd_across=None, gsd_along=None):
    """Geometric characteristics of a contour (OpenCV point array): area and perimeter, px; box;
    length and width of the minimum-area rectangle; aspect ratio L/W; extent = area / box area;
    solidity = area / area of the convex hull; circularity 4 pi A / P^2 (1 for a circle).
    With the GSD of the row of the centroid (m/px across and along the line of sight) the contour is
    scaled to metres and its area, length and width in metres are added."""
    a = float(cv2.contourArea(c))
    p = float(cv2.arcLength(c, True))
    x, y, w, h = cv2.boundingRect(c)
    m = cv2.moments(c)
    cx, cy = (m["m10"] / m["m00"], m["m01"] / m["m00"]) if m["m00"] > 0 else (x + w / 2, y + h / 2)
    (_, _), (rw, rh), ang = cv2.minAreaRect(c)
    L, W = max(rw, rh), min(rw, rh)
    hull = float(cv2.contourArea(cv2.convexHull(c)))
    f = dict(cx=cx, cy=cy, box=[x, y, x + w, y + h], area_px=a, perimeter_px=p, length_px=L, width_px=W,
             aspect=L / max(W, 1e-6), extent=a / max(w * h, 1e-6), solidity=a / max(hull, 1e-6),
             circularity=4 * math.pi * a / max(p * p, 1e-6))
    if gsd_across is not None:
        pts = c.reshape(-1, 2).astype(np.float32) * np.float32([gsd_across, gsd_along])
        (_, _), (mw, mh), _ = cv2.minAreaRect(pts)
        f.update(area_m2=a * gsd_across * gsd_along, length_m=max(mw, mh), width_m=min(mw, mh))
    return f


class TIIRacing:
    """TII-RATM ("Race Against the Machine"): flights of an FPV racing quadrotor in an indoor arena with
    racing gates, piloted by human FPV pilots and autonomous, speeds above 21 m/s. Camera Arducam IMX219,
    640 x 480 px, 120 frames per second (125 measured from the time stamps), wide-angle lens with a field of
    view of 155 x 115 deg, tilted up by 30-50 deg; IMU 500 Hz (gyroscope, accelerometer), motion capture,
    labels of the gates. Bosello M. et al. IEEE Robotics and Automation Letters 9(4), 3799-3806 (2024).
    Licence CC BY 4.0.

    The frames of a flight are stored in random order in a zip archive inside the release archive. The first
    call of clip() downloads the camera archive of the flight once (about 200 MB, transferred in parallel
    parts) and keeps in the cache the frames of the requested interval and of the preset SEGMENTS."""

    RELEASE = "https://github.com/tii-racing/drone-racing-dataset/releases/download/v3.0.0/"
    RAW = "https://raw.githubusercontent.com/tii-racing/drone-racing-dataset/main/"
    # local header offset and compressed size of the members in the release archive piloted.zip
    FLIGHTS = {"flight-16p-trackRATM": dict(asset="piloted_zipchunk01", camera=(1574086012, 201903685),
                                            camera_csv=(1549423339, 33752), imu=(1546851674, 488328),
                                            meta=(1574085613, 258))}
    SEGMENTS = {"flight-16p-trackRATM": {"A: take-off and first gates (0.5-4.5 s)": (0.5, 4.5),
                                         "B: slalom between gates (15.0-19.0 s)": (15.0, 19.0)}}
    ROTATION_KEY = {50: "trackRATM", 40: "lemniscate", 60: "piloted"}

    def __init__(self, flight="flight-16p-trackRATM"):
        if flight not in self.FLIGHTS:
            raise ValueError(f"flight {flight} is not prepared; available: {list(self.FLIGHTS)}")
        self.flight = flight
        self.info = self.FLIGHTS[flight]
        self.url = self.RELEASE + self.info["asset"]

    def _member(self, key):
        off, csize = self.info[key]
        path = _cache("tii", self.flight, key + ".bin")
        if os.path.exists(path):
            return open(path, "rb").read()
        raw = _read_url(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, min_bytes=30 + csize)
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {key}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = zlib.decompressobj(-15).decompress(raw[30 + n + m: 30 + n + m + csize])
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return data

    def metadata(self):
        """Flight metadata (YAML of the data set) as a flat dict."""
        out, sect = {}, ""
        for line in self._member("meta").decode().splitlines():
            if not line.strip() or ":" not in line:
                continue
            key, val = line.split(":", 1)
            ind = len(key) - len(key.lstrip())
            key = key.strip()
            if ind == 0:
                sect = key
            full = key if ind == 0 else f"{sect}.{key}"
            val = val.strip()
            if val:
                try:
                    val = float(val) if "." in val else int(val)
                except ValueError:
                    pass
                out[full] = val
        return out

    def frame_table(self):
        """Table of the camera frames: index, file, timestamp_us, t (s from the first frame)."""
        import pandas as pd
        t = pd.read_csv(io.BytesIO(self._member("camera_csv")))
        t["index"] = np.arange(len(t))
        t["t"] = (t.timestamp - t.timestamp.iloc[0]) / 1e6
        return t.rename(columns={"filename": "file"})[["index", "file", "timestamp", "t"]]

    def imu(self):
        """IMU table: timestamp_us, t (s from the first camera frame), accel_x/y/z m/s^2, gyro_x/y/z rad/s
        (axes of the body of the drone)."""
        import pandas as pd
        d = pd.read_csv(io.BytesIO(self._member("imu")))
        d["t"] = (d.timestamp - self.frame_table().timestamp.iloc[0]) / 1e6
        return d

    def calibration(self):
        """Camera matrix K and distortion coefficients D (pinhole model with radial and tangential terms)
        of the piloted trackRATM flights."""
        path = fetch(self.RAW + "camera_calibration/calib_p-trackRATM.json", _cache("tii", "calib_p-trackRATM.json"))
        d = json.load(open(path))
        return np.array(d["mtx"], float), np.array(d["dist"], float).ravel()

    def camera_to_body(self):
        """Rotation matrix R (3 x 3) from the camera axes (x right, y down, z forward) to the body axes of
        the drone, for the camera angle of the flight (file drone_to_camera.json of the data set)."""
        path = fetch(self.RAW + "camera_calibration/drone_to_camera.json", _cache("tii", "drone_to_camera.json"))
        q = json.load(open(path))["rotation"][self.ROTATION_KEY.get(int(self.metadata().get("camera.angle", 50)), "trackRATM")]
        x, y, z, w = q["x"], q["y"], q["z"], q["w"]
        return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                         [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                         [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])

    def _download_camera_zip(self, path, part=16 << 20, threads=6):
        off, csize = self.info["camera"]
        hdr = _read_url(self.url, {"Range": f"bytes={off}-{off + 29}"}, min_bytes=30)
        n, m = struct.unpack("<HH", hdr[26:30])
        start = off + 30 + n + m
        pieces = [(start + p, min(start + csize, start + p + part) - 1) for p in range(0, csize, part)]
        with open(path, "wb") as fh:
            fh.truncate(csize)

        def get(rg):
            a, b = rg
            data = _read_url(self.url, {"Range": f"bytes={a}-{b}"}, timeout=120, min_bytes=b - a + 1)
            with open(path, "r+b") as fh:
                fh.seek(a - start)
                fh.write(data)
            return len(data)

        with ThreadPoolExecutor(threads) as ex:
            return sum(ex.map(get, pieces))

    def _extract(self, wanted):
        """Stream the camera archive of the flight and keep the frames with the indices in `wanted`."""
        tmp = _cache("tii", self.flight, "_camera_deflate.part")
        self._download_camera_zip(tmp)
        d = zlib.decompressobj(-15)
        buf, kept = b"", 0
        with open(tmp, "rb") as fh:
            while True:
                chunk = fh.read(4 << 20)
                buf += d.decompress(chunk) if chunk else d.flush()
                while len(buf) >= 30 and buf[:4] == b"PK\x03\x04":
                    flag, comp = struct.unpack("<HH", buf[6:10])
                    csz, usz, nl, el = struct.unpack("<IIHH", buf[18:30])
                    need = 30 + nl + el + csz
                    if len(buf) < need:
                        break
                    name = buf[30:30 + nl].decode("utf-8", "replace")
                    data = buf[30 + nl + el:need]
                    buf = buf[need:]
                    base = name.rsplit("/", 1)[-1]
                    if base.endswith(".jpg") and int(base.split("_")[0]) in wanted:
                        if comp == 8:
                            data = zlib.decompressobj(-15).decompress(data)
                        with open(_cache("tii", self.flight, "frames", base), "wb") as out:
                            out.write(data)
                        kept += 1
                if not chunk or (len(buf) >= 4 and buf[:4] == b"PK\x01\x02"):
                    break
        os.remove(tmp)
        return kept

    def clip(self, t0, t1, step=1):
        """Frames (RGB, 480 x 640 x 3) of the interval t0...t1 s with every `step`-th frame, and their table."""
        tab = self.frame_table()
        sel = tab[(tab.t >= t0) & (tab.t < t1)].iloc[::max(1, int(step))]
        miss = [i for i, f in zip(sel["index"], sel.file) if not os.path.exists(_cache("tii", self.flight, "frames", f))]
        if miss:
            wanted = set(miss)
            for a, b in self.SEGMENTS.get(self.flight, {}).values():
                wanted |= set(tab[(tab.t >= a) & (tab.t < b)]["index"])
            self._extract(wanted)
        frames = [_rgb(_cache("tii", self.flight, "frames", f)) for f in sel.file]
        return frames, sel.reset_index(drop=True)


def undistort_maps(K, D, size, alpha=0.0):
    """Maps of cv2.remap that remove the lens distortion, and the new camera matrix.
    alpha = 0 keeps only valid pixels (the image is cropped), alpha = 1 keeps the whole field of view."""
    newK, _ = cv2.getOptimalNewCameraMatrix(K, D, size, alpha, size)
    m1, m2 = cv2.initUndistortRectifyMap(K, D, None, newK, size, cv2.CV_16SC2)
    return m1, m2, newK


def gyro_rotation(t, w, t1, t2):
    """Rotation vector (rad) of the body between the times t1 and t2 obtained by integration of the angular
    rate w (N x 3, rad/s) measured at the times t (s): R = prod exp([w_k dt_k])."""
    m = (t >= t1) & (t <= t2)
    idx = np.where(m)[0]
    if idx.size == 0:
        return np.zeros(3)
    tt = np.concatenate([[t1], t[idx], [t2]])
    ww = np.vstack([w[max(idx[0] - 1, 0)], w[idx], w[idx[-1]]])
    R = np.eye(3)
    for k in range(len(tt) - 1):
        dt = tt[k + 1] - tt[k]
        if dt <= 0:
            continue
        Rk, _ = cv2.Rodrigues((ww[k] * dt).reshape(3, 1))
        R = R @ Rk
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()


def rotation_from_homography(H, K):
    """Rotation of the camera between two frames from their homography, assuming a pure rotation:
    H = K R K^-1, so R = K^-1 H K scaled to det = 1 and projected on the nearest rotation matrix.
    Returns the rotation vector (rad) of R in the axes of the camera."""
    if H is None or not np.all(np.isfinite(H)):
        return np.full(3, np.nan)
    R = np.linalg.inv(K) @ H @ K
    d = np.linalg.det(R)
    if not np.isfinite(d) or abs(d) < 1e-12:
        return np.full(3, np.nan)                          # degenerate homography: no rotation can be computed
    R = R / np.cbrt(d)
    U, _, Vt = np.linalg.svd(R)
    R = U @ Vt
    if np.linalg.det(R) < 0:
        R = -R
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()


def smooth_rotations(rotvecs, sigma):
    """Smoothed orientations of the camera: rotvecs (N x 3) are absolute orientations; every orientation is
    replaced by the Gaussian-weighted mean of its neighbours (sigma in frames) computed in the tangent space
    of the orientation itself. Returns N x 3 rotation vectors."""
    Rs = [cv2.Rodrigues(np.asarray(r, float).reshape(3, 1))[0] for r in rotvecs]
    n = len(Rs)
    half = int(max(1, round(3 * sigma)))
    out = []
    for i in range(n):
        a, b = max(0, i - half), min(n, i + half + 1)
        wts = np.exp(-0.5 * ((np.arange(a, b) - i) / max(sigma, 1e-6)) ** 2)
        wts /= wts.sum()
        acc = np.zeros(3)
        for j, wj in zip(range(a, b), wts):
            d, _ = cv2.Rodrigues(Rs[i].T @ Rs[j])
            acc += wj * d.ravel()
        Rm = Rs[i] @ cv2.Rodrigues(acc.reshape(3, 1))[0]
        out.append(cv2.Rodrigues(Rm)[0].ravel())
    return np.array(out)


def interframe_psnr(frames, valid=None):
    """Inter-frame transformation fidelity (ITF): mean PSNR between consecutive frames, dB; a stabilised
    video has a higher ITF. `valid` (optional list of masks) limits the comparison to valid pixels."""
    vals = []
    for k in range(1, len(frames)):
        a, b = _f(gray(frames[k - 1])), _f(gray(frames[k]))
        m = np.ones(a.shape, bool) if valid is None else (valid[k - 1] & valid[k])
        if m.sum() < 100:
            continue
        mse = float(((a - b)[m] ** 2).mean())
        vals.append(10 * math.log10(255.0 ** 2 / max(mse, 1e-6)))
    return float(np.mean(vals)) if vals else float("nan")
'''

with open(os.path.join(WORK_DIR, "aitbc.py"), "w", encoding="utf-8") as fh:
    fh.write(LIB_SOURCE)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
import aitbc
aitbc = importlib.reload(aitbc)
aitbc.setup_plots()

if os.environ.get("AITBC_DATA"):
    cache_dir = os.environ["AITBC_DATA"]
elif CACHE_ON_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    cache_dir = "/content/drive/MyDrive/AITBC/data_cache"
else:
    cache_dir = os.path.join(WORK_DIR, "aitbc_data")
aitbc.set_data_root(cache_dir)

t0 = time.time()
hit, kust, m3ot = aitbc.HITUAV(), aitbc.Kust4K(), aitbc.M3OT()
tii = aitbc.TIIRacing("flight-16p-trackRATM")
KUST_EDGE = list(aitbc.Kust4K.DAY[:6]) + list(aitbc.Kust4K.NIGHT[:6])          # 6 day and 6 night pairs
_im = hit.images
_busy = (_im.Person + _im.Car) >= 4
HIT_EVAL_NIGHT = _im[(_im.split == "test") & _im.night & _busy].sample(40, random_state=21).file.tolist()
HIT_EVAL_DAY = _im[(_im.split == "test") & ~_im.night & _busy].sample(24, random_state=21).file.tolist()
HIT_POLARITY = _im[_im.split != "train"].sample(160, random_state=11).file.tolist()
HIT_OBLIQUE = ["0_80_40_0_06933.jpg", "0_120_30_0_09449.jpg", "1_110_30_0_02555.jpg", "0_100_40_0_03347.jpg"]
if PREFETCH_ALL:
    kust.prefetch(KUST_EDGE)
    hit.prefetch(HIT_EVAL_NIGHT + HIT_EVAL_DAY + HIT_POLARITY + HIT_OBLIQUE)
    for f in (300, 500):
        m3ot.prefetch("1-08", "ir", start=f, count=1)
VERSIONS = {"Python": platform.python_version(), "NumPy": np.__version__, "OpenCV": cv2.__version__,
            "pandas": pd.__version__, "Matplotlib": matplotlib.__version__, "aitbc": aitbc.__version__}
n_files, size_mb = aitbc.cache_size()
display(pd.DataFrame({"Value": list(VERSIONS.values()) + ["Google Colab" if IN_COLAB else "local Jupyter",
                                                           f"{aitbc.DATA_ROOT}: {n_files} files, {size_mb:.0f} MB",
                                                           f"{time.time() - t0:.0f} s"]},
                     index=list(VERSIONS) + ["environment", "cache of the data sets", "preparation time"]))

### Block 0.2. Course project

**Purpose.** To connect the course project created in Notebook 02. The results of this notebook are saved in it: figures and tables in `outputs`, the FPV video fragment in `data/video`, the stabilised video in `outputs/video` and the parameters of the lessons in `project.json`.

**Parameters.** `USE_DRIVE` — the project is on Google Drive (Colab asks for access to the Google account); otherwise it is on the disk of the session. `PROJECT_PATH` — folder of the project; the same as in Notebooks 02–04.

**Output.** The folders of the project and the lessons already recorded in `project.json`.

In [ ]:
#@title 0.2 Course project { display-mode: "form" }
USE_DRIVE = True #@param {type:"boolean"}
PROJECT_PATH = "AITBC/AITBC_project" #@param {type:"string"}

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    PROJECT_ROOT = os.path.join("/content/drive/MyDrive", PROJECT_PATH)
else:
    PROJECT_ROOT = os.path.join(WORK_DIR, PROJECT_PATH)
    if USE_DRIVE:
        print("Google Drive is available only in Google Colab; the project is used in", PROJECT_ROOT)
for d in ["data/images", "data/video", "data/thermal", "data/annotations", "frames/visible", "frames/thermal",
          "models", "outputs/figures", "outputs/video", "outputs/reports"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
FIG_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures")
REPORT_DIR = os.path.join(PROJECT_ROOT, "outputs", "reports")
VIDEO_DIR = os.path.join(PROJECT_ROOT, "outputs", "video")


def update_project_json(lesson, extra=None):
    path = os.path.join(PROJECT_ROOT, "project.json")
    meta = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else {"project": os.path.basename(PROJECT_ROOT)}
    meta.setdefault("lessons", [])
    if lesson not in meta["lessons"]:
        meta["lessons"].append(lesson)
    meta["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    meta["versions"] = VERSIONS
    meta.update(extra or {})
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(meta, fh, indent=2, ensure_ascii=False, default=str)
    print(f"project.json updated: lesson {lesson}" + (f", keys: {', '.join(extra)}" if extra else ""))


meta = json.load(open(os.path.join(PROJECT_ROOT, "project.json"), encoding="utf-8")) if os.path.exists(os.path.join(PROJECT_ROOT, "project.json")) else {}
print("Project:", PROJECT_ROOT)
print("Lessons recorded in project.json:", ", ".join(meta.get("lessons", [])) or "none (new project)")

---
# Lesson 3.1 (Lecture). Extraction of edges, silhouettes and contours of objects by classical methods

**Learning questions**
1. The concept of the edge of an object in an image and the basic approaches to its algorithmic extraction.
2. Extraction of the edges of an object by the Sobel and Canny methods with regard to their practical application.
3. Methods of binarisation of an image, including the global threshold processing by the Otsu method.
4. Contour analysis and computation of the geometric characteristics of the extracted objects.

**Classical methods** compute features of an image by fixed formulas: the gradient of brightness, a threshold, a contour and its geometric characteristics. They need no training data, work in milliseconds on an onboard computer and their result can be explained step by step. Their parameters (thresholds, sizes of windows) are chosen by the developer; the lecture shows by measurements on real frames how strongly the result depends on them.

In [ ]:
#@title Frames of Lesson 3.1 (run once) { display-mode: "form" }
SOURCES_31 = ["Kust4K: day, visible", "Kust4K: night, visible", "Kust4K: day, thermal (luminance of the palette)",
              "Kust4K: night, thermal (luminance of the palette)", "HIT-UAV: night, thermal (white-hot)",
              "HIT-UAV: day, thermal (white-hot)", "M3OT 1-08: dusk, thermal (black-hot)"]


def frame_31(source, number=1):
    # dict(img, grey, labels, objects, caption, altitude, angle) of a real frame of a data set
    k = int(number) - 1
    if source.startswith("Kust4K"):
        fid = (aitbc.Kust4K.DAY if "day" in source else aitbc.Kust4K.NIGHT)[k % 6]
        p = kust.pair(fid)
        img = p["rgb"] if "visible" in source else p["tir_gray"]
        return dict(img=img, grey=aitbc.gray(img), labels=p["labels"], objects=[], altitude=None, angle=None,
                    caption=f"Kust4K {fid} ({'night' if p['night'] else 'day'}), {'visible' if 'visible' in source else 'thermal'}")
    if source.startswith("HIT-UAV"):
        files = HIT_EVAL_NIGHT if "night" in source else HIT_EVAL_DAY
        f = files[k % len(files)]
        g, objs = hit.load(f)
        r = hit.images.set_index("file").loc[f]
        return dict(img=g, grey=g, labels=None, objects=objs, altitude=int(r.altitude), angle=int(r.angle),
                    caption=f"HIT-UAV {f}: {r.altitude} m, camera angle {r.angle}°, {'night' if r.night else 'day'}")
    f0 = (300, 500)[k % 2]
    g = aitbc.to_white_hot(m3ot.frames("1-08", "ir", start=f0, count=1)[0], "black-hot")[..., 0]
    return dict(img=g, grey=g, labels=None, objects=m3ot.objects(m3ot.tracks("1-08", "ir"), f0 + 1), altitude=None, angle=None,
                caption=f"M3OT 1-08 (dusk), thermal frame {f0 + 1}, inverted to white-hot")


print("Sources of Lesson 3.1:", "; ".join(SOURCES_31))

## Block 3.1.1. The edge of an object and the gradient of brightness

**Edge.** An edge is a line in the image along which the brightness changes abruptly: the boundary between an object and its background, between two surfaces or the boundary of a shadow. In a thermal frame an edge separates surfaces of different temperature or emissivity. The edge is found where the derivative of the brightness is large. A digital image is a grid of samples; the derivative is replaced by a difference, combined with smoothing across the direction of the difference (Sobel operator):

$$K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \quad K_y = K_x^{\mathsf T}, \qquad G_x = I * K_x, \quad G_y = I * K_y, \tag{1}$$

$$|G| = \sqrt{G_x^2 + G_y^2}, \qquad \theta = \operatorname{atan2}(G_y, G_x), \tag{2}$$

where $*$ is the convolution, $|G|$ the magnitude and $\theta$ the direction of the gradient; the edge runs perpendicular to $\theta$.

**Parameters.** `SOURCE`, `NUMBER` — frame; `BLUR` — size of the Gaussian smoothing before the operator (0 — without it); `SOBEL_KSIZE` — size of the Sobel kernel (1 — a central difference without smoothing); `ROW` — row of the profile as a share of the height of the frame.

**Expected result.** The frame, $G_x$, $G_y$, $|G|$, the direction $\theta$ in colour (brightness = magnitude), the profile of the brightness and of $|G|$ along the chosen row.

**What to observe.** $G_x$ responds to vertical edges and $G_y$ to horizontal ones; the sign shows whether the brightness grows or falls. The magnitude is large not only at the boundaries of objects but also at road markings, shadows and texture of vegetation. Without smoothing, the noise of the sensor gives small gradients in every pixel of a thermal frame. A larger kernel suppresses the noise and makes the edges thicker. The operator gives a value in every pixel, so a threshold or a thinning of the edges is needed to obtain lines (block 3.1.2).

**Control question 1.** Why is the derivative of the brightness computed with smoothing across its direction? What happens to the magnitude of the gradient on an edge after inversion of the palette of a thermal frame (white-hot → black-hot)?

In [ ]:
#@title 3.1.1 Gradient of brightness: Sobel operator { run: "auto" }
SOURCE = "Kust4K: day, visible" #@param ["Kust4K: day, visible", "Kust4K: night, visible", "Kust4K: day, thermal (luminance of the palette)", "Kust4K: night, thermal (luminance of the palette)", "HIT-UAV: night, thermal (white-hot)", "HIT-UAV: day, thermal (white-hot)", "M3OT 1-08: dusk, thermal (black-hot)"]
NUMBER = 1 #@param {type:"slider", min:1, max:6, step:1}
BLUR = 3 #@param [0, 3, 5, 7] {type:"raw"}
SOBEL_KSIZE = 3 #@param [1, 3, 5, 7] {type:"raw"}
ROW = 0.5 #@param {type:"slider", min:0.05, max:0.95, step:0.05}

fr = frame_31(SOURCE, NUMBER)
g = fr["grey"]
gs = cv2.GaussianBlur(g, (BLUR, BLUR), 0) if BLUR else g
gx = cv2.Sobel(gs, cv2.CV_32F, 1, 0, ksize=SOBEL_KSIZE)                  # formula (1)
gy = cv2.Sobel(gs, cv2.CV_32F, 0, 1, ksize=SOBEL_KSIZE)
mag, theta = np.hypot(gx, gy), np.arctan2(gy, gx)                         # formula (2)
hsv = np.dstack([((theta + np.pi) / (2 * np.pi) * 179).astype(np.uint8), np.full(g.shape, 255, np.uint8),
                 np.clip(mag / np.percentile(mag, 99) * 255, 0, 255).astype(np.uint8)])
direction = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
y = int(ROW * (g.shape[0] - 1))

fig, ax = plt.subplots(2, 3, figsize=(18, 9.5))
lim = np.percentile(np.abs(np.r_[gx.ravel(), gy.ravel()]), 99)
panels = [(fr["img"], "frame", None), (gx, "G_x (red: brightness grows to the right)", "RdBu_r"),
          (gy, "G_y (red: brightness grows downwards)", "RdBu_r"), (mag, "magnitude |G|", "magma"),
          (direction, "direction θ (colour), magnitude (brightness)", None)]
for a, (im_, t_, cm) in zip(ax.ravel(), panels):
    if cm == "RdBu_r":
        a.imshow(im_, cmap=cm, vmin=-lim, vmax=lim)
    elif cm == "magma":
        a.imshow(im_, cmap=cm, vmax=np.percentile(im_, 99))
    else:
        a.imshow(im_, cmap="gray" if im_.ndim == 2 else None)
    a.axhline(y, color="cyan", lw=0.8)
    a.set_title(t_, fontsize=10)
    a.axis("off")
a = ax[1, 2]
a.plot(g[y].astype(float), color="0.3", lw=0.9, label="brightness I")
a2 = a.twinx()
a2.plot(mag[y], color="C3", lw=0.9, label="|G|")
a.set_xlabel("column x")
a.set_ylabel("brightness, grey levels")
a2.set_ylabel("|G|", color="C3")
a.set_title(f"Profile of row y = {y}", fontsize=10)
a.grid(alpha=0.3)
fig.suptitle(fr["caption"], fontsize=11)
plt.tight_layout()
plt.show()
print(f"|G|: median {np.median(mag):.1f}, 90th percentile {np.percentile(mag, 90):.1f}, 99th percentile {np.percentile(mag, 99):.1f} "
      f"(the strongest edges are few: the median is {np.percentile(mag, 99) / max(np.median(mag), 1e-6):.0f} times smaller than the 99th percentile)")

## Block 3.1.2. Edges by the Sobel and Canny methods: comparison with the boundaries of the annotation

**Canny method.** (a) Gaussian smoothing; (b) gradient (1)–(2); (c) *non-maximum suppression*: a pixel is kept when its $|G|$ is not smaller than the values of its two neighbours along the direction $\theta$ — the edges become one pixel thick; (d) *hysteresis* with two thresholds:

$$|G| \ge T_{high} \;\Rightarrow\; \text{strong edge}, \qquad T_{low} \le |G| < T_{high} \;\Rightarrow\; \text{edge only if connected to a strong edge}. \tag{3}$$

A frequently used rule for the thresholds is $T_{low} = (1 - 0.33)\,m$, $T_{high} = (1 + 0.33)\,m$, where $m$ is the median brightness of the frame.

**Reference.** The pixel labels of Kust4K give the boundaries between the labelled regions (road, building, car, tree, person…). A detected edge pixel is correct when a boundary pixel lies within the tolerance $\tau$:

$$P = \frac{|E \cap B_\tau|}{|E|}, \qquad R = \frac{|B \cap E_\tau|}{|B|}, \qquad F_1 = \frac{2PR}{P + R}, \tag{4}$$

where $E$ is the set of detected edge pixels, $B$ the set of boundary pixels and the index $\tau$ means the dilation by $\tau$ pixels.

**Parameters.** `SOURCE`, `NUMBER` — frame of Kust4K; `CANNY_LOW`, `CANNY_HIGH` — thresholds (3); `MEDIAN_RULE` — thresholds from the median of the frame; `BLUR`; `L2_GRADIENT` — magnitude by (2) instead of $|G_x| + |G_y|$; `SOBEL_SHARE` — share of the pixels with the largest $|G|$ kept as Sobel edges; `TOLERANCE_PX` — $\tau$ in (4).

**Expected result.** The reference boundaries, the edges of Canny and of the Sobel threshold, the measures (4) for the frame and a table for 12 frames (6 day, 6 night, visible and thermal): fixed thresholds, the median rule and the best threshold chosen separately for every frame.

**What to observe.** Both methods find many edges that are not boundaries of labelled regions (road markings, shadows, texture), therefore $P$ is low: with the thresholds 50/150 $F_1$ is 0.33–0.48 in the four groups of frames, and even the best threshold chosen for every frame separately gives 0.41–0.61. No pair of thresholds suits all frames: the best $T_{low}$ ranges from 5 to 100 levels between the frames. The median rule does not solve this problem: in all four groups it is below the best threshold of the frame, most of all for night thermal frames (0.41 against 0.61). At night the thermal frame gives better edges of roads and buildings than the visible one ($F_1$ 0.48 against 0.36 with the fixed thresholds), whose median brightness is only about 17 levels. These limits of fixed parameters are one of the reasons for the methods with training of Topics 4 and 5.

**Control question 2.** Why does non-maximum suppression make the edges thin? What does the hysteresis (3) give compared with one threshold?

In [ ]:
#@title 3.1.2 Sobel and Canny compared with the boundaries of the annotation { run: "auto" }
SOURCE = "Kust4K: day, visible" #@param ["Kust4K: day, visible", "Kust4K: night, visible", "Kust4K: day, thermal (luminance of the palette)", "Kust4K: night, thermal (luminance of the palette)"]
NUMBER = 1 #@param {type:"slider", min:1, max:6, step:1}
CANNY_LOW = 50 #@param {type:"slider", min:0, max:250, step:5}
CANNY_HIGH = 150 #@param {type:"slider", min:5, max:255, step:5}
MEDIAN_RULE = False #@param {type:"boolean"}
BLUR = 5 #@param [0, 3, 5, 7] {type:"raw"}
L2_GRADIENT = True #@param {type:"boolean"}
SOBEL_SHARE = 0.10 #@param {type:"slider", min:0.02, max:0.3, step:0.01}
TOLERANCE_PX = 3 #@param {type:"slider", min:1, max:6, step:1}


def canny_edges(g, low, high, median_rule=False):
    gs = cv2.GaussianBlur(g, (BLUR, BLUR), 0) if BLUR else g
    if median_rule:
        m = float(np.median(gs))
        low, high = int(max(0, 0.67 * m)), int(min(255, 1.33 * m))
    return cv2.Canny(gs, low, max(high, low + 1), L2gradient=L2_GRADIENT), (low, high)


def sobel_edges(g, share):
    gs = cv2.GaussianBlur(g, (BLUR, BLUR), 0) if BLUR else g
    mag = np.hypot(cv2.Sobel(gs, cv2.CV_32F, 1, 0), cv2.Sobel(gs, cv2.CV_32F, 0, 1))
    return (mag > np.percentile(mag, 100 * (1 - share))).astype(np.uint8) * 255


fr = frame_31(SOURCE, NUMBER)
ref = aitbc.label_boundaries(fr["labels"], ignore=(0,))
ec, (lo, hi) = canny_edges(fr["grey"], CANNY_LOW, CANNY_HIGH, MEDIAN_RULE)
es = sobel_edges(fr["grey"], SOBEL_SHARE)
sc, ss = aitbc.edge_map_scores(ec, ref, TOLERANCE_PX), aitbc.edge_map_scores(es, ref, TOLERANCE_PX)   # formula (4)

fig, ax = plt.subplots(1, 4, figsize=(19, 4.4))
for a, (im_, t_) in zip(ax, ((fr["img"], fr["caption"]), (ref, "boundaries of the labelled regions (reference)"),
                             (ec, f"Canny {lo}/{hi}: P = {sc['precision']:.2f}, R = {sc['recall']:.2f}, F1 = {sc['f1']:.2f}"),
                             (es, f"Sobel, {SOBEL_SHARE:.0%} largest |G|: F1 = {ss['f1']:.2f}"))):
    a.imshow(im_, cmap="gray" if im_.ndim == 2 else None)
    a.set_title(t_, fontsize=9)
    a.axis("off")
plt.tight_layout()
plt.show()

rows = []
for fid in KUST_EDGE:
    p = kust.pair(fid)
    b = aitbc.label_boundaries(p["labels"], ignore=(0,))
    for ch, im_ in (("visible", p["rgb"]), ("thermal", p["tir_gray"])):
        gg = aitbc.gray(im_)
        f_fixed = aitbc.edge_map_scores(canny_edges(gg, CANNY_LOW, CANNY_HIGH)[0], b, TOLERANCE_PX)
        f_med = aitbc.edge_map_scores(canny_edges(gg, 0, 0, True)[0], b, TOLERANCE_PX)["f1"]
        best = max((aitbc.edge_map_scores(canny_edges(gg, t, 2 * t)[0], b, TOLERANCE_PX)["f1"], t)
                   for t in (5, 10, 20, 30, 40, 60, 80, 100, 130))
        rows.append(dict(time="night" if p["night"] else "day", channel=ch, median_brightness=float(np.median(gg)),
                         P_fixed=f_fixed["precision"], R_fixed=f_fixed["recall"], F1_fixed=f_fixed["f1"], F1_median_rule=f_med,
                         F1_best_of_frame=best[0], best_T_low=best[1],
                         F1_Sobel=aitbc.edge_map_scores(sobel_edges(gg, SOBEL_SHARE), b, TOLERANCE_PX)["f1"]))
tab = pd.DataFrame(rows)
summary = tab.groupby(["time", "channel"]).median(numeric_only=True)
summary["best_T_low_range"] = tab.groupby(["time", "channel"]).best_T_low.agg(lambda s: f"{s.min()}–{s.max()}")
display(summary.round(3).style.set_caption(f"Medians over 6 frames of every group; fixed thresholds {CANNY_LOW}/{CANNY_HIGH}, τ = {TOLERANCE_PX} px"))
tab.to_csv(os.path.join(REPORT_DIR, "lesson31_edges.csv"), index=False)

## Block 3.1.3. Binarisation of an image, global threshold of Otsu

**Binarisation** divides the pixels into two classes by a threshold $t$: $b(x, y) = 1$ if $I(x, y) > t$, otherwise 0. **Otsu's method** chooses the threshold that separates the histogram into two classes with the largest between-class variance:

$$t^* = \arg\max_t \sigma_B^2(t), \qquad \sigma_B^2(t) = \omega_0(t)\,\omega_1(t)\,\big(\mu_0(t) - \mu_1(t)\big)^2, \tag{5}$$

where $\omega_0, \omega_1$ are the shares of the pixels below and above $t$ and $\mu_0, \mu_1$ their mean brightness. The method has no parameters and works well when the histogram has two modes of comparable size, for example a warm road and a cool field.

**Small targets.** A person seen from 100 m occupies a few tens of pixels — far less than 1 % of the frame — and does not form its own mode of the histogram. Local contrast is therefore extracted first: the *white top-hat* $I - (I \circ S)$ (difference between the frame and its morphological opening with a structuring element $S$ larger than the target) keeps only bright details smaller than $S$, and the threshold is applied to it (Lesson 3.3).

**Parameters.** `SOURCE`, `NUMBER` — thermal frame; `PREPROCESSING` — none or white top-hat; `TOP_HAT_M` — size of $S$ in metres (converted to pixels with the GSD of the frame of HIT-UAV).

**Expected result.** The frame, its histogram with the curve $\sigma_B^2(t)$ and the threshold $t^*$, the binary mask; for HIT-UAV — the share of the annotated persons and vehicles whose centre lies in the mask. The threshold of formula (5) is compared with `cv2.threshold(..., THRESH_OTSU)`.

**What to observe.** Without preprocessing, Otsu's threshold divides the frame into large warm and cool areas (roads, roofs, fields): in the default night frame the mask covers 60 % of the frame (43–81 % in six night frames of the set). Most persons lie in this mask, but together with the warm ground, so the mask does not separate them. After the top-hat with $S$ = 3 m the histogram is dominated by the background close to zero, and the mask covers 1–11 % of the frame; in the default frame it still contains 9 of the 10 annotated persons. In the full pipeline on the 40 night test frames of HIT-UAV (block 3.3.4) the global Otsu threshold gives the recall of persons 0.25, the top-hat with a threshold 0.84.

**Control question 3.** For which histogram does Otsu's method give a good threshold? Why is it not suitable for small warm targets without preprocessing?

In [ ]:
#@title 3.1.3 Otsu's threshold { run: "auto" }
SOURCE = "HIT-UAV: night, thermal (white-hot)" #@param ["HIT-UAV: night, thermal (white-hot)", "HIT-UAV: day, thermal (white-hot)", "Kust4K: night, thermal (luminance of the palette)", "M3OT 1-08: dusk, thermal (black-hot)"]
NUMBER = 1 #@param {type:"slider", min:1, max:6, step:1}
PREPROCESSING = "none" #@param ["none", "white top-hat"]
TOP_HAT_M = 3.0 #@param {type:"slider", min:1, max:15, step:0.5}

fr = frame_31(SOURCE, NUMBER)
g = cv2.GaussianBlur(fr["grey"], (3, 3), 0)
if PREPROCESSING == "white top-hat":
    gsd_c = hit.gsd(fr["altitude"], fr["angle"]) if fr["altitude"] else 0.15
    se_px = max(3, int(round(TOP_HAT_M / gsd_c))) | 1
    src = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (se_px, se_px)))
    note = f"white top-hat, S = {TOP_HAT_M:g} m = {se_px} px"
else:
    src, note = g, "without preprocessing"
t_star, sigma_b, hist = aitbc.otsu(src)                                   # formula (5)
t_cv, MASK_313 = cv2.threshold(src, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
FRAME_313 = fr

fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
ax[0].imshow(fr["img"], cmap="gray")
ax[0].set_title(fr["caption"], fontsize=9)
ax[1].bar(np.arange(256), hist / hist.sum(), width=1.0, color="0.55")
ax[1].set_yscale("log")
a2 = ax[1].twinx()
a2.plot(sigma_b, color="C3", lw=1.2)
a2.set_ylabel("σ_B²(t)", color="C3")
ax[1].axvline(t_star, color="C3", ls="--")
ax[1].set_title(f"Histogram ({note}) and σ_B²(t): t* = {t_star}", fontsize=9)
ax[1].set_xlabel("grey level t")
ax[1].set_ylabel("share of pixels (logarithmic)")
ax[2].imshow(MASK_313, cmap="gray")
ax[2].set_title(f"mask I > t*: {np.mean(MASK_313 > 0):.1%} of the pixels", fontsize=9)
for a in (ax[0], ax[2]):
    a.axis("off")
plt.tight_layout()
plt.show()

rows = {"threshold t* by formula (5)": t_star, "threshold of cv2.threshold (THRESH_OTSU)": int(t_cv),
        "share of the pixels in the mask": f"{np.mean(MASK_313 > 0):.1%}"}
for cls in ("Person", "Car", "vehicle"):
    obs = [o for o in fr["objects"] if o["cls"] == cls]
    if obs:
        inside = [MASK_313[int(min(max((o["box"][1] + o["box"][3]) / 2, 0), MASK_313.shape[0] - 1)),
                           int(min(max((o["box"][0] + o["box"][2]) / 2, 0), MASK_313.shape[1] - 1))] > 0 for o in obs]
        rows[f"annotated {cls} with the centre in the mask"] = f"{sum(inside)} of {len(obs)}"
display(pd.DataFrame({"Value": rows}))

## Block 3.1.4. Contour analysis and geometric characteristics of objects

**Contour.** A contour is the closed boundary of a connected region of a binary mask, stored as a list of boundary points (`cv2.findContours`). The mode `RETR_EXTERNAL` keeps only outer contours, `RETR_TREE` also the contours of holes with their hierarchy. For every contour the geometric characteristics are computed:

$$A, \quad P, \quad \frac{L}{W}, \quad e = \frac{A}{A_{box}}, \quad s = \frac{A}{A_{hull}}, \quad c = \frac{4\pi A}{P^2}, \tag{6}$$

where $A$ is the area and $P$ the perimeter, $L \times W$ the size of the minimum-area rotated rectangle, $e$ the extent (filling of the bounding box), $s$ the solidity (filling of the convex hull) and $c$ the circularity (1 for a circle). With the ground sampling distance (Lesson 1.1) the area and the sizes are converted to metres:

$$A_{m^2} = A_{px}\,\mathrm{GSD}^2, \qquad L_m = L_{px}\,\mathrm{GSD}. \tag{7}$$

The sizes in metres do not depend on the flight altitude and can be compared with the known sizes of objects.

**Parameters.** `RETRIEVAL` — mode of `findContours`; `MIN_AREA_PX` — smallest area of a contour shown in the table; `TABLE_ROWS` — number of rows. The mask of block 3.1.3 is used.

**Expected result.** The contours of the mask on the frame, the table of characteristics (6)–(7) of the largest contours and the sizes of the annotated objects of HIT-UAV in metres, computed from the rotated boxes of the annotation of the frames with a vertical view (camera angle 80–90°).

**What to observe.** The annotation of frames with a vertical view gives the reference sizes (medians): a person seen from above is about 0.9 × 0.7 m (0.7 m²; 90 % of the persons below 1.2 m²), a bicycle with its rider about 2.4 × 1.0 m (2.3 m²), a car about 5.9 × 2.7 m (16 m²; 10 % of the cars below 7 m²). These values are used for the filtering of contours in block 3.3.4: a contour of 0.1–2.5 m² is a candidate person, of 2.5–40 m² a candidate vehicle; bicycles lie near the boundary of the two classes. Contours of large warm surfaces (roofs, roads) have areas of hundreds of m² (in the default frame the two largest contours are about 730 and 770 m²) and are removed by the limit of the area.

**Control question 4.** Which characteristics of (6) do not depend on the size of an object in pixels? Why is the area in m² (7) a better feature than the area in pixels for frames taken from different altitudes?

In [ ]:
#@title 3.1.4 Contours and their geometric characteristics { run: "auto" }
RETRIEVAL = "RETR_EXTERNAL" #@param ["RETR_EXTERNAL", "RETR_LIST", "RETR_TREE"]
MIN_AREA_PX = 4 #@param {type:"slider", min:1, max:100, step:1}
TABLE_ROWS = 10 #@param {type:"slider", min:5, max:30, step:1}

fr = FRAME_313
cnts, hier = cv2.findContours(MASK_313, getattr(cv2, RETRIEVAL), cv2.CHAIN_APPROX_SIMPLE)
gsd_c = hit.gsd(fr["altitude"], fr["angle"]) if fr["altitude"] else None
feats = []
for c in cnts:
    if cv2.contourArea(c) < MIN_AREA_PX:
        continue
    f = aitbc.contour_features(c, gsd_c, gsd_c) if gsd_c else aitbc.contour_features(c)      # formulas (6)-(7)
    feats.append(f)
F = pd.DataFrame(feats)
vis = cv2.cvtColor(fr["grey"], cv2.COLOR_GRAY2RGB)
cv2.drawContours(vis, [c for c in cnts if cv2.contourArea(c) >= MIN_AREA_PX], -1, (255, 60, 60), 1)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(vis)
ax.set_title(f"{fr['caption']}: {len(feats)} contours of {len(cnts)} with A ≥ {MIN_AREA_PX} px ({RETRIEVAL})", fontsize=9)
ax.axis("off")
plt.show()
if len(F):
    cols = ["area_px", "perimeter_px", "length_px", "width_px", "aspect", "extent", "solidity", "circularity"] + (["area_m2", "length_m", "width_m"] if gsd_c else [])
    display(F.sort_values("area_px", ascending=False)[cols].head(TABLE_ROWS).reset_index(drop=True).round(2)
            .style.set_caption("Largest contours of the mask of block 3.1.3" + (f" (GSD at the centre {gsd_c:.3f} m/px)" if gsd_c else "")))

o = hit.objects[(hit.objects.angle >= 80) & hit.objects.cls.isin(["Person", "Bicycle", "Car", "OtherVehicle"])].dropna(subset=["width_px"]).copy()
o["gsd"] = [hit.gsd(a, b) for a, b in zip(o.altitude, o.angle)]
o["length_m"], o["width_m"] = o.length_px * o.gsd, o.width_px * o.gsd
o["area_m2"] = o.length_m * o.width_m
SIZE_REF = o.groupby("cls")[["length_m", "width_m", "area_m2"]].quantile([0.1, 0.5, 0.9]).unstack().round(2)
SIZE_REF.insert(0, ("objects", ""), o.groupby("cls").size())
display(SIZE_REF.style.set_caption("Sizes of the annotated objects of HIT-UAV in metres (rotated boxes, camera angle 80–90°): percentiles 10, 50, 90"))

### Questions for self-check of Lesson 3.1

1. What is an edge of an object in a visible frame and in a thermal frame? Which physical quantities change across it?
2. Write the Sobel kernels and the formulas of the magnitude and direction of the gradient.
3. List the four stages of the Canny method and explain the role of each of them.
4. Why can no pair of thresholds of the Canny method be chosen that suits day and night frames of the same scene?
5. Write the criterion of Otsu's method. When is it suitable, and why are small warm targets not separated by it?
6. Which geometric characteristics of a contour separate a person from a car in a thermal frame taken from a UAV, and why must the area be converted to square metres?

---
# Lesson 3.3 (Practical class). Features of silhouettes and thermal signatures in thermal frames

**Learning questions**
1. Basic geometric transformations of images: scaling, rotation, translation, perspective transformations.
2. Preparation of a thermal frame: conversion to grey levels, application of CLAHE.
3. Morphological processing of the obtained binary mask.
4. Detection of contours and their filtering by geometric characteristics.
5. Visualisation of the results of the processing by overlaying contours or bounding boxes on the original frame.

**Task of the lesson.** A classical module of detection of thermal signatures is built step by step: frame → grey levels in the white-hot convention → local contrast and threshold → morphology → contours → filtering by sizes in metres → persons and vehicles on the frame. Every step is checked on the annotated night frames of HIT-UAV.

**Result of the lesson.** The detection pipeline with the chosen parameters, its recall and precision on 40 annotated night frames, the figure with the detections and the table of contours in `outputs`, the parameters in `project.json`.

## Task 3.3.1. Geometric transformations: scaling, rotation, translation, perspective transformation

**Affine transformation.** Scaling by $s$, rotation by $\alpha$ about the centre $c$ and translation by $t$:

$$\begin{bmatrix} x' \\ y' \end{bmatrix} = s\begin{bmatrix} \cos\alpha & \sin\alpha \\ -\sin\alpha & \cos\alpha \end{bmatrix}\left(\begin{bmatrix} x \\ y \end{bmatrix} - c\right) + c + t, \tag{8}$$

the matrix $2 \times 3$ of `cv2.getRotationMatrix2D(c, α, s)` with $t$ added to its last column ($\alpha > 0$ — counter-clockwise). The boxes of the annotation are transformed by their four corners: after a rotation the axis-aligned box of the corners contains, in addition to the object, a part of the background.

**Perspective transformation to the top view.** A camera at the height $h$ looks forward with the depression angle $\vartheta$ (the camera angle of HIT-UAV; 90° — vertical view). A point $(X, Y)$ of the flat ground is mapped to the image by the homography

$$\tilde{\mathbf x} = H_g \begin{bmatrix} X \\ Y \\ 1 \end{bmatrix}, \qquad H_g = K\,[\,\mathbf r_1 \;\; \mathbf r_2 \;\; -h\,\mathbf r_3\,], \tag{9}$$

where $K$ is the camera matrix (from the field of view 25° × 20° of the Zenmuse XT2) and $\mathbf r_i$ the columns of the rotation of the camera; $H_g^{-1}$ gives the top view with a constant GSD.

**Parameters.** `FRAME` — an oblique frame of HIT-UAV (altitude and camera angle from the file name); `SCALE`, `ANGLE_DEG`, `SHIFT_X`, `SHIFT_Y` — parameters of (8); `MAX_RANGE_M` — largest ground distance shown in the top view.

**Expected result.** The frame with the boxes, the result of (8) with the transformed boxes, the top view (9) with the boxes; the growth of the boxes after rotation, measured on the annotation of HIT-UAV; the length of cars in metres in four bands of rows of oblique frames for three models of the GSD.

**What to observe.** In the top view the lines of a parking lot become parallel and perpendicular, as on the ground. Cars in the far part of the frame are stretched along the line of sight: the perspective transformation is exact only for points of the ground, while the box of a car also contains its side. The rotation of a frame by 15°, 30° and 45° increases the area of the boxes of the annotation (median) by 1.5, 1.9 and 2.0 times — an augmentation by rotation needs rotated boxes or a recomputation of the labels. For the length of cars in oblique frames: one GSD of the centre of the frame makes the cars of the far rows 26 % shorter than those of the near rows (4.0 m against 5.4 m); the GSD of the row computed from the slant range (model «range of the row») gives 4.6–4.9 m in all bands; the ground-plane model, which also divides by the sine of the angle of view, overestimates the far cars (6.7 m). The model «range of the row» is used in task 3.3.4.

**Control question 5.** Why does an axis-aligned box grow after rotation of a frame? Why is a perspective transformation to the top view exact for road markings and not for cars?

In [ ]:
#@title 3.3.1 Geometric transformations of a thermal frame (code shown) { run: "auto" }
FRAME = "0_80_40_0_06933.jpg: day, 80 m, 40°, parking lot" #@param ["0_80_40_0_06933.jpg: day, 80 m, 40°, parking lot", "0_120_30_0_09449.jpg: day, 120 m, 30°, parking lot and buildings", "1_110_30_0_02555.jpg: night, 110 m, 30°, persons on a sports ground", "0_100_40_0_03347.jpg: day, 100 m, 40°, sports ground"]
SCALE = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.05}
ANGLE_DEG = 30 #@param {type:"slider", min:-180, max:180, step:5}
SHIFT_X = 0 #@param {type:"slider", min:-200, max:200, step:10}
SHIFT_Y = 0 #@param {type:"slider", min:-200, max:200, step:10}
MAX_RANGE_M = 250 #@param {type:"slider", min:50, max:500, step:10}

f = FRAME.split(":")[0]
g, objs = hit.load(f)
alt, ang = int(f.split("_")[1]), int(f.split("_")[2])
H_, W_ = g.shape
boxes = np.array([o["box"] for o in objs], float).reshape(-1, 4)
colors = [hit.COLORS.get(o["cls"], (255, 255, 255)) for o in objs]

M = cv2.getRotationMatrix2D(((W_ - 1) / 2, (H_ - 1) / 2), ANGLE_DEG, SCALE)          # formula (8)
M[:, 2] += (SHIFT_X, SHIFT_Y)
aff = cv2.warpAffine(g, M, (W_, H_))
boxes_aff = aitbc.transform_boxes(boxes, np.vstack([M, [0, 0, 1]]))

K_xt2 = aitbc.camera_matrix(W_, H_, hit.CAMERA["hfov_deg"], hit.CAMERA["vfov_deg"])
Hg, (Wt, Ht), gsd_top, ext = aitbc.ground_homography(alt, ang, K_xt2, (W_, H_), max_range_m=MAX_RANGE_M)   # formula (9)
top = cv2.warpPerspective(g, Hg, (Wt, Ht))
boxes_top = aitbc.transform_boxes(boxes, Hg)


def draw(ax, img, bxs, title):
    ax.imshow(img, cmap="gray")
    for b, c in zip(bxs, colors):
        ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=np.array(c) / 255, lw=0.9))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


fig, ax = plt.subplots(1, 3, figsize=(19, 6), gridspec_kw=dict(width_ratios=[1, 1, Wt / Ht * H_ / W_]))
draw(ax[0], g, boxes, f"{f}: {alt} m, camera angle {ang}°")
draw(ax[1], aff, boxes_aff, f"(8): s = {SCALE:g}, α = {ANGLE_DEG}°, t = ({SHIFT_X}, {SHIFT_Y}) px")
draw(ax[2], top, boxes_top, f"(9): top view, {gsd_top:.3f} m/px, ground distance {ext[2]:.0f}–{ext[3]:.0f} m"
     + (" (MAX_RANGE_M is below the nearest row)" if ext[3] > MAX_RANGE_M + 1 else ""))
plt.tight_layout()
plt.show()

ov = hit.objects[(hit.objects.angle >= 80) & hit.objects.cls.isin(["Person", "Car"])]
rows = []
for a_ in (15, 30, 45, ANGLE_DEG):
    r_ = math.radians(a_)
    wn = np.abs(ov.w * math.cos(r_)) + np.abs(ov.h * math.sin(r_))
    hn = np.abs(ov.w * math.sin(r_)) + np.abs(ov.h * math.cos(r_))
    rows.append({"rotation, deg": a_, "median growth of the box area": float(np.median(wn * hn / (ov.w * ov.h)))})
display(pd.DataFrame(rows).drop_duplicates().set_index("rotation, deg").round(2)
        .style.set_caption(f"Boxes of {len(ov)} persons and cars of HIT-UAV (vertical views) after rotation"))

cars = hit.objects[(hit.objects.cls == "Car") & (hit.objects.angle <= 50)].dropna(subset=["length_px"]).copy()
cars["row"] = (cars.y1 + cars.y2) / 2
across, along = aitbc.oblique_gsd(cars.altitude.values, cars.angle.values, cars.row.values)
cars["central GSD of the frame"] = cars.length_px * [hit.gsd(a, b) for a, b in zip(cars.altitude, cars.angle)]
cars["range of the row"] = cars.length_px * across
cars["ground plane (range and foreshortening)"] = cars.length_px * np.sqrt(across * along)
cars["band of rows"] = pd.cut(cars.row, [0, 128, 256, 384, 512], labels=["0–127 (far)", "128–255", "256–383", "384–511 (near)"])
GSD_CHECK = cars.groupby("band of rows", observed=True)[["central GSD of the frame", "range of the row", "ground plane (range and foreshortening)"]].median()
display(GSD_CHECK.round(2).style.set_caption(f"Median length of {len(cars)} cars of HIT-UAV in oblique frames (camera angle 30–50°), m"))

## Task 3.3.2. Preparation of a thermal frame: grey levels, CLAHE and the polarity of thermal signatures

**Grey levels in one convention.** The thermal cameras of the data sets record the frames in different palettes: white-hot (HIT-UAV: warm = bright), black-hot (M3OT: warm = dark) and a pseudo-colour palette (Kust4K: purple = cold, yellow = warm). Before the processing the frame is converted to one channel of grey levels in the white-hot convention: the luminance of the palette, inverted for black-hot (`aitbc.to_white_hot`). CLAHE (Lesson 2.4) may be applied afterwards.

**Polarity of a signature.** An object is *warm* in the frame when its brightest part differs from the background more than its darkest part:

$$\text{warm:}\quad P_{90}(\text{box}) - \operatorname{med}(\text{ring}) \;\ge\; \operatorname{med}(\text{ring}) - P_{10}(\text{box}), \tag{10}$$

where $P_{90}$, $P_{10}$ are the percentiles of the brightness inside the box of the annotation and the ring is the background around the box.

**Parameters.** `SOURCE`, `NUMBER` — thermal frame; `APPLY_CLAHE`, `CLIP_LIMIT`, `TILES` — CLAHE after the conversion.

**Expected result.** The frame in its palette, the grey frame in the white-hot convention (and after CLAHE), their histograms; the shares of warm and cold signatures of persons, cars and bicycles on 160 frames of HIT-UAV by day and at night.

**What to observe.** At night almost all persons are warm (97 %), while 86 % of cars and 82 % of bicycles are *colder* than the asphalt that kept the heat of the day. By day the picture changes: the ground is heated by the sun, and only two thirds of persons are warmer than their background. A detector of warm spots therefore finds persons at night, but must look for cold spots to find parked cars and fails for persons by day (block 3.3.4). CLAHE raises the visible contrast of the background, which also raises the contrast of the clutter; its effect on the detection is measured in block 3.3.4.

**Control question 6.** Why are parked cars colder than the road at night? Which polarity of the mask is needed to find persons and parked cars in one night frame?

In [ ]:
#@title 3.3.2 Preparation of a thermal frame and the polarity of signatures (code shown) { run: "auto" }
SOURCE = "HIT-UAV: night, test frames (white-hot)" #@param ["HIT-UAV: night, test frames (white-hot)", "HIT-UAV: day, test frames (white-hot)", "M3OT 1-08: dusk (black-hot)", "Kust4K: night (pseudo-colour palette)"]
NUMBER = 1 #@param {type:"slider", min:1, max:40, step:1}
APPLY_CLAHE = False #@param {type:"boolean"}
CLIP_LIMIT = 2.0 #@param {type:"slider", min:1, max:8, step:0.5}
TILES = 8 #@param {type:"slider", min:2, max:16, step:1}

GSD_ASSUMED = 0.145      # m/px of M3OT and Kust4K: thermal camera f = 9.1 mm, pixel 12 um, altitude about 110 m (assumption)
k = NUMBER - 1
if SOURCE.startswith("HIT-UAV"):
    files = HIT_EVAL_NIGHT if "night" in SOURCE else HIT_EVAL_DAY
    f = files[k % len(files)]
    raw, objs = hit.load(f)
    r = hit.images.set_index("file").loc[f]
    shown, palette, alt, ang = raw, hit.PALETTE, int(r.altitude), int(r.angle)
    caption = f"HIT-UAV {f}: {alt} m, {ang}°, {'night' if r.night else 'day'}"
elif SOURCE.startswith("M3OT"):
    f0 = (300, 500)[k % 2]
    shown = m3ot.frames("1-08", "ir", start=f0, count=1)[0]
    objs, palette, alt, ang, f = m3ot.objects(m3ot.tracks("1-08", "ir"), f0 + 1), m3ot.PALETTE, None, None, f"m3ot_1-08_{f0 + 1}"
    caption = f"M3OT 1-08, thermal frame {f0 + 1} (black-hot)"
else:
    fid = aitbc.Kust4K.NIGHT[k % 6]
    p = kust.pair(fid)
    shown, palette, alt, ang, f = p["tir"], kust.PALETTE, None, None, f"kust4k_{fid}"
    objs = aitbc.boxes_from_labels(p["labels"], (3, 4, 5, 7), kust.CLASSES, min_area=20)
    caption = f"Kust4K {fid} (night), pseudo-colour palette"
grey = aitbc.to_white_hot(shown, palette)[..., 0]                         # one channel, warm = bright
prepared = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=(TILES, TILES)).apply(grey) if APPLY_CLAHE else grey
PREP_33 = dict(file=f, grey=prepared, shown=shown, objects=objs, altitude=alt, angle=ang, caption=caption,
               gsd=hit.gsd(alt, ang) if alt else GSD_ASSUMED, clahe=APPLY_CLAHE)

fig, ax = plt.subplots(1, 4, figsize=(19, 4.3), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.9]))
ax[0].imshow(shown, cmap="gray" if shown.ndim == 2 else None)
ax[0].set_title(f"{caption}\npalette: {palette}", fontsize=8.5)
ax[1].imshow(grey, cmap="gray", vmin=0, vmax=255)
ax[1].set_title("grey levels, white-hot (warm = bright)", fontsize=9)
ax[2].imshow(prepared, cmap="gray", vmin=0, vmax=255)
ax[2].set_title(f"after CLAHE c = {CLIP_LIMIT:g}, {TILES}×{TILES}" if APPLY_CLAHE else "without CLAHE", fontsize=9)
for a in ax[:3]:
    a.axis("off")
ax[3].hist(grey.ravel(), bins=64, range=(0, 255), alpha=0.6, label="white-hot")
if APPLY_CLAHE:
    ax[3].hist(prepared.ravel(), bins=64, range=(0, 255), alpha=0.6, label="after CLAHE")
ax[3].legend(fontsize=8)
ax[3].set_xlabel("grey level")
ax[3].set_title("histograms", fontsize=9)
plt.tight_layout()
plt.show()

if "POLARITY" not in globals():
    recs = []
    for fp in HIT_POLARITY:
        gp, op_ = hit.load(fp)
        night = bool(hit.images.set_index("file").loc[fp].night)
        for ob in op_:
            if ob["cls"] not in ("Person", "Car", "Bicycle"):
                continue
            x1, y1, x2, y2 = [int(round(v)) for v in ob["box"]]
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            inner = gp[max(0, y1):y2, max(0, x1):x2].astype(float)
            pad = max(4, int(0.5 * max(x2 - x1, y2 - y1)))
            X1, Y1, X2, Y2 = max(0, x1 - pad), max(0, y1 - pad), min(gp.shape[1], x2 + pad), min(gp.shape[0], y2 + pad)
            ring = gp[Y1:Y2, X1:X2].astype(float)
            keep = np.ones(ring.shape, bool)
            keep[max(0, y1) - Y1:y2 - Y1, max(0, x1) - X1:x2 - X1] = False
            bg = np.median(ring[keep])
            warm = np.percentile(inner, 90) - bg >= bg - np.percentile(inner, 10)         # formula (10)
            recs.append(dict(time="night" if night else "day", cls=ob["cls"], warm=warm))
    POLARITY = pd.DataFrame(recs)
pol = POLARITY.groupby(["cls", "time"]).warm.agg(["size", "mean"]).rename(columns={"size": "objects", "mean": "warm share"})
pol["cold share"] = 1 - pol["warm share"]
display(pol.round(2).style.set_caption(f"Polarity (10) of the signatures on {len(HIT_POLARITY)} frames of HIT-UAV (validation and test parts)"))

## Task 3.3.3. Morphological processing of the binary mask

**Morphological operations** with a structuring element $S$ (a disc of the given diameter):

$$I \ominus S = \min_{s \in S} I(x + s), \quad I \oplus S = \max_{s \in S} I(x + s), \quad I \circ S = (I \ominus S) \oplus S, \quad I \bullet S = (I \oplus S) \ominus S, \tag{11}$$

erosion, dilation, opening (removes bright details smaller than $S$) and closing (fills dark gaps smaller than $S$). The *white top-hat* $I - I \circ S$ keeps warm details smaller than $S$, the *black top-hat* $I \bullet S - I$ cold details. The same operations applied to the binary mask remove isolated pixels (opening) and join the parts of one object (closing).

**Pipeline of the mask.** prepared frame (3.3.2) → white / black top-hat with $S$ given in metres → threshold → opening → closing.

**Parameters.** `POLARITY` — warm, cold or both kinds of signatures (10); `MASK_METHOD` — threshold: Otsu on the whole frame, Otsu on the top-hat, mean + $k\sigma$ of the top-hat, adaptive local mean; `SE_M` — diameter of $S$ in metres; `K`; `OPEN_PX`, `CLOSE_PX` — sizes of the opening and closing of the mask in pixels (1 — not applied).

**Expected result.** The stages of the processing and a table for each stage: number of connected regions, share of pixels in the mask and the annotated objects whose centre lies in the mask.

**What to observe.** The top-hat removes the large warm surfaces (roads, roofs) and keeps the small signatures. In the default frame a threshold of mean + 4σ of the top-hat leaves 1.5 % of the pixels. The opening 3 × 3 removes single pixels of noise (60 → 45 regions in the default frame) and separates touching blobs: on the 40 night frames of block 3.3.4 the recall of persons is 0.84 with the opening and 0.81 without it. Check with `OPEN_PX = 5` how many distant persons, which are only 2–4 px wide, remain in the mask. The closing joins the parts of a car, but can merge persons standing close to each other. With `POLARITY = cold` the parked cars of a night frame appear in the mask.

**Control question 7.** Which operation (11) is used to remove isolated noise pixels from a mask, and which to fill gaps inside an object? Why is the size of $S$ set in metres?

In [ ]:
#@title 3.3.3 Morphological processing of the mask (code shown) { run: "auto" }
POLARITY = "warm" #@param ["warm", "cold", "both"]
MASK_METHOD = "top-hat + mean + k*std" #@param ["Otsu (global)", "top-hat + Otsu", "top-hat + mean + k*std", "adaptive (local mean)"]
SE_M = 3.0 #@param {type:"slider", min:1, max:15, step:0.5}
K = 4.0 #@param {type:"slider", min:2, max:8, step:0.5}
OPEN_PX = 3 #@param [1, 3, 5] {type:"raw"}
CLOSE_PX = 1 #@param [1, 3, 5, 7] {type:"raw"}


def mask_stages(grey, gsd, polarity, method, se_m, k, open_px, close_px):
    se_px = max(3, int(round(se_m / gsd))) | 1
    raw = aitbc.thermal_mask(grey, polarity, method, se_px=se_px, k=k)        # top-hat (11) and threshold
    opened = raw if open_px <= 1 else cv2.morphologyEx(raw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_px, open_px)))
    closed = opened if close_px <= 1 else cv2.morphologyEx(opened, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_px, close_px)))
    return dict(raw=raw, opened=opened, closed=closed, se_px=se_px)


P33 = dict(polarity=POLARITY, method=MASK_METHOD, se_m=SE_M, k=K, open_px=OPEN_PX, close_px=CLOSE_PX)
st = mask_stages(PREP_33["grey"], PREP_33["gsd"], **P33)
MASK_33 = st["closed"]
g = cv2.GaussianBlur(PREP_33["grey"], (3, 3), 0)
se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (st["se_px"], st["se_px"]))
th_w = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, se)
th_b = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, se)

fig, ax = plt.subplots(1, 5, figsize=(20, 4.3))
for a, (im_, t_) in zip(ax, ((PREP_33["grey"], PREP_33["caption"]), (th_w if POLARITY != "cold" else th_b, f"{'white' if POLARITY != 'cold' else 'black'} top-hat, S = {SE_M:g} m = {st['se_px']} px"),
                             (st["raw"], f"threshold: {MASK_METHOD}"), (st["opened"], f"opening {OPEN_PX}×{OPEN_PX}"),
                             (st["closed"], f"closing {CLOSE_PX}×{CLOSE_PX}"))):
    a.imshow(im_, cmap="gray" if im_ is not th_w and im_ is not th_b else "magma")
    a.set_title(t_, fontsize=8.5)
    a.axis("off")
plt.tight_layout()
plt.show()


def centre_hits(mask, objs, classes):
    obs = [o for o in objs if o["cls"] in classes]
    hits = sum(mask[int(np.clip((o["box"][1] + o["box"][3]) / 2, 0, mask.shape[0] - 1)),
                    int(np.clip((o["box"][0] + o["box"][2]) / 2, 0, mask.shape[1] - 1))] > 0 for o in obs)
    return f"{hits} of {len(obs)}" if obs else "—"


rows = []
for name in ("raw", "opened", "closed"):
    m = st[name]
    n, _ = cv2.connectedComponents(m)
    rows.append({"stage": {"raw": "threshold", "opened": "after opening", "closed": "after closing"}[name], "regions": n - 1,
                 "share of pixels": f"{(m > 0).mean():.2%}", "persons with the centre in the mask": centre_hits(m, PREP_33["objects"], ("Person", "person")),
                 "vehicles with the centre in the mask": centre_hits(m, PREP_33["objects"], ("Car", "OtherVehicle", "vehicle", "car", "truck"))})
display(pd.DataFrame(rows).set_index("stage"))

## Task 3.3.4. Contours and their filtering by geometric characteristics

**Filtering.** For every contour of the mask the characteristics (6) are computed, and its area and sizes are converted to metres (7) with the GSD of its row. A contour is kept as a thermal signature when

$$A_{min} \le A_{m^2} \le A_{max}, \qquad \frac{L}{W} \le r_{max}, \qquad s \ge s_{min}, \tag{12}$$

and it is assigned the class *person* when $A_{m^2} <$ `PERSON_MAX_M2`, otherwise *vehicle* (block 3.1.4: a person about 1 m², a car about 13 m²).

**Evaluation.** A detection is correct when its centre lies in a box of the annotation (enlarged by 2 px); the recall is counted separately for persons and vehicles; the precision is the share of correct detections; the class is correct when a person is detected as a person and a vehicle as a vehicle.

**Parameters.** `AREA_MIN_M2`, `AREA_MAX_M2`, `MAX_ASPECT`, `MIN_SOLIDITY` — limits (12); `PERSON_MAX_M2` — boundary of the classes; `GSD_MODEL` — GSD of the row of the contour (task 3.3.1); `EVALUATE`, `EVAL_SET` — evaluation on 40 night or 24 day test frames of HIT-UAV with the mask of block 3.3.3 (about 5 s); `RUN_ABLATION` — the same evaluation with one stage changed at a time (about 30 s).

**Expected result.** The kept and the rejected contours of the current frame with the reasons of rejection; the recall, precision and class correctness on the chosen set; the table of variants.

**What to observe.** On the 40 night test frames (343 persons, 74 vehicles) the pipeline finds 0.84 of the persons and 0.65 of the vehicles with a precision of 0.33; the class is correct for 84 % of the correct detections. The false detections are lamps, warm parts of buildings and edges of trees that have the size of a person. Without the geometric filter the recall of persons grows to 0.92, but the precision falls to 0.22. The global Otsu threshold instead of the top-hat gives a recall of 0.25, CLAHE before the mask lowers it to 0.60 (as in Lesson 2.4). The mask of warm and cold signatures raises the recall of vehicles to 0.80 but lowers that of persons to 0.57: the cold details merge with warm persons and add false contours. The central GSD of the frame instead of the GSD of the row changes the result little on these frames (0.83 against 0.84). By day the recall of persons falls to 0.20 (89 persons on 24 frames): a third of the persons are not warmer than the heated ground (task 3.3.2). The classical pipeline is a fast and explainable detector of warm targets at night; the precision and the work by day require the features learned by the networks of Topics 4 and 5.

**Control question 8.** Which false detections are removed by the limits of the area in m², and which remain? Why does the recall by day fall so strongly?

In [ ]:
#@title 3.3.4 Contours and their filtering (code shown) { run: "auto" }
AREA_MIN_M2 = 0.1 #@param {type:"slider", min:0.02, max:1.0, step:0.02}
AREA_MAX_M2 = 40.0 #@param {type:"slider", min:5, max:100, step:5}
MAX_ASPECT = 5.0 #@param {type:"slider", min:1.5, max:10, step:0.5}
MIN_SOLIDITY = 0.6 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
PERSON_MAX_M2 = 2.5 #@param {type:"slider", min:0.5, max:6, step:0.25}
GSD_MODEL = "range of the row" #@param ["central GSD of the frame", "range of the row", "ground plane (range and foreshortening)"]
EVALUATE = True #@param {type:"boolean"}
EVAL_SET = "night: 40 test frames" #@param ["night: 40 test frames", "day: 24 test frames"]
RUN_ABLATION = True #@param {type:"boolean"}

F33 = dict(area_min=AREA_MIN_M2, area_max=AREA_MAX_M2, max_aspect=MAX_ASPECT, min_solidity=MIN_SOLIDITY,
           person_max=PERSON_MAX_M2, gsd_model=GSD_MODEL)


def row_gsd(model, alt, ang, row, gsd_default):
    if alt is None:
        return gsd_default, gsd_default
    if model.startswith("central"):
        v = hit.gsd(alt, ang)
        return v, v
    across, along = aitbc.oblique_gsd(alt, ang, row)
    return (float(across), float(across)) if model.startswith("range") else (float(across), float(along))


def detect_signatures(grey, alt, ang, gsd_default, mask_par, filt, clahe=False):
    if clahe:
        grey = cv2.createCLAHE(2.0, (8, 8)).apply(grey)
    mask = mask_stages(grey, hit.gsd(alt, ang) if alt else gsd_default, **mask_par)["closed"]
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cnts:
        if cv2.contourArea(c) < 2:
            continue
        cy = aitbc.contour_features(c)["cy"]
        f = aitbc.contour_features(c, *row_gsd(filt["gsd_model"], alt, ang, cy, gsd_default))     # formulas (6)-(7)
        reason = ("area < min" if f["area_m2"] < filt["area_min"] else "area > max" if f["area_m2"] > filt["area_max"]
                  else "elongated" if f["aspect"] > filt["max_aspect"] else "low solidity" if f["solidity"] < filt["min_solidity"] else "")
        f.update(keep=reason == "", reason=reason, contour=c, cls="Person" if f["area_m2"] < filt["person_max"] else "vehicle")   # (12)
        out.append(f)
    return out


def evaluate_pipeline(files, mask_par, filt, clahe=False):
    c = dict(P=0, Pf=0, V=0, Vf=0, det=0, ok=0, cls=0)
    for fname in files:
        gg, objs = hit.load(fname)
        r = hit.images.set_index("file").loc[fname]
        dets = [d for d in detect_signatures(gg, int(r.altitude), int(r.angle), None, mask_par, filt, clahe) if d["keep"]]
        c["det"] += len(dets)
        used = set()
        for d in dets:
            hits = [j for j, o in enumerate(objs) if o["box"][0] - 2 <= d["cx"] <= o["box"][2] + 2 and o["box"][1] - 2 <= d["cy"] <= o["box"][3] + 2]
            if hits:
                c["ok"] += 1
                used.update(hits)
                c["cls"] += (d["cls"] == "Person") == (objs[hits[0]]["cls"] == "Person")
        for j, o in enumerate(objs):
            if o["cls"] == "Person":
                c["P"] += 1
                c["Pf"] += j in used
            elif o["cls"] in ("Car", "OtherVehicle"):
                c["V"] += 1
                c["Vf"] += j in used
    return {"recall of persons": c["Pf"] / max(c["P"], 1), "recall of vehicles": c["Vf"] / max(c["V"], 1),
            "precision": c["ok"] / max(c["det"], 1), "class correct": c["cls"] / max(c["ok"], 1),
            "detections": c["det"], "persons": c["P"], "vehicles": c["V"]}


DETS_33 = detect_signatures(PREP_33["grey"], PREP_33["altitude"], PREP_33["angle"], PREP_33["gsd"], P33, F33)
kept = [d for d in DETS_33 if d["keep"]]
vis_ = cv2.cvtColor(PREP_33["grey"], cv2.COLOR_GRAY2RGB)
cv2.drawContours(vis_, [d["contour"] for d in DETS_33 if not d["keep"]], -1, (230, 40, 40), 1)
cv2.drawContours(vis_, [d["contour"] for d in kept if d["cls"] == "Person"], -1, (0, 229, 255), 2)
cv2.drawContours(vis_, [d["contour"] for d in kept if d["cls"] != "Person"], -1, (255, 235, 59), 2)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(vis_)
ax.set_title(f"{PREP_33['caption']}: kept {len(kept)} (cyan — person, yellow — vehicle), rejected {len(DETS_33) - len(kept)} (red)", fontsize=9)
ax.axis("off")
plt.show()
print("Rejected contours by reason:", pd.Series([d["reason"] for d in DETS_33 if not d["keep"]]).value_counts().to_dict())

if EVALUATE:
    files = HIT_EVAL_NIGHT if EVAL_SET.startswith("night") else HIT_EVAL_DAY
    EVAL_33 = evaluate_pipeline(files, P33, F33)
    display(pd.DataFrame({"Value": EVAL_33}).round(3).style.set_caption(f"Evaluation on HIT-UAV, {EVAL_SET}"))
    if RUN_ABLATION:
        variants = {"pipeline of blocks 3.3.3-3.3.4": (P33, F33, False),
                    "without geometric filter (12)": (P33, dict(F33, area_min=0, area_max=1e9, max_aspect=1e9, min_solidity=0), False),
                    "global Otsu instead of the top-hat": (dict(P33, method="Otsu (global)"), F33, False),
                    "top-hat + Otsu": (dict(P33, method="top-hat + Otsu"), F33, False),
                    "CLAHE before the mask": (P33, F33, True),
                    "without opening": (dict(P33, open_px=1), F33, False),
                    "structuring element 8 m": (dict(P33, se_m=8.0), F33, False),
                    "polarity: warm and cold": (dict(P33, polarity="both"), F33, False),
                    "central GSD of the frame": (P33, dict(F33, gsd_model="central GSD of the frame"), False)}
        t0 = time.time()
        ABL_33 = pd.DataFrame({name: evaluate_pipeline(files, mp, fp, cl) for name, (mp, fp, cl) in variants.items()}).T
        display(ABL_33.round(3).style.set_caption(f"Variants of the pipeline, {EVAL_SET} ({time.time() - t0:.0f} s)"))
        ABL_33.to_csv(os.path.join(REPORT_DIR, "lesson33_variants.csv"))

## Task 3.3.5. Visualisation: contours and boxes on the original frame

**Purpose.** To present the result of the module in the form used by an operator: the original frame in its palette with the contours or boxes of the detected signatures, the class and the size in metres, and — for checking — the boxes of the annotation.

**Parameters.** `OVERLAY` — contours, boxes or both; `SHOW_ANNOTATION` — boxes of the annotation (white, thin); `LABELS` — class and area in m² near each detection; `SAVE` — save the figure and the table of detections in the project.

**Expected result.** The figure `outputs/figures/lesson33_<frame>.png`, the table `outputs/reports/lesson33_detections.csv` (class, centre, box, area and sizes in metres, aspect, solidity) and the record of the lesson in `project.json`.

**What to observe.** On a night frame the persons are found as small warm contours; the boxes of the annotation show the missed persons (for example adjacent persons merged into one contour) and the false detections without an annotation box.

In [ ]:
#@title 3.3.5 Visualisation of the results (code shown) { run: "auto" }
OVERLAY = "contours and boxes" #@param ["contours", "boxes", "contours and boxes"]
SHOW_ANNOTATION = True #@param {type:"boolean"}
LABELS = True #@param {type:"boolean"}
SAVE = True #@param {type:"boolean"}

base = PREP_33["shown"]
out = cv2.cvtColor(base, cv2.COLOR_GRAY2RGB) if base.ndim == 2 else base.copy()
if SHOW_ANNOTATION:
    for o in PREP_33["objects"]:
        x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 255, 255), 1)
for d in kept:
    col = (0, 229, 255) if d["cls"] == "Person" else (255, 235, 59)
    if OVERLAY != "boxes":
        cv2.drawContours(out, [d["contour"]], -1, col, 1)
    if OVERLAY != "contours":
        x1, y1, x2, y2 = d["box"]
        cv2.rectangle(out, (x1 - 2, y1 - 2), (x2 + 2, y2 + 2), col, 1)
    if LABELS:
        cv2.putText(out, f"{'P' if d['cls'] == 'Person' else 'V'} {d['area_m2']:.1f}", (int(d["box"][0]), max(8, int(d["box"][1]) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, col, 1, cv2.LINE_AA)
fig, ax = plt.subplots(figsize=(11, 8.8))
ax.imshow(out)
ax.set_title(f"{PREP_33['caption']}: {sum(d['cls'] == 'Person' for d in kept)} persons (P), {sum(d['cls'] != 'Person' for d in kept)} vehicles (V); "
             f"white — annotation", fontsize=9)
ax.axis("off")
if SAVE:
    name = os.path.splitext(os.path.basename(str(PREP_33["file"])))[0]
    fig_path = os.path.join(FIG_DIR, f"lesson33_{name}.png")
    fig.savefig(fig_path, dpi=110, bbox_inches="tight")
plt.show()
table = pd.DataFrame([{k_: (round(v, 3) if isinstance(v, float) else v) for k_, v in d.items() if k_ not in ("contour", "keep", "reason")} for d in kept])
display(table.head(10))
if SAVE:
    table.to_csv(os.path.join(REPORT_DIR, "lesson33_detections.csv"), index=False)
    update_project_json("3.3", {"lesson_3_3": dict(frame=PREP_33["caption"], clahe=PREP_33["clahe"], mask=P33, filter=F33,
                                                   detections=len(kept), figure=os.path.relpath(fig_path, PROJECT_ROOT),
                                                   evaluation=({k_: round(float(v), 3) for k_, v in EVAL_33.items()} if EVALUATE else None))})

### Report of Lesson 3.3

Double-click this cell and write the answers.

1. Frame, parameters of (8) and (9), growth of the boxes after rotation, the GSD model chosen and why; answer to control question 5:
2. Polarity of the signatures of persons and cars by day and at night; answer to control question 6:
3. Parameters of the mask and the effect of each stage; answer to control question 7:
4. Limits (12), recall, precision and class correctness on the night and day sets, the most important variant of the table; answer to control question 8:
5. Saved figure and table of detections:

---
# Lesson 3.4 (Practical class). Stages of the stabilisation of the video stream of an FPV drone

**Learning questions**
1. Reading of a sequence of frames of a test FPV video fragment.
2. Detection of ORB keypoints on a pair of consecutive frames.
3. Matching of the detected keypoints and quantitative estimation of the inter-frame shift.
4. Computation of the homography matrix and geometric alignment of frames.
5. Saving of the stabilised video stream and comparative analysis of the results of the processing.

**Data.** Flight 16p of the TII-RATM data set: a human FPV pilot flies a racing quadrotor through the gates of an indoor arena (31 s, 337 m). The onboard camera (Arducam IMX219, 640 × 480 px, wide-angle lens 155° × 115°, tilted up by 50°) records 125 frames per second; the gyroscope of the flight controller measures the angular rate at 500 Hz. The gyroscope is an independent sensor: it is used here as the reference of the rotation of the camera between frames, which the image methods estimate from the frames alone.

**Result of the lesson.** The FPV fragment in `data/video`, the measured dependence of the matching on the frame rate, the comparison of the motion models, the stabilised video (original and stabilised side by side) in `outputs/video`, the table of the comparative analysis in `outputs/reports` and the parameters in `project.json`.

## Task 3.4.1. Reading a sequence of frames of a test FPV video fragment

**Purpose.** To prepare a test FPV fragment as a video file, as it is recorded by the video receiver of an operator, to read it frame by frame with OpenCV and to estimate how fast the camera rotates between frames.

**Rotation between frames.** With the frame rate $f$ and every $n$-th frame used, the interval is $\Delta t = n / f$, and the rotation of the camera between two used frames is

$$\varphi \approx |\boldsymbol\omega|\,\Delta t, \tag{13}$$

where $\boldsymbol\omega$ is the angular rate measured by the gyroscope (in the axes of the camera: x — right, y — down, z — along the optical axis).

**Parameters.** `SEGMENT` — fragment of the flight (A — take-off and the first gates, B — slalom between gates; 4 s each); `FRAME_STEP` — every $n$-th frame (1 — 125 fps, 2 — 63 fps, 4 — 31 fps, 8 — 16 fps, as with a slower camera or a reduced video link).

**Expected result.** The video file `data/video/fpv_tii16p_<A|B>_<fps>fps.mp4` and its properties read by OpenCV, frames of the fragment, the angular rate of the camera and the distribution of the rotation (13) between the used frames.

**What to observe.** The first run downloads the camera archive of the flight once (about 200 MB, 1–3 min) and keeps the frames of both fragments in the cache. The pilot turns the drone at up to about 790 °/s. In fragment B the median rotation between consecutive frames is 1.0° at 126 fps (95th percentile 6°), 2.0° at 63 fps, 4.1° at 32 fps and 8.6° at 16 fps (95th percentile 47°): the content of consecutive frames overlaps less and less, which is the main condition of the matching of task 3.4.3.

**Control question 9.** How does the rotation between frames change when every 4th frame is used? Why is the frame rate of an FPV camera more important for stabilisation than its resolution?

In [ ]:
#@title 3.4.1 Test FPV video fragment (code shown) { display-mode: "form" }
SEGMENT = "B: slalom between gates (15.0-19.0 s)" #@param ["B: slalom between gates (15.0-19.0 s)", "A: take-off and first gates (0.5-4.5 s)"]
FRAME_STEP = 2 #@param [1, 2, 4, 8] {type:"raw"}

t_start, t_end = tii.SEGMENTS[tii.flight][SEGMENT]
t0 = time.time()
clip_all, clip_tab = tii.clip(t_start, t_end, step=1)                    # frames of the data set, 125 fps
times_all = clip_tab.t.values
fps_cam = 1.0 / np.median(np.diff(times_all))
frames_src = clip_all[::FRAME_STEP]
fps_eff = fps_cam / FRAME_STEP
seg = SEGMENT[0]
video_path = os.path.join(PROJECT_ROOT, "data", "video", f"fpv_tii16p_{seg}_{fps_eff:.0f}fps.mp4")
aitbc.write_video(video_path, frames_src, fps_eff)
info = aitbc.video_info(video_path)
cap = cv2.VideoCapture(video_path)                                         # reading frame by frame
video_frames = []
while True:
    ok, fr_ = cap.read()
    if not ok:
        break
    video_frames.append(cv2.cvtColor(fr_, cv2.COLOR_BGR2RGB))
cap.release()

imu = tii.imu()
R_cb = tii.camera_to_body()                                                # camera axes -> body axes
w_cam = imu[["gyro_x", "gyro_y", "gyro_z"]].values @ R_cb                 # rows: (R_cb^T w_body)^T
m = (imu.t >= t_start) & (imu.t <= t_end)
times = times_all[::FRAME_STEP]
phi = np.array([np.degrees(np.linalg.norm(aitbc.gyro_rotation(imu.t.values, w_cam, a, b))) for a, b in zip(times[:-1], times[1:])])   # (13)
ROT_STEPS = {}
for st_ in (1, 2, 4, 8):
    tt_ = times_all[::st_]
    ph_ = [np.degrees(np.linalg.norm(aitbc.gyro_rotation(imu.t.values, w_cam, a, b))) for a, b in zip(tt_[:-1], tt_[1:])]
    ROT_STEPS[st_] = {"fps": round(fps_cam / st_), "median φ, °": float(np.median(ph_)), "95th percentile φ, °": float(np.percentile(ph_, 95))}
FPV = dict(segment=SEGMENT, step=FRAME_STEP, fps=fps_eff, frames=video_frames, times=times, path=video_path, imu_t=imu.t.values,
           w_cam=w_cam, clip_all=[cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in clip_all], times_all=times_all)   # grey frames at 125 fps
del clip_all, frames_src

fig = plt.figure(figsize=(19, 7.5))
gs_ = fig.add_gridspec(2, 6)
for j, i in enumerate(np.linspace(0, len(video_frames) - 1, 6).astype(int)):
    a = fig.add_subplot(gs_[0, j])
    a.imshow(video_frames[i])
    a.set_title(f"frame {i}, t = {times[i]:.2f} s", fontsize=8.5)
    a.axis("off")
a = fig.add_subplot(gs_[1, :4])
for k_, lab in enumerate(("ω_x (tilt)", "ω_y (pan)", "ω_z (roll about the optical axis)")):
    a.plot(imu.t[m], np.degrees(w_cam[m.values, k_]), lw=0.7, label=lab)
a.set_xlabel("time of the flight, s")
a.set_ylabel("angular rate, °/s")
a.legend(fontsize=8)
a.grid(alpha=0.3)
a.set_title("Gyroscope in the axes of the camera", fontsize=9)
a = fig.add_subplot(gs_[1, 4:])
a.hist(phi, bins=40, color="C2")
a.set_xlabel(f"rotation between used frames φ (13), ° (Δt = {1000 / fps_eff:.1f} ms)")
a.set_title(f"median {np.median(phi):.2f}°, 95th percentile {np.percentile(phi, 95):.2f}°", fontsize=9)
a.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"lesson34_fragment_{seg}.png"), dpi=90)
plt.show()
display(pd.DataFrame({"Value": {"fragment": SEGMENT, "frames read from the video file": len(video_frames),
                                "frame rate of the camera (time stamps), fps": round(fps_cam, 1), "used frame rate, fps": round(fps_eff, 1),
                                "size of a frame": f"{info['width']} × {info['height']}", "codec / size of the file": f"{info['codec']} / {info['size_mb']:.1f} MB",
                                "largest angular rate in the fragment, °/s": round(float(np.degrees(np.linalg.norm(w_cam[m.values], axis=1)).max())),
                                "time of preparation, s": round(time.time() - t0)}}))
display(pd.DataFrame(ROT_STEPS).T.rename_axis("frame step").round(2).style.set_caption("Rotation of the camera between used frames (13) for every frame step"))

## Task 3.4.2. ORB keypoints on a pair of consecutive frames

**ORB** (Oriented FAST and Rotated BRIEF). (a) *FAST*: a pixel is a corner when on the circle of 16 pixels around it there are at least 9 consecutive pixels brighter or darker than its brightness by more than the threshold $t$; (b) a pyramid of images with the scale factor $s$ gives keypoints of different sizes; (c) the orientation of a keypoint is the direction of the intensity centroid of its neighbourhood,

$$\theta = \operatorname{atan2}(m_{01}, m_{10}), \qquad m_{pq} = \sum_{x, y} x^p y^q I(x, y); \tag{14}$$

(d) *rotated BRIEF*: the descriptor is a string of 256 bits, each bit compares the brightness of two pixels of a fixed pattern turned by $\theta$. The descriptor does not change when the frame turns, which is necessary for an FPV camera that rolls.

**Lens distortion.** The wide-angle lens bends straight lines; the frames are corrected with the camera matrix and the distortion coefficients of the data set (`cv2.initUndistortRectifyMap`). A homography (task 3.4.4) describes the motion of a camera without distortion.

**Parameters.** `PAIR_POSITION` — position of the pair in the fragment; `N_FEATURES` — largest number of keypoints; `FAST_THRESHOLD` — $t$; `SCALE_FACTOR`, `N_LEVELS` — pyramid; `UNDISTORT` — correction of the distortion.

**Expected result.** The keypoints on both frames (circle — size, line — orientation), their number, the coverage of the frame (share of the cells of a grid 8 × 6 with at least one keypoint) and the time of the detection.

**What to observe.** The keypoints concentrate on the gates, the lamps and the edges of windows; the floor and the walls without texture have few of them. A lower FAST threshold gives more keypoints and a better coverage, but more of them lie on noise and on motion blur. After the correction of the distortion the black corners of the frame contain no keypoints.

**Control question 10.** Why is the orientation (14) computed for every keypoint? What limits the use of keypoints on the floor of the arena?

In [ ]:
#@title 3.4.2 ORB keypoints (code shown) { run: "auto" }
PAIR_POSITION = 0.5 #@param {type:"slider", min:0, max:1, step:0.01}
N_FEATURES = 1000 #@param {type:"slider", min:200, max:3000, step:100}
FAST_THRESHOLD = 20 #@param {type:"slider", min:5, max:60, step:1}
SCALE_FACTOR = 1.2 #@param {type:"slider", min:1.1, max:2.0, step:0.05}
N_LEVELS = 8 #@param {type:"slider", min:1, max:12, step:1}
UNDISTORT = True #@param {type:"boolean"}

K_cam, D_cam = tii.calibration()
map1, map2, K_und = aitbc.undistort_maps(K_cam, D_cam, (640, 480), alpha=0.0)
K_use = K_und if UNDISTORT else K_cam


def prepare(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    return cv2.remap(g, map1, map2, cv2.INTER_LINEAR) if UNDISTORT else g


levels = min(N_LEVELS, int(math.log(480 / 63.0) / math.log(SCALE_FACTOR)) + 1)    # the smallest level must hold a 31 x 31 patch
if levels < N_LEVELS:
    print(f"N_LEVELS reduced to {levels}: with the scale factor {SCALE_FACTOR} a deeper level would be smaller than the ORB patch")
orb = cv2.ORB_create(nfeatures=N_FEATURES, scaleFactor=SCALE_FACTOR, nlevels=levels, fastThreshold=FAST_THRESHOLD)
i_a = int(round(PAIR_POSITION * (len(FPV["frames"]) - 2)))
i_b = i_a + 1
ga, gb = prepare(FPV["frames"][i_a]), prepare(FPV["frames"][i_b])
t_orb = aitbc.time_call(lambda: orb.detectAndCompute(ga, None), repeats=3)
ka, da = orb.detectAndCompute(ga, None)
kb, db = orb.detectAndCompute(gb, None)


def coverage(kps, shape, nx=8, ny=6):
    cells = {(int(k.pt[0] * nx / shape[1]), int(k.pt[1] * ny / shape[0])) for k in kps}
    return len(cells) / (nx * ny)


PAIR = dict(ia=i_a, ib=i_b, ga=ga, gb=gb, ka=ka, da=da, kb=kb, db=db, K=K_use, undistort=UNDISTORT)
fig, ax = plt.subplots(1, 2, figsize=(18, 6.8))
for a, g_, k_, i_ in ((ax[0], ga, ka, i_a), (ax[1], gb, kb, i_b)):
    a.imshow(cv2.drawKeypoints(cv2.cvtColor(g_, cv2.COLOR_GRAY2RGB), k_, None, (0, 255, 0), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS))
    a.set_title(f"frame {i_} (t = {FPV['times'][i_]:.3f} s): {len(k_)} keypoints, coverage {coverage(k_, g_.shape):.0%}", fontsize=9)
    a.axis("off")
plt.tight_layout()
plt.show()
print(f"Time of ORB detection and description of one frame: {t_orb:.1f} ms; distortion corrected: {UNDISTORT}")

## Task 3.4.3. Matching of keypoints and quantitative estimation of the inter-frame shift

**Matching.** The distance of two binary descriptors is the Hamming distance — the number of different bits:

$$d_H(\mathbf a, \mathbf b) = \sum_{i=1}^{256} a_i \oplus b_i, \qquad \text{match if } \frac{d_1}{d_2} < \rho, \tag{15}$$

where $d_1$, $d_2$ are the distances to the nearest and the second nearest descriptor of the other frame (ratio test). Cross-check keeps a pair only when each descriptor is the nearest one for the other.

**Shift and rotation of the content.** The displacement of a matched pair is $\Delta \mathbf p = \mathbf p_b - \mathbf p_a$; the median of the displacements is the estimate of the inter-frame shift. A similarity transformation estimated by RANSAC (`cv2.estimateAffinePartial2D`) also gives the rotation of the content about the image centre; it equals the roll of the camera about the optical axis with the opposite sign and is compared with the gyroscope.

**Parameters.** `RATIO` — $\rho$; `CROSS_CHECK`; `SHOW_MATCHES` — number of drawn pairs; `COMPARE_STEPS` — the same measurements on 60 pairs of the fragment for every frame step 1, 2, 4, 8 (about 15 s).

**Expected result.** The matched pairs, the field of displacements, the histogram of the shifts; for the pair — the median shift and the rotation of the content compared with the gyroscope; the table of the matching for four frame rates.

**What to observe.** At 126 fps the median shift of the matched keypoints is about 4 px, the share of inliers about 0.8 and 3–7 % of the pairs fail (fragments A and B); at 16 fps the shift is about 26 px, the share of inliers falls to about 0.45 and 27–33 % of the pairs fail. The rotation of the content follows the roll measured by the gyroscope with the opposite sign: at 63 fps the correlation is 0.90–0.94 and the median error 0.2–0.3°. At 16 fps in fragment B the estimate of the rotation becomes unreliable (correlation −0.66): strongly rotated frames give many wrong matches. The frame rate determines whether a stabilisation by image matching is possible.

**Control question 11.** Why is the ratio test (15) used instead of a fixed threshold of the Hamming distance? Why does the rotation of the content have the opposite sign to the roll of the camera?

In [ ]:
#@title 3.4.3 Matching and the inter-frame shift (code shown) { run: "auto" }
RATIO = 0.75 #@param {type:"slider", min:0.5, max:0.95, step:0.01}
CROSS_CHECK = False #@param {type:"boolean"}
SHOW_MATCHES = 80 #@param {type:"slider", min:10, max:300, step:10}
COMPARE_STEPS = True #@param {type:"boolean"}

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=CROSS_CHECK)


def match(da, db):
    if da is None or db is None or len(da) < 2 or len(db) < 2:
        return []
    if CROSS_CHECK:
        return sorted(bf.match(da, db), key=lambda m: m.distance)
    return [p[0] for p in bf.knnMatch(da, db, k=2) if len(p) == 2 and p[0].distance < RATIO * p[1].distance]   # (15)


def roll_gyro(t1, t2):
    return np.degrees(aitbc.gyro_rotation(FPV["imu_t"], FPV["w_cam"], t1, t2)[2])


mm = match(PAIR["da"], PAIR["db"])
pa = np.float32([PAIR["ka"][m.queryIdx].pt for m in mm])
pb = np.float32([PAIR["kb"][m.trainIdx].pt for m in mm])
A, inl = cv2.estimateAffinePartial2D(pa, pb, method=cv2.RANSAC, ransacReprojThreshold=3.0) if len(mm) >= 3 else (None, None)
d = pb - pa
fig, ax = plt.subplots(1, 3, figsize=(19, 5.2), gridspec_kw=dict(width_ratios=[2, 1, 0.8]))
sel = mm[:SHOW_MATCHES]
ax[0].imshow(cv2.drawMatches(PAIR["ga"], PAIR["ka"], PAIR["gb"], PAIR["kb"], sel, None, matchColor=(0, 255, 0), flags=2))
ax[0].set_title(f"{len(mm)} matches ({'cross-check' if CROSS_CHECK else f'ratio test ρ = {RATIO}'}), {len(sel)} drawn", fontsize=9)
ax[0].axis("off")
ax[1].imshow(PAIR["gb"], cmap="gray")
if len(mm):
    good = inl.ravel() > 0 if inl is not None else np.ones(len(mm), bool)
    ax[1].quiver(pa[good, 0], pa[good, 1], d[good, 0], d[good, 1], color="lime", angles="xy", scale_units="xy", scale=1, width=0.003)
    ax[1].quiver(pa[~good, 0], pa[~good, 1], d[~good, 0], d[~good, 1], color="red", angles="xy", scale_units="xy", scale=1, width=0.003)
ax[1].set_title("displacements (green — inliers, red — outliers)", fontsize=9)
ax[1].axis("off")
ax[2].hist(np.hypot(d[:, 0], d[:, 1]), bins=40, color="C0")
ax[2].set_xlabel("|Δp|, px")
ax[2].set_title("shifts of the matched keypoints", fontsize=9)
plt.tight_layout()
plt.show()
ang_img = math.degrees(math.atan2(A[1, 0], A[0, 0])) if A is not None else float("nan")
roll = roll_gyro(FPV["times"][PAIR["ia"]], FPV["times"][PAIR["ib"]])
display(pd.DataFrame({"Value": {"matches": len(mm), "inliers of the similarity (RANSAC, 3 px)": int(inl.sum()) if inl is not None else 0,
                                "median shift, px": round(float(np.median(np.hypot(d[:, 0], d[:, 1]))), 1) if len(mm) else None,
                                "median shift (x, y), px": tuple(np.round(np.median(d, axis=0), 1)) if len(mm) else None,
                                "rotation of the content, °": round(ang_img, 2),
                                "roll of the camera by the gyroscope, °": round(roll, 2)}}))

if COMPARE_STEPS:
    t0 = time.time()
    rows = []
    clip, tt = FPV["clip_all"], FPV["times_all"]
    for step in (1, 2, 4, 8):
        starts = np.linspace(0, len(clip) - step - 1, 60).astype(int)
        recs = []
        for s_ in starts:
            ga_, gb_ = prepare(clip[s_]), prepare(clip[s_ + step])
            (k1, d1), (k2, d2) = orb.detectAndCompute(ga_, None), orb.detectAndCompute(gb_, None)
            m_ = match(d1, d2)
            r_ = dict(matches=len(m_), inliers=0, share=np.nan, shift_px=np.nan, ang=np.nan, roll=roll_gyro(tt[s_], tt[s_ + step]))
            if len(m_) >= 10:
                q1 = np.float32([k1[m.queryIdx].pt for m in m_])
                q2 = np.float32([k2[m.trainIdx].pt for m in m_])
                A_, in_ = cv2.estimateAffinePartial2D(q1, q2, method=cv2.RANSAC, ransacReprojThreshold=3.0)
                if A_ is not None:
                    r_.update(inliers=int(in_.sum()), share=float(in_.mean()), ang=math.degrees(math.atan2(A_[1, 0], A_[0, 0])),
                              shift_px=float(np.median(np.hypot(*(q2 - q1)[in_.ravel() > 0].T))))
            recs.append(r_)
        R = pd.DataFrame(recs)
        okr = R[R.inliers >= 15]
        rows.append({"frame step": step, "fps": round(fps_cam / step), "median matches": R.matches.median(),
                     "median share of inliers": okr.share.median(), "median shift, px": okr.shift_px.median(),
                     "failed pairs (< 15 inliers)": float((R.inliers < 15).mean()),
                     "corr(content rotation, −roll)": float(np.corrcoef(okr.ang, -okr.roll)[0, 1]) if len(okr) > 3 else np.nan,
                     "median |error| of the rotation, °": float(np.median(np.abs(okr.ang + okr.roll))) if len(okr) else np.nan,
                     "median |roll| by the gyroscope, °": float(np.median(np.abs(R.roll)))})
    MATCH_STEPS = pd.DataFrame(rows).set_index("frame step")
    display(MATCH_STEPS.round(3).style.set_caption(f"Matching on 60 pairs of fragment {SEGMENT[0]} for every frame step ({time.time() - t0:.0f} s)"))
    MATCH_STEPS.to_csv(os.path.join(REPORT_DIR, "lesson34_matching.csv"))

## Task 3.4.4. Homography and geometric alignment of frames

**Models of the inter-frame motion** (homogeneous coordinates $\tilde{\mathbf x} = (x, y, 1)^{\mathsf T}$):
- translation — 2 parameters $(t_x, t_y)$;
- similarity — 4 parameters (shift, rotation, scale);
- homography — 8 parameters:

$$\tilde{\mathbf x}_b \sim H\,\tilde{\mathbf x}_a, \qquad H = \begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1 \end{bmatrix}. \tag{16}$$

A homography describes exactly the images of a camera that only rotates ($H = K R K^{-1}$) and the images of a flat scene. It is estimated by RANSAC: a model is computed from a random minimal set of $m$ pairs ($m$ = 4 for $H$), the pairs with the reprojection error below the threshold are counted, and the best model is refined on its inliers. The number of iterations for the probability $p$ to draw at least one set without outliers when the share of inliers is $w$:

$$N = \frac{\ln(1 - p)}{\ln(1 - w^m)}. \tag{17}$$

**Alignment.** The first frame of the pair is transformed by the model (`cv2.warpPerspective`) and compared with the second one.

**Parameters.** `RANSAC_THRESHOLD` — largest reprojection error of an inlier, px; `CONFIDENCE` — $p$ in (17).

**Expected result.** The table of the three models: inliers, median reprojection error, mean absolute difference of the frames after alignment; the difference of the frames before and after the alignment with the homography; the rotation of the camera computed from $H$ (pure rotation model) compared with the gyroscope; $N$ by (17).

**What to observe.** In the default pair the homography has the most inliers (109 of 146 against 93 for the similarity) and the smallest difference after alignment (2.6 grey levels against 3.6 without alignment). The residual difference remains at the near objects (gates, the floor), because the drone also moves forward at several metres per second and the scene of a hall is not flat (parallax). For the same reason the rotation computed from $H$ agrees with the gyroscope only approximately; on 55 pairs of fragment B the correlation is 0.85 for the roll, 0.84 for the pan and 0.57 for the tilt, which is distorted most by the forward motion. With a lower frame rate the homography is fitted to fewer inliers and can align a pair worse than the similarity. With $w$ = 0.6 and $p$ = 0.99 formula (17) gives 34 iterations for a homography, with $w$ = 0.75 — 13.

**Control question 12.** In which two cases does a homography describe the motion between two frames exactly? Why does the alignment of an FPV frame with a homography leave a residual difference at near objects?

In [ ]:
#@title 3.4.4 Homography and alignment (code shown) { run: "auto" }
RANSAC_THRESHOLD = 3.0 #@param {type:"slider", min:0.5, max:10, step:0.5}
CONFIDENCE = 0.99 #@param {type:"slider", min:0.9, max:0.999, step:0.001}

ga, gb = PAIR["ga"], PAIR["gb"]
valid_a = cv2.remap(np.full(ga.shape, 255, np.uint8), map1, map2, cv2.INTER_NEAREST) > 0 if PAIR["undistort"] else np.ones(ga.shape, bool)


def reproj_error(Hm, p1, p2):
    q = cv2.perspectiveTransform(p1.reshape(-1, 1, 2), Hm).reshape(-1, 2)
    return np.hypot(*(q - p2).T)


models = {}
t_med = np.median(pb - pa, axis=0)
models["translation (median shift)"] = np.array([[1, 0, t_med[0]], [0, 1, t_med[1]], [0, 0, 1]], float)
A_s, _ = cv2.estimateAffinePartial2D(pa, pb, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_THRESHOLD, confidence=CONFIDENCE)
models["similarity (4 parameters)"] = np.vstack([A_s, [0, 0, 1]])
H_, inl_h = cv2.findHomography(pa, pb, cv2.RANSAC, RANSAC_THRESHOLD, confidence=CONFIDENCE)    # formula (16)
if H_ is None:
    H_, inl_h = models["similarity (4 parameters)"], np.ones((len(pa), 1), np.uint8)
    print("The homography could not be estimated for this pair; the similarity is used instead.")
models["homography (8 parameters)"] = H_
diff0 = np.abs(gb.astype(np.float32) - ga.astype(np.float32))[valid_a].mean()
rows, warped = [], {}
for name, Hm in models.items():
    e = reproj_error(Hm, pa, pb)
    wa = cv2.warpPerspective(ga, Hm, (ga.shape[1], ga.shape[0]))
    wv = cv2.warpPerspective(valid_a.astype(np.uint8), Hm, (ga.shape[1], ga.shape[0])) > 0
    warped[name] = (wa, wv)
    rows.append({"model": name, "inliers (error < threshold)": int((e < RANSAC_THRESHOLD).sum()),
                 "median reprojection error, px": float(np.median(e)),
                 "mean |difference| after alignment": float(np.abs(gb.astype(np.float32) - wa.astype(np.float32))[wv & valid_a].mean())})
rows.insert(0, {"model": "without alignment", "inliers (error < threshold)": int((np.hypot(*(pb - pa).T) < RANSAC_THRESHOLD).sum()),
                "median reprojection error, px": float(np.median(np.hypot(*(pb - pa).T))), "mean |difference| after alignment": float(diff0)})
display(pd.DataFrame(rows).set_index("model").round(2).style.set_caption(f"Pair of frames {PAIR['ia']}–{PAIR['ib']}, {len(pa)} matches"))

wa, wv = warped["homography (8 parameters)"]
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
ax[0].imshow(np.dstack([gb, ga, ga]))
ax[0].set_title("before alignment: frame b (red) over frame a (cyan)", fontsize=9)
ax[1].imshow(np.abs(gb.astype(np.int16) - ga.astype(np.int16)), cmap="magma", vmin=0, vmax=80)
ax[1].set_title(f"|b − a|: mean {diff0:.1f}", fontsize=9)
dd = np.abs(gb.astype(np.int16) - wa.astype(np.int16)) * (wv & valid_a)
ax[2].imshow(dd, cmap="magma", vmin=0, vmax=80)
ax[2].set_title(f"|b − H(a)| after the homography: mean {dd[wv & valid_a].mean():.1f}", fontsize=9)
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

rv_img = aitbc.rotation_from_homography(H_, PAIR["K"])
rv_gyro = aitbc.gyro_rotation(FPV["imu_t"], FPV["w_cam"], FPV["times"][PAIR["ia"]], FPV["times"][PAIR["ib"]])
w_in = float(inl_h.mean())
display(pd.DataFrame({"from the homography (pure rotation)": np.degrees(-rv_img), "gyroscope": np.degrees(rv_gyro)},
                     index=["tilt about x, °", "pan about y, °", "roll about the optical axis, °"]).round(2)
        .style.set_caption("Rotation of the camera between the frames of the pair" + ("" if PAIR["undistort"] else " (distortion not corrected: K of the lens used)")))
n_iter = 1 if w_in ** 4 >= 1 - 1e-9 else (math.inf if w_in <= 0 else math.ceil(math.log(1 - CONFIDENCE) / math.log(1 - w_in ** 4)))
print(f"Share of inliers of the homography w = {w_in:.2f}: N = ln(1 − p)/ln(1 − w⁴) = {n_iter} iterations for p = {CONFIDENCE} (formula 17)")

t0 = time.time()
cmp_rows = []
for s_ in np.linspace(0, len(FPV["frames"]) - 2, 60).astype(int):
    g1, g2 = prepare(FPV["frames"][s_]), prepare(FPV["frames"][s_ + 1])
    (k1, d1), (k2, d2) = orb.detectAndCompute(g1, None), orb.detectAndCompute(g2, None)
    m_ = match(d1, d2)
    if len(m_) < 15:
        continue
    q1 = np.float32([k1[m.queryIdx].pt for m in m_])
    q2 = np.float32([k2[m.trainIdx].pt for m in m_])
    Hq, iq = cv2.findHomography(q1, q2, cv2.RANSAC, RANSAC_THRESHOLD, confidence=CONFIDENCE)
    if Hq is None or iq.sum() < 15:
        continue
    rv_h = aitbc.rotation_from_homography(Hq, PAIR["K"])
    if not np.all(np.isfinite(rv_h)):
        continue
    cmp_rows.append(np.r_[np.degrees(-rv_h),
                          np.degrees(aitbc.gyro_rotation(FPV["imu_t"], FPV["w_cam"], FPV["times"][s_], FPV["times"][s_ + 1]))])
C = np.array(cmp_rows).reshape(-1, 6)
ROT_CHECK = pd.DataFrame({"correlation": [np.corrcoef(C[:, j], C[:, j + 3])[0, 1] for j in range(3)],
                          "median |error|, °": [np.median(np.abs(C[:, j] - C[:, j + 3])) for j in range(3)],
                          "median |rotation| by the gyroscope, °": [np.median(np.abs(C[:, j + 3])) for j in range(3)]},
                         index=["tilt about x", "pan about y", "roll about the optical axis"])
display(ROT_CHECK.round(2).style.set_caption(f"Rotation from the homography against the gyroscope on {len(C)} pairs of the fragment ({time.time() - t0:.0f} s)"))

## Task 3.4.5. Saving of the stabilised video stream and comparative analysis

**Stabilisation by smoothing of the trajectory.** For every pair of consecutive frames the similarity transformation gives the shift and rotation of the content $(\Delta x_k, \Delta y_k, \Delta\alpha_k)$ (task 3.4.3; when a pair fails, the motion of the previous pair is kept). Their sums form the trajectory of the camera in the image, which is smoothed by a moving average over $2r + 1$ frames:

$$\mathbf T_k = \sum_{j \le k} (\Delta x_j, \Delta y_j, \Delta\alpha_j), \qquad \bar{\mathbf T}_k = \frac{1}{2r + 1}\sum_{j = k - r}^{k + r} \mathbf T_j, \tag{18}$$

and every frame is transformed by the correction $\bar{\mathbf T}_k - \mathbf T_k$: the slow, intended motion of the pilot remains, the fast oscillations are removed. The borders of the transformed frames that have no data are black or are cut off by a zoom.

**Measures of the comparison.**
- *high-frequency jitter* — standard deviation of the trajectory around its moving average with $r$ = 0.1 s, in pixels (shift) and degrees (rotation); for the stabilised video the motion is measured again on its frames (keypoints only in the valid part; pairs without an estimate count as zero motion);
- *jitter of the rotation by the gyroscope* — the same measure for the rotation of the content computed from the gyroscope, $-\int \omega_z\,dt$, before and after the correction: an independent check that does not depend on the image matching;
- *inter-frame transformation fidelity* (higher means a steadier video):

$$\mathrm{ITF} = \frac{1}{N - 1}\sum_{k=2}^{N} \mathrm{PSNR}(I_{k-1}, I_k); \tag{19}$$

- *share of valid pixels* — the part of the frame that keeps image data after the correction.

**Parameters.** `SMOOTHING_RADIUS_S` — $r$ of (18) in seconds; `BORDER` — black borders or a zoom; `ZOOM`; `RUN_RADIUS_SWEEP` — the measures for $r$ = 0.1, 0.25 and 0.5 s (about 20 s); `SAVE_VIDEO` — the original and the stabilised video side by side in `outputs/video`.

**Expected result.** The trajectories $x$, $y$, $\alpha$ before and after smoothing, frames of the original and of the stabilised video, the table of the measures and its file `outputs/reports/lesson34_stabilisation.csv`, the video file and the record in `project.json`.

**What to observe.** At 63 fps and $r$ = 0.25 s the high-frequency jitter falls by 43–53 %: in fragment B from 33 to 19 px in shift and from 2.8° to 1.6° in rotation, in fragment A from 33 to 16 px and from 2.7° to 1.3°. The gyroscope confirms the result independently: the jitter of the rotation falls from 3.1° to 2.0° (B) and from 2.5° to 1.4° (A). About 87–89 % of the pixels remain valid. ITF grows only by about 0.3 dB, because the forward motion and the parallax of the near gates remain. A larger $r$ cuts off a larger part of the frame (74 % of valid pixels at $r$ = 0.5 s) and delays the intended turns of the pilot. In fragment B at $r$ = 0.5 s the motion measured again on the stabilised frames even grows (48 px), because more pairs fail at the wide black borders (23 of 239), while the gyroscope shows a further small decrease (1.9°): a measure computed from the processed frames themselves needs an independent check. At 32 and 16 fps more pairs fail and the trajectory becomes less reliable. An operator of an FPV drone needs a moderate smoothing without delay; a recorded video for analysis allows a stronger one.

**Control question 13.** Why is the trajectory (18) smoothed instead of compensating every inter-frame motion completely? What limits the radius of the smoothing for a pilot who controls the drone by the video?

In [ ]:
#@title 3.4.5 Stabilised video and comparative analysis (code shown) { display-mode: "form" }
SMOOTHING_RADIUS_S = 0.25 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
BORDER = "black borders" #@param ["black borders", "crop by zoom"]
ZOOM = 1.15 #@param {type:"slider", min:1.0, max:1.5, step:0.05}
MOTION_MODEL = "similarity (shift and rotation)" #@param ["similarity (shift and rotation)", "translation (shift only)"]
RUN_RADIUS_SWEEP = True #@param {type:"boolean"}
SAVE_VIDEO = True #@param {type:"boolean"}

greys = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in FPV["frames"]]
fps = FPV["fps"]
orb_s = cv2.ORB_create(1000)


def interframe_motion(frames, masks=None):
    # keypoints only inside the valid part of a frame: the edges of black borders must not be matched
    er = np.ones((15, 15), np.uint8)
    kd = [orb_s.detectAndCompute(g, None if masks is None else cv2.erode(masks[k].astype(np.uint8) * 255, er)) for k, g in enumerate(frames)]
    T, ok = [np.zeros(3)], [True]
    for k in range(1, len(frames)):
        (k1, d1), (k2, d2) = kd[k - 1], kd[k]
        m_ = match(d1, d2)
        A_ = None
        if len(m_) >= 10:
            q1 = np.float32([k1[m.queryIdx].pt for m in m_])
            q2 = np.float32([k2[m.trainIdx].pt for m in m_])
            A_, in_ = cv2.estimateAffinePartial2D(q1, q2, method=cv2.RANSAC, ransacReprojThreshold=3.0)
            if A_ is not None and in_.sum() < 15:
                A_ = None
        if A_ is None:
            T.append(T[-1].copy())
            ok.append(False)
        elif MOTION_MODEL.startswith("translation"):
            dxy = np.median((q2 - q1)[in_.ravel() > 0], axis=0)
            T.append(np.array([dxy[0], dxy[1], 0.0]))
            ok.append(True)
        else:
            T.append(np.array([A_[0, 2], A_[1, 2], math.atan2(A_[1, 0], A_[0, 0])]))
            ok.append(True)
    return np.array(T), np.array(ok)


def moving_average(x, r):
    if r < 1:
        return x.copy()
    xp = np.pad(x, ((r, r), (0, 0)), mode="edge")
    k = np.ones(2 * r + 1) / (2 * r + 1)
    return np.stack([np.convolve(xp[:, j], k, mode="valid") for j in range(x.shape[1])], 1)


def stabilise(frames, T, r, zoom=1.0):
    traj = np.cumsum(T, 0)
    smooth = moving_average(traj, r)                                    # formula (18)
    corr = smooth - traj
    h, w = frames[0].shape[:2]
    Z = cv2.getRotationMatrix2D((w / 2, h / 2), 0, zoom)
    out, valid = [], []
    for f, (dx, dy, da) in zip(frames, corr):
        M = np.array([[math.cos(da), -math.sin(da), dx], [math.sin(da), math.cos(da), dy], [0, 0, 1]])
        M = (np.vstack([Z, [0, 0, 1]]) @ M)[:2]
        out.append(cv2.warpAffine(f, M, (w, h)))
        valid.append(cv2.warpAffine(np.full((h, w), 255, np.uint8), M, (w, h)) > 0)
    return out, valid, traj, smooth


def jitter(T, ok, r):
    # high-frequency part of the trajectory; pairs without an estimate count as zero motion (no accumulation of errors)
    T = np.where(ok[:, None], T, 0.0)
    traj = np.cumsum(T, 0)
    e = (traj - moving_average(traj, r)).std(0)
    return float(np.hypot(e[0], e[1])), float(np.degrees(e[2]))


# independent reference of the rotation of the content: alpha_k = -(roll of the camera), integrated gyroscope
roll_steps = [aitbc.gyro_rotation(FPV["imu_t"], FPV["w_cam"], a, b)[2] for a, b in zip(FPV["times"][:-1], FPV["times"][1:])]
alpha_gyro = -np.r_[0.0, np.cumsum(roll_steps)]


def hf_std(x, r):
    return float(np.degrees((x - moving_average(x[:, None], r)[:, 0]).std()))


def compare(r_s, zoom):
    r = max(1, int(round(r_s * fps)))
    out, valid, traj, smooth = stabilise(greys, T_raw, r, zoom)
    T_st, ok_st = interframe_motion(out, valid)
    r_j = max(2, int(round(0.1 * fps)))
    j0, j1 = jitter(T_raw, ok_raw, r_j), jitter(T_st, ok_st, r_j)
    corr_alpha = (smooth - traj)[:, 2]
    return out, valid, traj, smooth, {"radius r, s": r_s, "radius r, frames": r,
                                      "jitter of shift: original, px": j0[0], "jitter of shift: stabilised, px": j1[0],
                                      "jitter of rotation: original, °": j0[1], "jitter of rotation: stabilised, °": j1[1],
                                      "jitter of rotation by the gyroscope: original, °": hf_std(alpha_gyro, r_j),
                                      "jitter of rotation by the gyroscope: stabilised, °": hf_std(alpha_gyro + corr_alpha, r_j),
                                      "pairs without estimate: original / stabilised": f"{int((~ok_raw).sum())} / {int((~ok_st).sum())}",
                                      "ITF original, dB": aitbc.interframe_psnr(greys), "ITF stabilised, dB": aitbc.interframe_psnr(out, valid),
                                      "share of valid pixels": float(np.mean([v.mean() for v in valid]))}


t0 = time.time()
T_raw, ok_raw = interframe_motion(greys)
zoom = ZOOM if BORDER == "crop by zoom" else 1.0
stab, valid, traj, smooth, res = compare(SMOOTHING_RADIUS_S, zoom)
print(f"Motion model: {MOTION_MODEL}; pairs without a reliable estimate (motion of the previous pair kept): {int((~ok_raw).sum())} of {len(greys) - 1}")

fig, ax = plt.subplots(1, 3, figsize=(19, 3.8))
tt = FPV["times"]
for j, (lab, sc) in enumerate((("x, px", 1), ("y, px", 1), ("α, °", 180 / math.pi))):
    ax[j].plot(tt, traj[:, j] * sc, lw=0.8, label="trajectory T_k")
    ax[j].plot(tt, smooth[:, j] * sc, lw=1.6, label=f"smoothed, r = {SMOOTHING_RADIUS_S:g} s")
    ax[j].set_xlabel("time of the flight, s")
    ax[j].set_ylabel(lab)
    ax[j].grid(alpha=0.3)
ax[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"lesson34_trajectory_{FPV['segment'][0]}.png"), dpi=90)
plt.show()
idx = np.linspace(0, len(greys) - 1, 5).astype(int)
fig, ax = plt.subplots(2, 5, figsize=(19, 6))
for j, i in enumerate(idx):
    ax[0, j].imshow(greys[i], cmap="gray")
    ax[0, j].set_title(f"original, frame {i}", fontsize=8.5)
    ax[1, j].imshow(stab[i], cmap="gray")
    ax[1, j].set_title(f"stabilised, frame {i}", fontsize=8.5)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

rows = [res]
if RUN_RADIUS_SWEEP:
    for r_s in (0.1, 0.25, 0.5):
        if abs(r_s - SMOOTHING_RADIUS_S) > 1e-9:
            rows.append(compare(r_s, zoom)[4])
STAB = pd.DataFrame(rows).set_index("radius r, s").sort_index()
display(STAB.style.format(precision=2).set_caption(f"Fragment {FPV['segment'][0]}, {fps:.0f} fps, {BORDER} ({time.time() - t0:.0f} s)"))
STAB.to_csv(os.path.join(REPORT_DIR, "lesson34_stabilisation.csv"))
if SAVE_VIDEO:
    side = [np.hstack([g_, s_]) for g_, s_ in zip(greys, stab)]
    out_path = os.path.join(VIDEO_DIR, f"fpv_stabilised_{FPV['segment'][0]}_{fps:.0f}fps.mp4")
    aitbc.write_video(out_path, side, fps)
    print(f"Saved: {out_path} (left — original, right — stabilised), {aitbc.video_info(out_path)['size_mb']:.1f} MB")
update_project_json("3.4", {"lesson_3_4": dict(fragment=FPV["segment"], frame_step=FPV["step"], fps=round(fps, 1),
                                               video=os.path.relpath(FPV["path"], PROJECT_ROOT), smoothing_radius_s=SMOOTHING_RADIUS_S,
                                               border=BORDER, motion_model=MOTION_MODEL,
                                               measures={k_: (round(float(v), 3) if not isinstance(v, str) else v) for k_, v in res.items()},
                                               stabilised_video=os.path.relpath(out_path, PROJECT_ROOT) if SAVE_VIDEO else None)})

### Report of Lesson 3.4

Double-click this cell and write the answers.

1. Fragment, frame step, rotation between frames by the gyroscope; answer to control question 9:
2. Parameters of ORB, number and coverage of the keypoints; answer to control question 10:
3. Matches, shift, rotation compared with the gyroscope, table for four frame rates; answer to control question 11:
4. Comparison of the motion models, rotation from the homography and from the gyroscope; answer to control question 12:
5. Radius of the smoothing, jitter, ITF and valid pixels before and after the stabilisation, saved video; answer to control question 13:

---
## Self-check of Topic 3

1. A thermal frame of a UAV at night contains persons, parked cars and a warm road. Which steps of the classical processing are needed to obtain the boxes of the persons, and which polarity is needed for the cars?
2. Why do the Canny edges agree only partly with the boundaries of the annotation, and why can no thresholds be chosen for all frames?
3. Which geometric characteristics of contours are computed in metres, and which model of the GSD is used for an oblique frame?
4. An FPV video is transmitted at 30 frames per second instead of 120. How does this change the matching of keypoints and the stabilisation?
5. When is a homography an exact model of the motion between two frames, and why does a stabilised FPV video still contain residual motion?

## Data sources and licences

The course library reads the files directly from the repositories of the authors; the data are not redistributed with the notebook. When images or results are published, the data sets are cited.

1. **HIT-UAV.** Suo J., Wang T., Zhang X., Chen H., Zhou W., Shi W. HIT-UAV: A high-altitude infrared thermal dataset for Unmanned Aerial Vehicle-based object detection. *Scientific Data*, 10, 227 (2023). https://doi.org/10.1038/s41597-023-02066-6 · Data: https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset · Licence CC BY 4.0.
2. **Kust4K.** Ouyang J., Wang Q., Shang Y., Jin P., Zhong H., Zhou L., Shen T. An RGB-TIR dataset from UAV platform for robust urban traffic scenes semantic segmentation. *Scientific Data*, 12, 1701 (2025). https://doi.org/10.1038/s41597-025-05994-7 · Data: https://doi.org/10.6084/m9.figshare.29476610 · Licence CC BY 4.0.
3. **M3OT.** Nie Z., Xue L., Fang Z., Ren J., Wei Y., Zheng J. M3OT: A Multi-Drone Multi-Modality dataset for Multi-Object Tracking. *Scientific Data*, 12, 1927 (2025). https://doi.org/10.1038/s41597-025-06204-0 · Data: https://doi.org/10.6084/m9.figshare.28308887 · Licence CC BY 4.0.
4. **TII-RATM.** Bosello M., Aguiari D., Keuter Y., Pallotta E., Kiade S., Caminati G., Pinzarrone F., Halepota J., Panerati J., Pau G. Race Against the Machine: A Fully-Annotated, Open-Design Dataset of Autonomous and Piloted High-Speed Flight. *IEEE Robotics and Automation Letters*, 9(4), 3799–3806 (2024). https://doi.org/10.1109/LRA.2024.3371288 · Data: https://github.com/tii-racing/drone-racing-dataset (release v3.0.0) · Licence CC BY 4.0.

**Next notebook.** Notebook 06 — Topic 4, Lessons 4.1 (lecture) and 4.2 (group class): machine learning in the recognition of military objects, HOG features, SVM and Random Forest, metrics of classification; convolutional networks, MobileNet, transfer learning and export to ONNX.